In [ ]:
# ============================================================
# FIXED-CANDIDATE FULL CATALOG REBUILD
# ------------------------------------------------------------
# Frozen branch:
#   source: rho_eff ~ (1/r^2) d/dr [ r * V_bar^2 ]
#   screened field equation
#   fixed Rs = 1.5
#   shape map: V/V_flat = U/U_inf
#
# Frozen amplitude law:
#   U(r_t) = 0.6 U_inf
#   V_flat^2 = C * [r_t u(r_t)]^alpha * [Vbar^2(r_t)]^beta
#
# with fixed constants:
#   C     = 164.0
#   alpha = 0.175
#   beta  = 0.55
#
# No fitting in this script.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_fixed_candidate_catalog_rebuild")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(vbar2_from_rot(rot), 0.0)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

# ------------------------------------------------------------
# RUN FULL CATALOG
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()
rotmod_map = {display_name_from_path(p): p for p in rotmod_files}

rows = []
fails = []

print("=" * 60)
print("RUNNING FIXED-CANDIDATE FULL CATALOG")
print("=" * 60)
print(f"Candidate files : {len(rotmod_files)}")
print(f"Fixed Rs        : {Rs_fixed}")
print(f"Fixed f         : {F_FRAC}")
print(f"Fixed alpha     : {ALPHA}")
print(f"Fixed beta      : {BETA}")
print(f"Fixed C         : {C_AMP}")
print()

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        U_inf = float(np.max(U))
        rt = radius_at_fraction(r, U, F_FRAC)

        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            raise RuntimeError("Invalid transition quantities")

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_obs = np.maximum(vbar2_from_rot(rot), 0.0)
        vbar2_interp = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)
        chi2_mean = float(np.mean(((rot["vobs"] - V_obs_pred) / rot["ev"]) ** 2))

        obs_shape = rot["vobs"] / max(float(np.max(rot["vobs"])), 1.0e-12)
        pred_shape = V_obs_pred / max(float(np.max(V_obs_pred)), 1.0e-12)
        shape_rmse = safe_rmse(obs_shape, pred_shape)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "Rs_fixed": Rs_fixed,
            "rt": rt,
            "u_t": u_t,
            "vbar2_t": vbar2_t,
            "carrier": carrier,
            "V_flat_pred": V_flat_pred,
            "rmse": rmse,
            "mae": mae,
            "chi2_mean": chi2_mean,
            "shape_rmse": shape_rmse,
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

res_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(res_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

summary = {
    "successful_galaxies": int(len(res_df)),
    "failed_galaxies": int(len(fail_df)),
    "median_rmse": float(res_df["rmse"].median()),
    "mean_rmse": float(res_df["rmse"].mean()),
    "p90_rmse": float(np.percentile(res_df["rmse"], 90)),
    "p95_rmse": float(np.percentile(res_df["rmse"], 95)),
    "median_mae": float(res_df["mae"].median()),
    "median_chi2_mean": float(res_df["chi2_mean"].median()),
    "median_shape_rmse": float(res_df["shape_rmse"].median()),
}

best20 = res_df.sort_values("rmse").head(20).reset_index(drop=True)
worst20 = res_df.sort_values("rmse", ascending=False).head(20).reset_index(drop=True)

# Simple bins for tail view
res_df["vmax_bin"] = pd.cut(
    res_df["vmax_obs"],
    bins=[-np.inf, 80, 150, np.inf],
    labels=["low", "mid", "high"]
)

bin_summary = (
    res_df.groupby("vmax_bin", observed=True)[["rmse", "mae", "chi2_mean", "shape_rmse"]]
    .median()
    .reset_index()
)

print()
print("=" * 60)
print("FIXED-CANDIDATE SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:22s} = {v}")

print()
print("=" * 60)
print("MEDIAN METRICS BY VMAX BIN")
print("=" * 60)
print(bin_summary.to_string(index=False))

print()
print("=" * 60)
print("BEST 20 BY RMSE")
print("=" * 60)
print(best20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("=" * 60)
print("WORST 20 BY RMSE")
print("=" * 60)
print(worst20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

RES_CSV = os.path.join(OUTDIR, "fixed_candidate_results.csv")
FAIL_CSV = os.path.join(OUTDIR, "fixed_candidate_failures.csv")
BEST_CSV_OUT = os.path.join(OUTDIR, "fixed_candidate_best20.csv")
WORST_CSV_OUT = os.path.join(OUTDIR, "fixed_candidate_worst20.csv")
BIN_CSV = os.path.join(OUTDIR, "fixed_candidate_bin_summary.csv")
TXT_OUT = os.path.join(OUTDIR, "fixed_candidate_summary.txt")

res_df.to_csv(RES_CSV, index=False)
fail_df.to_csv(FAIL_CSV, index=False)
best20.to_csv(BEST_CSV_OUT, index=False)
worst20.to_csv(WORST_CSV_OUT, index=False)
bin_summary.to_csv(BIN_CSV, index=False)

with open(TXT_OUT, "w", encoding="utf-8") as f:
    f.write("FIXED-CANDIDATE SUMMARY\n")
    f.write("=" * 60 + "\n")
    for k, v in summary.items():
        f.write(f"{k:22s} = {v}\n")
    f.write("\nMEDIAN METRICS BY VMAX BIN\n")
    f.write("=" * 60 + "\n")
    f.write(bin_summary.to_string(index=False))
    f.write("\n\nBEST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(best20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))
    f.write("\n\nWORST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

Auto-extracting LTG rotmods from: /content/Rotmod_LTG (4).zip
RUNNING FIXED-CANDIDATE FULL CATALOG
Candidate files : 175
Fixed Rs        : 1.5
Fixed f         : 0.6
Fixed alpha     : 0.175
Fixed beta      : 0.55
Fixed C         : 164.0

Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

FIXED-CANDIDATE SUMMARY
successful_galaxies    = 175
failed_galaxies        = 0
median_rmse            = 20.38786367728187
mean_rmse              = 32.72222113537459
p90_rmse               = 77.35749478059131
p95_rmse               = 88.8575199776623
median_mae             = 17.608721572718746
median_chi2_mean       = 26.12511787253237
median_shape_rmse      = 0.15760201986858705

MEDIAN METRICS BY VMAX BIN
vmax_bin      rmse       mae  chi2_mean  shape_rmse
     low 11.457480 10.528257  14.716546    0.128219
     mid 18.074875 15.106115  13.274674    0.115517
    hig

In [ ]:
# ============================================================
# HIGH-END AMPLITUDE TAIL CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Tail correction:
#   V_flat,corr = V_flat,base * [1 + eta * (Vbar2_t / V0^2)^delta]
#
# Only amplitude is modified. Source/field/shape stay fixed.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_tail_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00]
V0_LIST    = [120.0, 150.0, 180.0, 210.0, 240.0]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(vbar2_from_rot(rot), 0.0)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING TAIL-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_obs = np.maximum(vbar2_from_rot(rot), 0.0)
        vbar2_interp = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "vbar2_t": vbar2_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for V0 in V0_LIST:
            rmses = []
            vmax_bins = []

            for _, row in df.iterrows():
                correction = 1.0 + eta * (row["vbar2_t"] / (V0**2)) ** delta
                V_flat_pred = row["V_flat_base"] * correction
                V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

                V_obs_pred = interp1d(
                    row["r_grid"], V_theory,
                    kind="linear", bounds_error=False, fill_value="extrapolate"
                )(row["r_obs"])

                rmse = safe_rmse(row["v_obs"], V_obs_pred)
                rmses.append(rmse)
                vmax_bins.append(row["vmax_obs"])

            rmses = np.asarray(rmses, float)
            vmax_bins = np.asarray(vmax_bins, float)

            high_mask = vmax_bins >= 150.0
            high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

            summary_rows.append({
                "eta": eta,
                "delta": delta,
                "V0": V0,
                "median_rmse": float(np.median(rmses)),
                "p90_rmse": float(np.percentile(rmses, 90)),
                "high_bin_median_rmse": high_med,
            })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("HIGH-END TAIL CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_tail_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_tail_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END TAIL CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING TAIL-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END TAIL CORRECTION SUMMARY
 eta  delta    V0  median_rmse  p90_rmse  high_bin_median_rmse
0.00   0.25 120.0    20.387864 77.357495             63.789441
0.00   0.25 150.0    20.387864 77.357495             63.789441
0.00   0.25 180.0    20.387864 77.357495             63.789441
0.00   0.25 210.0    20.387864 77.357495             63.789441
0.00   0.25 240.0    20.387864 77.357495             63.789441
0.00   0.50 120.0    20.387864 77.357495             63.789441
0.00   0.50 150.0    20.387864 77.357495             63.789441
0.00   0.50 180.0    20.387864 77.357495             63.789441
0.00   0.50 210.0    20.387864 77.357495             63.789441
0.00   0.50 240.0    20.387864 77.357495             63.789441
0.00   0.75 120.0    20.387864 77.357495             6

In [ ]:
# ============================================================
# HIGH-END FAILURE-MODE AUDIT
# ------------------------------------------------------------
# Audits the fixed candidate branch:
#
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Focus:
#   Understand what the worst high-vmax failures share.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_failure_audit")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files
    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break
    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")
    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_components(rot):
    vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
    vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
    vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2
    return vgas2, vdisk2, vbul2, vbar2

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    _, _, _, vbar2 = vbar2_components(rot)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("RUNNING FAILURE-MODE AUDIT")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2, vdisk2, vbul2, vbar2_obs = vbar2_components(rot)
        vgas2_i  = interp1d(rot["r"], vgas2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vdisk2_i = interp1d(rot["r"], vdisk2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_i  = interp1d(rot["r"], vbul2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_i  = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vgas2_t = max(float(vgas2_i(rt)), 0.0)
        vdisk2_t = max(float(vdisk2_i(rt)), 0.0)
        vbul2_t = max(float(vbul2_i(rt)), 0.0)
        vbar2_t = max(float(vbar2_i(rt)), 1.0e-30)

        bulge_frac_t = vbul2_t / vbar2_t if vbar2_t > 0 else np.nan
        disk_frac_t = vdisk2_t / vbar2_t if vbar2_t > 0 else np.nan
        gas_frac_t = vgas2_t / vbar2_t if vbar2_t > 0 else np.nan

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "rt": rt,
            "rt_over_rmax": rt / max(float(np.max(r_obs)), 1.0e-12),
            "u_t": u_t,
            "rt_ut": rt * u_t,
            "vbar2_t": vbar2_t,
            "vbar_t": np.sqrt(vbar2_t),
            "bulge_frac_t": bulge_frac_t,
            "disk_frac_t": disk_frac_t,
            "gas_frac_t": gas_frac_t,
            "V_flat_pred": V_flat_pred,
            "Vflat_over_vmax": V_flat_pred / max(float(np.max(rot["vobs"])), 1.0e-12),
            "rmse": rmse,
            "mae": mae,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)
high = df[df["vmax_obs"] >= HIGH_VMAX_CUT].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

worst20 = high.head(20).copy()
best20 = high.tail(20).copy().sort_values("rmse").reset_index(drop=True)

summary_vars = [
    "vmax_obs", "r_max", "rt", "rt_over_rmax", "u_t", "rt_ut",
    "vbar2_t", "vbar_t", "bulge_frac_t", "disk_frac_t", "gas_frac_t",
    "V_flat_pred", "Vflat_over_vmax", "rmse", "mae"
]

summary_df = pd.DataFrame({
    "quantity": summary_vars,
    "worst20_median": [float(worst20[c].median()) for c in summary_vars],
    "best20_median": [float(best20[c].median()) for c in summary_vars],
})
summary_df["worst_minus_best"] = summary_df["worst20_median"] - summary_df["best20_median"]

print()
print("=" * 60)
print("HIGH-END FAILURE AUDIT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("=" * 60)
print("WORST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(worst20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

print()
print("=" * 60)
print("BEST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(best20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_failure_summary.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "high_end_worst20.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "high_end_best20.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "high_end_full_audit.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_failure_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END FAILURE AUDIT SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\nWORST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))
    f.write("\n\nBEST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(best20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

RUNNING FAILURE-MODE AUDIT
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END FAILURE AUDIT SUMMARY
       quantity  worst20_median  best20_median  worst_minus_best
       vmax_obs      274.000000     186.000000         88.000000
          r_max       44.215000      26.930000         17.285000
             rt        4.236476       3.368903          0.867573
   rt_over_rmax        0.154626       0.124820          0.029805
            u_t        0.097594       0.084961          0.012633
          rt_ut        0.423070       0.275523          0.147548
        vbar2_t    77995.021748   35983.678528      42011.343220
         vbar_t      279.272915     189.684292         89.588623
   bulge_frac_t        0.281828       0.000000          0.281828
    disk_frac_t        0.717890       0.987273         -0.269383
     gas_frac_t        0.000091       

In [ ]:
# ============================================================
# HIGH-END FAILURE-MODE AUDIT
# ------------------------------------------------------------
# Audits the fixed candidate branch:
#
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Focus:
#   Understand what the worst high-vmax failures share.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_failure_audit")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files
    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break
    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")
    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_components(rot):
    vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
    vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
    vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2
    return vgas2, vdisk2, vbul2, vbar2

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    _, _, _, vbar2 = vbar2_components(rot)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("RUNNING FAILURE-MODE AUDIT")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2, vdisk2, vbul2, vbar2_obs = vbar2_components(rot)
        vgas2_i  = interp1d(rot["r"], vgas2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vdisk2_i = interp1d(rot["r"], vdisk2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_i  = interp1d(rot["r"], vbul2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_i  = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vgas2_t = max(float(vgas2_i(rt)), 0.0)
        vdisk2_t = max(float(vdisk2_i(rt)), 0.0)
        vbul2_t = max(float(vbul2_i(rt)), 0.0)
        vbar2_t = max(float(vbar2_i(rt)), 1.0e-30)

        bulge_frac_t = vbul2_t / vbar2_t if vbar2_t > 0 else np.nan
        disk_frac_t = vdisk2_t / vbar2_t if vbar2_t > 0 else np.nan
        gas_frac_t = vgas2_t / vbar2_t if vbar2_t > 0 else np.nan

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "rt": rt,
            "rt_over_rmax": rt / max(float(np.max(r_obs)), 1.0e-12),
            "u_t": u_t,
            "rt_ut": rt * u_t,
            "vbar2_t": vbar2_t,
            "vbar_t": np.sqrt(vbar2_t),
            "bulge_frac_t": bulge_frac_t,
            "disk_frac_t": disk_frac_t,
            "gas_frac_t": gas_frac_t,
            "V_flat_pred": V_flat_pred,
            "Vflat_over_vmax": V_flat_pred / max(float(np.max(rot["vobs"])), 1.0e-12),
            "rmse": rmse,
            "mae": mae,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)
high = df[df["vmax_obs"] >= HIGH_VMAX_CUT].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

worst20 = high.head(20).copy()
best20 = high.tail(20).copy().sort_values("rmse").reset_index(drop=True)

summary_vars = [
    "vmax_obs", "r_max", "rt", "rt_over_rmax", "u_t", "rt_ut",
    "vbar2_t", "vbar_t", "bulge_frac_t", "disk_frac_t", "gas_frac_t",
    "V_flat_pred", "Vflat_over_vmax", "rmse", "mae"
]

summary_df = pd.DataFrame({
    "quantity": summary_vars,
    "worst20_median": [float(worst20[c].median()) for c in summary_vars],
    "best20_median": [float(best20[c].median()) for c in summary_vars],
})
summary_df["worst_minus_best"] = summary_df["worst20_median"] - summary_df["best20_median"]

print()
print("=" * 60)
print("HIGH-END FAILURE AUDIT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("=" * 60)
print("WORST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(worst20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

print()
print("=" * 60)
print("BEST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(best20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_failure_summary.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "high_end_worst20.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "high_end_best20.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "high_end_full_audit.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_failure_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END FAILURE AUDIT SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\nWORST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))
    f.write("\n\nBEST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(best20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

RUNNING FAILURE-MODE AUDIT
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END FAILURE AUDIT SUMMARY
       quantity  worst20_median  best20_median  worst_minus_best
       vmax_obs      274.000000     186.000000         88.000000
          r_max       44.215000      26.930000         17.285000
             rt        4.236476       3.368903          0.867573
   rt_over_rmax        0.154626       0.124820          0.029805
            u_t        0.097594       0.084961          0.012633
          rt_ut        0.423070       0.275523          0.147548
        vbar2_t    77995.021748   35983.678528      42011.343220
         vbar_t      279.272915     189.684292         89.588623
   bulge_frac_t        0.281828       0.000000          0.281828
    disk_frac_t        0.717890       0.987273         -0.269383
     gas_frac_t        0.000091       

In [ ]:
# ============================================================
# TRANSITION COMPACTNESS CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Compactness correction:
#   c_t = Vbar^2(rt) / rt
#   V_flat,corr^2 = V_flat,base^2 * [1 + eta * (c_t / c0)^delta]
#
# Goal:
#   improve the high-vmax tail without reopening the model.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_transition_compactness_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00]
C0_LIST    = [5_000.0, 10_000.0, 20_000.0, 40_000.0, 80_000.0]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING COMPACTNESS-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vbar2 = np.maximum(
            rot["vgas"] * np.abs(rot["vgas"])
            + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
            + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
            0.0
        )
        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        c_t = vbar2_t / max(rt, 1.0e-12)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "c_t": c_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for c0 in C0_LIST:
            rmses = []
            vmaxs = []

            for _, row in df.iterrows():
                correction = 1.0 + eta * (row["c_t"] / c0) ** delta
                V_flat_pred = row["V_flat_base"] * np.sqrt(correction)
                V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

                V_obs_pred = interp1d(
                    row["r_grid"], V_theory,
                    kind="linear", bounds_error=False, fill_value="extrapolate"
                )(row["r_obs"])

                rmse = safe_rmse(row["v_obs"], V_obs_pred)
                rmses.append(rmse)
                vmaxs.append(row["vmax_obs"])

            rmses = np.asarray(rmses, float)
            vmaxs = np.asarray(vmaxs, float)

            high_mask = vmaxs >= 150.0
            high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

            summary_rows.append({
                "eta": eta,
                "delta": delta,
                "c0": c0,
                "median_rmse": float(np.median(rmses)),
                "p90_rmse": float(np.percentile(rmses, 90)),
                "high_bin_median_rmse": high_med,
            })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("COMPACTNESS-CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "compactness_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "compactness_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("COMPACTNESS-CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING COMPACTNESS-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

COMPACTNESS-CORRECTION SUMMARY
 eta  delta      c0  median_rmse  p90_rmse  high_bin_median_rmse
0.20   0.75 80000.0    20.492094 77.523220             63.289288
0.10   0.75 40000.0    20.475352 77.384626             63.452850
0.05   0.25 10000.0    20.720630 77.126245             63.592638
0.15   0.75 80000.0    20.465813 77.402844             63.598978
0.05   1.00 20000.0    20.421821 77.538450             63.623908
0.10   1.00 40000.0    20.421821 77.538450             63.623908
0.20   1.00 80000.0    20.421821 77.538450             63.623908
0.15   1.00 40000.0    20.438896 77.450237             63.664564
0.30   1.00 80000.0    20.438896 77.450237             63.664564
0.05   0.75 20000.0    20.461319 77.411665             63.669028
0.05   0.50 10000.0    20.5

In [ ]:
# ============================================================
# TWO-POPULATION NORMALIZATION TEST
# ------------------------------------------------------------
# Fixed branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = C_pop * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Split by bulge fraction at rt:
#   f_bulge(rt) = Vbulge^2(rt) / Vbar^2(rt)
#
# We test several thresholds and fit ONLY the normalization
# separately for low-bulge and high-bulge populations.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_two_population_normalization")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
ALPHA  = 0.175
BETA   = 0.55

# Bulge-fraction thresholds to try
THRESH_LIST = [0.05, 0.10, 0.20, 0.30, 0.40]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

def fit_C(subdf):
    x = subdf["carrier"].values.astype(float)
    y = subdf["V_flat_fit"].values.astype(float) ** 2
    m = np.isfinite(x) & np.isfinite(y) & (x > 0)
    return float(np.sum(x[m] * y[m]) / np.sum(x[m] ** 2))

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING TWO-POP TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2 = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
        vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
        vbul2 = np.maximum(UPS_BUL * rot["vbul"] * np.abs(rot["vbul"]), 0.0)
        vbar2 = vgas2 + vdisk2 + vbul2

        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_interp = interp1d(rot["r"], vbul2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        vbul2_t = max(float(vbul2_interp(rt)), 0.0)

        f_bulge_t = vbul2_t / vbar2_t if vbar2_t > 0 else 0.0
        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "f_bulge_t": f_bulge_t,
            "carrier": carrier,
            "V_flat_fit": float(np.max(rot["vobs"])),  # same target convention used in these tests
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for thresh in THRESH_LIST:
    low = df[df["f_bulge_t"] < thresh].copy()
    high = df[df["f_bulge_t"] >= thresh].copy()

    if len(low) < 10 or len(high) < 10:
        continue

    C_low = fit_C(low)
    C_high = fit_C(high)

    eval_rows = []
    for _, row in df.iterrows():
        C_use = C_low if row["f_bulge_t"] < thresh else C_high
        V_flat_pred = np.sqrt(max(C_use * row["carrier"], 0.0))
        V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

        V_obs_pred = interp1d(
            row["r_grid"], V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(row["r_obs"])

        rmse = safe_rmse(row["v_obs"], V_obs_pred)

        eval_rows.append({
            "vmax_obs": row["vmax_obs"],
            "rmse": rmse,
        })

    eval_df = pd.DataFrame(eval_rows)
    high_mask = eval_df["vmax_obs"] >= 150.0

    summary_rows.append({
        "thresh": thresh,
        "n_lowbulge": int(len(low)),
        "n_highbulge": int(len(high)),
        "C_lowbulge": C_low,
        "C_highbulge": C_high,
        "C_ratio_high_over_low": C_high / C_low if C_low != 0 else np.nan,
        "median_rmse": float(eval_df["rmse"].median()),
        "p90_rmse": float(np.percentile(eval_df["rmse"], 90)),
        "high_bin_median_rmse": float(eval_df.loc[high_mask, "rmse"].median()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("TWO-POPULATION NORMALIZATION SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "two_population_normalization_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "two_population_normalization_summary.txt"), "w", encoding="utf-8") as f:
    f.write("TWO-POPULATION NORMALIZATION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING TWO-POP TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

TWO-POPULATION NORMALIZATION SUMMARY
 thresh  n_lowbulge  n_highbulge  C_lowbulge  C_highbulge  C_ratio_high_over_low  median_rmse  p90_rmse  high_bin_median_rmse
   0.40         156           19  146.447719   203.300229               1.388210    20.603011 76.133411             61.506610
   0.30         155           20  147.671731   195.243356               1.322144    20.699052 76.818147             61.526319
   0.05         144           31  142.754414   174.181652               1.220149    20.969344 78.742874             63.592373
   0.20         151           24  141.596637   186.750763               1.318893    21.522831 79.221134             63.731267
   0.10         146           29  138.148873   182.260437               1.319305    21.745393 80.295051             64.23

In [ ]:
# ============================================================
# DERIVED SQRT-BRIDGE TEST
# ------------------------------------------------------------
# Field:
#   (1/r^2) d/dr (r^2 dm/dr) - (1/Rs^2)(m-m_inf) = -A rho_eff
#
# Define:
#   u(r) = m(r) - m_inf
#   U(r) = ∫_0^r u(s) ds
#
# Derived observable law:
#   g_obs(r) = K * U(r) / r
# so
#   V(r)^2 = K * U(r)
#   V(r)   = sqrt(K * U(r))
#
# One global K across the full catalog.
# Fixed source branch and fixed Rs = 1.5.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_derived_sqrt_bridge_test")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

# ------------------------------------------------------------
# BUILD FIELD TABLE
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()

rows = []
fails = []

print("=" * 60)
print("BUILDING SQRT-BRIDGE FIELD TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        if not np.isfinite(np.max(U)) or np.max(U) <= 0:
            raise RuntimeError("Invalid U profile")

        rows.append({
            "galaxy": galaxy,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": float(np.max(U)),
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

field_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(field_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

# ------------------------------------------------------------
# FIT GLOBAL K FROM V^2 = K U
# ------------------------------------------------------------

num = 0.0
den = 0.0

for _, row in field_df.iterrows():
    U_obs = interp1d(
        row["r_grid"], row["U_grid"],
        kind="linear", bounds_error=False, fill_value="extrapolate"
    )(row["r_obs"])

    m = np.isfinite(U_obs) & np.isfinite(row["v_obs"]) & (U_obs > 0)
    if np.sum(m) == 0:
        continue

    x = U_obs[m]
    y = row["v_obs"][m] ** 2
    num += np.sum(x * y)
    den += np.sum(x * x)

if den <= 0:
    raise RuntimeError("Could not fit global K")

K_global = num / den

# ------------------------------------------------------------
# EVALUATE
# ------------------------------------------------------------

res_rows = []

for _, row in field_df.iterrows():
    V_theory = np.sqrt(np.maximum(K_global * row["U_grid"], 0.0))
    V_obs_pred = interp1d(
        row["r_grid"], V_theory,
        kind="linear", bounds_error=False, fill_value="extrapolate"
    )(row["r_obs"])

    rmse = safe_rmse(row["v_obs"], V_obs_pred)
    mae = safe_mae(row["v_obs"], V_obs_pred)
    chi2_mean = float(np.mean(((row["v_obs"] - V_obs_pred) / row["e_obs"]) ** 2))

    obs_shape = row["v_obs"] / max(float(np.max(row["v_obs"])), 1.0e-12)
    pred_shape = V_obs_pred / max(float(np.max(V_obs_pred)), 1.0e-12)
    shape_rmse = safe_rmse(obs_shape, pred_shape)

    V_flat_pred = float(np.sqrt(max(K_global * row["U_inf"], 0.0)))

    res_rows.append({
        "galaxy": row["galaxy"],
        "n_points": len(row["r_obs"]),
        "vmax_obs": float(np.max(row["v_obs"])),
        "U_inf": row["U_inf"],
        "V_flat_pred": V_flat_pred,
        "rmse": rmse,
        "mae": mae,
        "chi2_mean": chi2_mean,
        "shape_rmse": shape_rmse,
    })

res_df = pd.DataFrame(res_rows)

summary = {
    "successful_galaxies": int(len(res_df)),
    "failed_galaxies": int(len(fail_df)),
    "K_global": float(K_global),
    "median_rmse": float(res_df["rmse"].median()),
    "mean_rmse": float(res_df["rmse"].mean()),
    "p90_rmse": float(np.percentile(res_df["rmse"], 90)),
    "p95_rmse": float(np.percentile(res_df["rmse"], 95)),
    "median_mae": float(res_df["mae"].median()),
    "median_chi2_mean": float(res_df["chi2_mean"].median()),
    "median_shape_rmse": float(res_df["shape_rmse"].median()),
}

res_df["vmax_bin"] = pd.cut(
    res_df["vmax_obs"],
    bins=[-np.inf, 80, 150, np.inf],
    labels=["low", "mid", "high"]
)

bin_summary = (
    res_df.groupby("vmax_bin", observed=True)[["rmse", "mae", "chi2_mean", "shape_rmse"]]
    .median()
    .reset_index()
)

best20 = res_df.sort_values("rmse").head(20).reset_index(drop=True)
worst20 = res_df.sort_values("rmse", ascending=False).head(20).reset_index(drop=True)

print()
print("=" * 60)
print("DERIVED SQRT-BRIDGE SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:22s} = {v}")

print()
print("=" * 60)
print("MEDIAN METRICS BY VMAX BIN")
print("=" * 60)
print(bin_summary.to_string(index=False))

print()
print("=" * 60)
print("BEST 20 BY RMSE")
print("=" * 60)
print(best20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("=" * 60)
print("WORST 20 BY RMSE")
print("=" * 60)
print(worst20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

res_df.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_results.csv"), index=False)
fail_df.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_failures.csv"), index=False)
bin_summary.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_bin_summary.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_best20.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_worst20.csv"), index=False)

with open(os.path.join(OUTDIR, "derived_sqrt_bridge_summary.txt"), "w", encoding="utf-8") as f:
    f.write("DERIVED SQRT-BRIDGE SUMMARY\n")
    f.write("=" * 60 + "\n")
    for k, v in summary.items():
        f.write(f"{k:22s} = {v}\n")
    f.write("\nMEDIAN METRICS BY VMAX BIN\n")
    f.write("=" * 60 + "\n")
    f.write(bin_summary.to_string(index=False))
    f.write("\n\nBEST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(best20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))
    f.write("\n\nWORST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING SQRT-BRIDGE FIELD TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

DERIVED SQRT-BRIDGE SUMMARY
successful_galaxies    = 175
failed_galaxies        = 0
K_global               = 36582.793628590174
median_rmse            = 43.36683077126389
mean_rmse              = 57.61969794883952
p90_rmse               = 121.34847005642484
p95_rmse               = 148.87374190105436
median_mae             = 41.33523616131542
median_chi2_mean       = 108.24295608329402
median_shape_rmse      = 0.1035927421678175

MEDIAN METRICS BY VMAX BIN
vmax_bin      rmse       mae  chi2_mean  shape_rmse
     low 41.847865 39.337542 149.385139    0.093540
     mid 28.832875 28.342812  42.857838    0.073662
    high 77.419750 65.099313 212.123979    0.147856

BEST 20 BY RMSE
     galaxy  vmax_obs  V_flat_pred      rmse       mae  shape_rmse
   UGC04483      24.3    

In [ ]:
import numpy as np
import glob
import os
import pandas as pd
from scipy.optimize import minimize_scalar
from numpy.linalg import lstsq

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 80

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = np.isfinite(r) & np.isfinite(vo) & np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) & (r > 0) & (vo > 0)
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vb2 = np.maximum(vg**2 + vd**2 + vb**2, 0.0)
    gb = vb2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vb2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    Phi = G * Mtot / np.maximum(Rmax, EPS)

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": np.maximum(gb, EPS),
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Phi": Phi,
        "Cgeom": Cgeom,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
        "logPhi": np.log10(Phi),
        "logCgeom": np.log10(Cgeom),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0/(P-1.0))
    return np.maximum(r * Mp, EPS)**QEXP

galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Step 1: best-F0 per galaxy + sign-class
# ------------------------------------------------------------
LOGF0_MIN = -2.0
LOGF0_MAX = 12.0

rows = []

for g in galaxies:
    r = g["r"]
    vo = g["vo"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    def objective(logF0):
        F0 = 10.0**logF0
        qshape = solve_qshape(r, source, F0)
        C = np.sum((vo**2) * qshape) / np.maximum(np.sum(qshape**2), EPS)
        vp = np.sqrt(np.maximum(C * qshape, 0.0))
        return np.sqrt(np.mean((vo - vp)**2))

    res = minimize_scalar(
        objective,
        bounds=(LOGF0_MIN, LOGF0_MAX),
        method="bounded",
        options={"xatol": 1e-3}
    )

    best_logF0 = float(res.x)
    best_F0 = 10.0**best_logF0
    qshape = solve_qshape(r, source, best_F0)
    C = np.sum((vo**2) * qshape) / np.maximum(np.sum(qshape**2), EPS)
    vp = np.sqrt(np.maximum(C * qshape, 0.0))

    rmse = np.sqrt(np.mean((vo - vp)**2))

    n = len(vo)
    i1 = max(1, n // 3)
    i2 = max(i1 + 1, 2 * n // 3)
    Vscale = max(np.max(vo), EPS)

    s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
    s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
    s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

    if g["logM"] >= 10.8:
        neg_count = int(s_in < -0.02) + int(s_mid < -0.02) + int(s_out < -0.02)
        pos_count = int(s_in >  0.02) + int(s_mid >  0.02) + int(s_out >  0.02)
        if neg_count >= 2:
            sign_class = "neg"
        elif pos_count >= 2:
            sign_class = "pos"
        else:
            sign_class = "mixed"
    else:
        sign_class = "lowmass"

    rows.append({
        "name": g["name"],
        "logM": g["logM"],
        "logPhi": g["logPhi"],
        "logCgeom": g["logCgeom"],
        "logF0": best_logF0,
        "rmse": rmse,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "sign_class": sign_class,
    })

fit_df = pd.DataFrame(rows)

print("\nCLASS COUNTS")
print(fit_df["sign_class"].value_counts().to_string())

# ------------------------------------------------------------
# Step 2: fit separate logF0 laws by class
# ------------------------------------------------------------
coef_by_class = {}

for cls in ["lowmass", "mixed", "neg", "pos"]:
    sub = fit_df[fit_df["sign_class"] == cls].copy()
    if len(sub) < 3:
        continue
    X = np.column_stack([
        sub["logM"].values,
        sub["logPhi"].values,
        sub["logCgeom"].values,
        np.ones(len(sub))
    ])
    y = sub["logF0"].values
    coef, _, _, _ = lstsq(X, y, rcond=None)
    coef_by_class[cls] = coef

print("\nPIECEWISE F0 LAWS")
for cls, coef in coef_by_class.items():
    a, b, c, d = coef
    print(f"{cls}: a={a:.4f}, b={b:.4f}, c={c:.4f}, d={d:.4f}")

# ------------------------------------------------------------
# Step 3: global test of piecewise law
# For high-mass galaxies, choose law by simple nearest-centroid in (logPhi,logCgeom)
# using the sign classes found above. Low-mass always uses lowmass law.
# ------------------------------------------------------------
centroids = {}
for cls in ["mixed", "neg", "pos"]:
    sub = fit_df[fit_df["sign_class"] == cls]
    if len(sub) >= 1:
        centroids[cls] = (
            sub["logPhi"].mean(),
            sub["logCgeom"].mean()
        )

def choose_highmass_class(logPhi, logCgeom):
    best_cls = None
    best_d2 = 1e99
    for cls, (mu_phi, mu_cg) in centroids.items():
        d2 = (logPhi - mu_phi)**2 + (logCgeom - mu_cg)**2
        if d2 < best_d2:
            best_d2 = d2
            best_cls = cls
    return best_cls

# reload galaxy objects for forward test
gal_map = {g["name"]: g for g in galaxies}

test_rows = []
num = 0.0
den = 0.0
raw = []

for _, row in fit_df.iterrows():
    g = gal_map[row["name"]]
    r = g["r"]
    vo = g["vo"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    if g["logM"] < 10.8:
        cls = "lowmass"
    else:
        cls = choose_highmass_class(g["logPhi"], g["logCgeom"])

    if cls not in coef_by_class:
        cls = "lowmass"

    a, b, c, d = coef_by_class[cls]
    logF0_pred = a*g["logM"] + b*g["logPhi"] + c*g["logCgeom"] + d
    F0 = 10.0**logF0_pred

    qshape = solve_qshape(r, source, F0)
    y = vo**2
    num += np.sum(y * qshape)
    den += np.sum(qshape**2)
    raw.append((g, qshape, cls, F0))

C_global = num / np.maximum(den, EPS)

for g, qshape, cls, F0 in raw:
    vo = g["vo"]
    vp = np.sqrt(np.maximum(C_global * qshape, 0.0))
    rmse = np.sqrt(np.mean((vo - vp)**2))
    mae = np.mean(np.abs(vo - vp))

    n = len(vo)
    i1 = max(1, n // 3)
    i2 = max(i1 + 1, 2 * n // 3)
    Vscale = max(np.max(vo), EPS)

    s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
    s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
    s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

    test_rows.append({
        "name": g["name"],
        "logM": g["logM"],
        "logPhi": g["logPhi"],
        "logCgeom": g["logCgeom"],
        "assigned_class": cls,
        "F0_pred": F0,
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
    })

test_df = pd.DataFrame(test_rows)
hi = test_df["logM"] >= 10.8

summary = {
    "global_C": C_global,
    "median_rmse": test_df["rmse"].median(),
    "mean_rmse": test_df["rmse"].mean(),
    "median_rmse_hiM": test_df.loc[hi, "rmse"].median() if np.any(hi) else np.nan,
    "mean_rmse_hiM": test_df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan,
    "inner": test_df["s_in"].median(),
    "mid": test_df["s_mid"].median(),
    "outer": test_df["s_out"].median(),
    "corr_logM": corr(test_df["logM"], test_df["rmse"]),
}

print("\nPIECEWISE GLOBAL TEST SUMMARY")
print(summary)

print("\nWORST HIGH-MASS GALAXIES UNDER PIECEWISE GLOBAL LAW")
print(
    test_df.loc[hi, ["name","assigned_class","logM","logPhi","logCgeom","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

CLASS COUNTS
sign_class
lowmass    96
mixed      33
neg        11
pos         3

PIECEWISE F0 LAWS
lowmass: a=0.4173, b=0.8618, c=2.1866, d=-1.2750
mixed: a=1.8218, b=-1.3744, c=1.8091, d=-5.6290
neg: a=2.9955, b=-2.2718, c=2.7334, d=-15.7957
pos: a=-0.2907, b=-0.9601, c=17.5944, d=4.2275

PIECEWISE GLOBAL TEST SUMMARY
{'global_C': np.float64(0.0004228701813810393), 'median_rmse': 76.642249886159, 'mean_rmse': np.float64(90.35823562405315), 'median_rmse_hiM': 140.0242110688724, 'mean_rmse_hiM': np.float64(144.40485341850123), 'inner': -0.46368469435724863, 'mid': -0.7018427655444298, 'outer': -0.7813307350050777, 'corr_logM': np.float64(0.7965959043495299)}

WORST HIGH-MASS GALAXIES UNDER PIECEWISE GLOBAL LAW
                  name assigned_class      logM   logPhi  logCgeom       rmse      s_in     s_mid     s_out
   UGC02487_rotmod.dat            pos 11.647878 4.376290  0.727035 307.700521 -0.864094 -0.818755 -0.730587
   UGC11914_rotmod.dat          mixed 11.05

In [ ]:
import numpy as np
import glob
import os
import pandas as pd
from scipy.optimize import minimize_scalar
from numpy.linalg import lstsq

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 80

DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = np.isfinite(r) & np.isfinite(vo) & np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) & (r > 0) & (vo > 0)
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vb2 = np.maximum(vg**2 + vd**2 + vb**2, 0.0)
    gb = vb2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vb2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    Phi = G * Mtot / np.maximum(Rmax, EPS)

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": np.maximum(gb, EPS),
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Phi": Phi,
        "Cgeom": Cgeom,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
        "logPhi": np.log10(Phi),
        "logRmax": np.log10(Rmax),
        "logR50": np.log10(R50),
        "logCgeom": np.log10(Cgeom),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0/(P-1.0))
    return np.maximum(r * Mp, EPS)**QEXP

galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

LOGF0_MIN = -2.0
LOGF0_MAX = 12.0

rows = []
for g in galaxies:
    r = g["r"]
    vo = g["vo"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    def objective(logF0):
        F0 = 10.0**logF0
        qshape = solve_qshape(r, source, F0)
        C = np.sum((vo**2) * qshape) / np.maximum(np.sum(qshape**2), EPS)
        vp = np.sqrt(np.maximum(C * qshape, 0.0))
        return np.sqrt(np.mean((vo - vp)**2))

    res = minimize_scalar(objective, bounds=(LOGF0_MIN, LOGF0_MAX), method="bounded", options={"xatol":1e-3})
    logF0 = float(res.x)

    rows.append({
        "name": g["name"],
        "logM": g["logM"],
        "logPhi": g["logPhi"],
        "logRmax": g["logRmax"],
        "logR50": g["logR50"],
        "logCgeom": g["logCgeom"],
        "logF0": logF0,
        "rmse": float(res.fun),
    })

df = pd.DataFrame(rows)
hi = df["logM"] >= 10.8
dfh = df.loc[hi].copy()

print("\nHIGH-MASS SAMPLE SIZE")
print(len(dfh))

# candidate one-variable invariants
cands = {
    "logM": dfh["logM"],
    "logPhi": dfh["logPhi"],
    "logRmax": dfh["logRmax"],
    "logR50": dfh["logR50"],
    "logCgeom": dfh["logCgeom"],
    "logM_plus_logRmax": dfh["logM"] + dfh["logRmax"],
    "logM_minus_logR50": dfh["logM"] - dfh["logR50"],
    "logM_plus_logCgeom": dfh["logM"] + dfh["logCgeom"],
    "logPhi_plus_logCgeom": dfh["logPhi"] + dfh["logCgeom"],
    "logRmax_minus_logR50": dfh["logRmax"] - dfh["logR50"],
}

rows_out = []
for name, x in cands.items():
    X = np.column_stack([x.values, np.ones(len(dfh))])
    y = dfh["logF0"].values
    coef, _, _, _ = lstsq(X, y, rcond=None)
    a, b = coef
    yhat = a*x.values + b
    resid = y - yhat
    rmse = np.sqrt(np.mean(resid**2))
    rows_out.append({
        "candidate": name,
        "slope": a,
        "intercept": b,
        "corr_with_logF0": corr(x.values, y),
        "fit_rmse_logF0": rmse
    })

out = pd.DataFrame(rows_out).sort_values("fit_rmse_logF0")

print("\nTOP 20 SIMPLE INVARIANTS FOR HIGH-MASS logF0")
print(out.to_string(index=False))

# best 2-variable fit among a few sensible pairs
pairs = [
    ("logM","logRmax"),
    ("logM","logR50"),
    ("logM","logCgeom"),
    ("logPhi","logCgeom"),
    ("logRmax","logR50"),
]

pair_rows = []
for a1, a2 in pairs:
    X = np.column_stack([dfh[a1].values, dfh[a2].values, np.ones(len(dfh))])
    y = dfh["logF0"].values
    coef, _, _, _ = lstsq(X, y, rcond=None)
    c1, c2, c0 = coef
    yhat = c1*dfh[a1].values + c2*dfh[a2].values + c0
    rmse = np.sqrt(np.mean((y-yhat)**2))
    pair_rows.append({
        "pair": f"{a1} + {a2}",
        "coef1": c1,
        "coef2": c2,
        "intercept": c0,
        "fit_rmse_logF0": rmse
    })

pair_df = pd.DataFrame(pair_rows).sort_values("fit_rmse_logF0")

print("\nBEST 2-VARIABLE FITS FOR HIGH-MASS logF0")
print(pair_df.to_string(index=False))

Usable galaxies: 143

HIGH-MASS SAMPLE SIZE
47

TOP 20 SIMPLE INVARIANTS FOR HIGH-MASS logF0
           candidate     slope  intercept  corr_with_logF0  fit_rmse_logF0
            logCgeom  3.295043   7.808478         0.424554        1.532476
logRmax_minus_logR50  3.295043   7.808478         0.424554        1.532476
             logRmax  2.749618   5.717775         0.379968        1.565646
  logM_plus_logCgeom  1.842982 -11.878305         0.350909        1.584958
              logPhi -1.924914  18.208426        -0.278851        1.625453
   logM_plus_logRmax  0.988219  -2.631403         0.252702        1.637656
   logM_minus_logR50  0.928963   0.381099         0.115284        1.681305
logPhi_plus_logCgeom  0.928963   5.366323         0.115284        1.681305
                logM  0.502754   4.326222         0.078035        1.687429
              logR50 -0.122256  10.063412        -0.016380        1.692363

BEST 2-VARIABLE FITS FOR HIGH-MASS logF0
             pair     coef1     coef2  i

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 80

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = np.isfinite(r) & np.isfinite(vo) & np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) & (r > 0) & (vo > 0)
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vb2 = np.maximum(vg**2 + vd**2 + vb**2, 0.0)
    gb = vb2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vb2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5 * Mtot), len(r) - 1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": np.maximum(gb, EPS),
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0 / (P - 1.0))
    return np.maximum(r * Mp, EPS)**QEXP

galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

alpha_vals = np.linspace(0.0, 3.0, 13)
beta_vals  = np.linspace(0.0, 4.0, 17)
k_vals     = np.logspace(-4, 4, 25)

results = []
best = None
best_score = np.inf

for alpha in alpha_vals:
    for beta in beta_vals:
        for k in k_vals:
            raw = []
            num = 0.0
            den = 0.0

            for g in galaxies:
                r = g["r"]
                gb = g["gb"]
                rho3d = g["rho3d"]

                g0 = 0.68 * g["gb_ref"]
                activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
                source = rho3d * activation

                F0 = k * (g["Rmax"]**alpha) * (g["Cgeom"]**beta)

                qshape = solve_qshape(r, source, F0)
                if not np.all(np.isfinite(qshape)):
                    continue

                y = g["vo"]**2
                num += np.sum(y * qshape)
                den += np.sum(qshape**2)

                raw.append((g, qshape, F0))

            if den <= 0 or len(raw) == 0:
                continue

            C = num / den

            rows = []
            for g, qshape, F0 in raw:
                vo = g["vo"]
                vp = np.sqrt(np.maximum(C * qshape, 0.0))

                rmse = np.sqrt(np.mean((vo - vp)**2))
                mae = np.mean(np.abs(vo - vp))

                n = len(vo)
                i1 = max(1, n // 3)
                i2 = max(i1 + 1, 2 * n // 3)
                Vscale = max(np.max(vo), EPS)

                s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
                s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
                s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

                rows.append({
                    "name": g["name"],
                    "rmse": rmse,
                    "mae": mae,
                    "s_in": s_in,
                    "s_mid": s_mid,
                    "s_out": s_out,
                    "logM": g["logM"],
                    "F0": F0,
                })

            df = pd.DataFrame(rows)

            hi = df["logM"] >= 10.8
            med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
            mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

            out = {
                "alpha": alpha,
                "beta": beta,
                "k": k,
                "global_C": C,
                "median_rmse": df["rmse"].median(),
                "mean_rmse": df["rmse"].mean(),
                "median_rmse_hiM": med_hi,
                "mean_rmse_hiM": mean_hi,
                "inner": df["s_in"].median(),
                "mid": df["s_mid"].median(),
                "outer": df["s_out"].median(),
                "corr_logM": corr(df["logM"], df["rmse"]),
            }
            results.append(out)

            score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
            if score < best_score:
                best_score = score
                best = out.copy()
                best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 GEOMETRY-BOUNDARY MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST GEOMETRY-BOUNDARY MODEL")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 GEOMETRY-BOUNDARY MODELS
 alpha  beta            k  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM
  2.75  0.50  4641.588834  0.012907    18.850995  28.577894        41.634775      52.056917 -0.042928 -0.069483 -0.071567   0.662431
  3.00  0.00 10000.000000  0.010736    20.435300  29.217360        41.671197      49.096855 -0.027068 -0.117438 -0.115438   0.615614
  2.75  0.25 10000.000000  0.012427    18.555405  28.336038        41.679438      50.285797 -0.023739 -0.075922 -0.070566   0.656120
  2.75  0.50 10000.000000  0.010699    20.916244  29.616100        41.789443      50.431294 -0.050891 -0.120285 -0.128727   0.612585
  3.00  0.25  4641.588834  0.011430    19.835375  29.015147        41.820211      50.347503 -0.048641 -0.108153 -0.109634   0.626488
  2.75  0.75  4641.588834  0.011345    20.040685  29.518306        42.185244      51.804193 -0.061419 -0.115205 -0.118361   0.624015
  2.50  0.75 10

In [ ]:
import numpy as np
import glob
import os
import pandas as pd
from scipy.optimize import minimize_scalar
from numpy.linalg import lstsq

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 80

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def safe_log10(x):
    return np.log10(np.maximum(x, EPS))

def fit_slope_loglog(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if np.sum(m) < 3:
        return np.nan
    X = np.column_stack([np.log10(x[m]), np.ones(np.sum(m))])
    Y = np.log10(y[m])
    coef, _, _, _ = lstsq(X, Y, rcond=None)
    return float(coef[0])

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = np.isfinite(r) & np.isfinite(vo) & np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) & (r > 0) & (vo > 0)
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    vbar = np.sqrt(vbar2)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    Phi = G * Mtot / np.maximum(Rmax, EPS)

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    # baryonic shape features
    i1 = max(3, len(r)//3)
    i2 = max(i1+2, 2*len(r)//3)

    inner_slope_vbar = fit_slope_loglog(r[:i1], vbar[:i1])
    inner_slope_gb   = fit_slope_loglog(r[:i1], gb[:i1])
    outer_slope_vbar = fit_slope_loglog(r[i2:], vbar[i2:])
    outer_slope_gb   = fit_slope_loglog(r[i2:], gb[i2:])

    jpk = int(np.argmax(vbar))
    Rpk = max(r[jpk], EPS)
    Vpk = max(vbar[jpk], EPS)
    Gpk = max(gb[jpk], EPS)

    # peak compactness / shape summaries
    peak_comp = Vpk**2 / Rpk
    span_ratio = Rmax / Rpk
    inner_mass_frac = Menc[min(np.searchsorted(r, 0.2*Rmax), len(r)-1)] / np.maximum(Mtot, EPS)

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Phi": Phi,
        "Cgeom": Cgeom,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
        "logPhi": np.log10(Phi),
        "logRmax": np.log10(Rmax),
        "logR50": np.log10(R50),
        "logCgeom": np.log10(Cgeom),
        "logRpk": np.log10(Rpk),
        "logVpk": np.log10(Vpk),
        "logGpk": np.log10(Gpk),
        "logPeakComp": np.log10(np.maximum(peak_comp, EPS)),
        "logSpanRatio": np.log10(np.maximum(span_ratio, EPS)),
        "inner_slope_vbar": inner_slope_vbar,
        "inner_slope_gb": inner_slope_gb,
        "outer_slope_vbar": outer_slope_vbar,
        "outer_slope_gb": outer_slope_gb,
        "logf20": np.log10(np.maximum(inner_mass_frac, EPS)),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0/(P-1.0))
    return np.maximum(r * Mp, EPS)**QEXP

galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

LOGF0_MIN = -2.0
LOGF0_MAX = 12.0

rows = []
for g in galaxies:
    r = g["r"]
    vo = g["vo"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    def objective(logF0):
        F0 = 10.0**logF0
        qshape = solve_qshape(r, source, F0)
        C = np.sum((vo**2) * qshape) / np.maximum(np.sum(qshape**2), EPS)
        vp = np.sqrt(np.maximum(C * qshape, 0.0))
        return np.sqrt(np.mean((vo - vp)**2))

    res = minimize_scalar(
        objective,
        bounds=(LOGF0_MIN, LOGF0_MAX),
        method="bounded",
        options={"xatol":1e-3}
    )

    rows.append({
        **{k:v for k,v in g.items() if not isinstance(v, np.ndarray)},
        "logF0": float(res.x),
        "rmse_bestF0": float(res.fun),
    })

df = pd.DataFrame(rows)
dfh = df[df["logM"] >= 10.8].copy()

print("\nHIGH-MASS SAMPLE SIZE")
print(len(dfh))

cands = [
    "logCgeom",
    "logRmax",
    "logR50",
    "logRpk",
    "logVpk",
    "logGpk",
    "logPeakComp",
    "logSpanRatio",
    "logf20",
    "inner_slope_vbar",
    "inner_slope_gb",
    "outer_slope_vbar",
    "outer_slope_gb",
    "logM",
    "logPhi",
]

one_var = []
for c in cands:
    x = dfh[c].values
    y = dfh["logF0"].values
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 5:
        continue
    X = np.column_stack([x[m], np.ones(np.sum(m))])
    coef, _, _, _ = lstsq(X, y[m], rcond=None)
    a, b = coef
    yhat = a*x[m] + b
    rmse = np.sqrt(np.mean((y[m]-yhat)**2))
    one_var.append({
        "candidate": c,
        "slope": a,
        "intercept": b,
        "corr_with_logF0": corr(x[m], y[m]),
        "fit_rmse_logF0": rmse
    })

one_df = pd.DataFrame(one_var).sort_values("fit_rmse_logF0")

print("\nTOP 20 ONE-VARIABLE PREDICTORS OF HIGH-MASS logF0")
print(one_df.head(20).to_string(index=False))

pairs = [
    ("logCgeom","logRmax"),
    ("logCgeom","logRpk"),
    ("logCgeom","inner_slope_vbar"),
    ("logCgeom","inner_slope_gb"),
    ("logRmax","logRpk"),
    ("logRmax","inner_slope_vbar"),
    ("logPeakComp","inner_slope_vbar"),
    ("logPeakComp","logSpanRatio"),
    ("logPhi","logCgeom"),
    ("logM","logCgeom"),
]

two_var = []
for a1, a2 in pairs:
    x1 = dfh[a1].values
    x2 = dfh[a2].values
    y = dfh["logF0"].values
    m = np.isfinite(x1) & np.isfinite(x2) & np.isfinite(y)
    if np.sum(m) < 5:
        continue
    X = np.column_stack([x1[m], x2[m], np.ones(np.sum(m))])
    coef, _, _, _ = lstsq(X, y[m], rcond=None)
    c1, c2, c0 = coef
    yhat = c1*x1[m] + c2*x2[m] + c0
    rmse = np.sqrt(np.mean((y[m]-yhat)**2))
    two_var.append({
        "pair": f"{a1} + {a2}",
        "coef1": c1,
        "coef2": c2,
        "intercept": c0,
        "fit_rmse_logF0": rmse
    })

two_df = pd.DataFrame(two_var).sort_values("fit_rmse_logF0")

print("\nBEST 2-VARIABLE PREDICTORS OF HIGH-MASS logF0")
print(two_df.to_string(index=False))

Usable galaxies: 143

HIGH-MASS SAMPLE SIZE
47

TOP 20 ONE-VARIABLE PREDICTORS OF HIGH-MASS logF0
       candidate     slope  intercept  corr_with_logF0  fit_rmse_logF0
    logSpanRatio  2.514284   8.370143         0.436414        1.522901
        logCgeom  3.295043   7.808478         0.424554        1.532476
          logf20  2.370191  10.730570         0.406229        1.546640
         logRmax  2.749618   5.717775         0.379968        1.565646
inner_slope_vbar -1.976925  10.670102        -0.282190        1.623801
  inner_slope_gb -0.988462   9.681639        -0.282190        1.623801
          logPhi -1.924914  18.208426        -0.278851        1.625453
          logGpk  0.818839   6.835275         0.172438        1.667236
     logPeakComp  0.818839   6.835275         0.172438        1.667236
          logRpk -1.181051  11.030233        -0.165683        1.669197
outer_slope_vbar -1.616444   9.284821        -0.149285        1.673623
  outer_slope_gb -0.808222   8.476599        -0.14

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 80

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def fit_slope_loglog(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if np.sum(m) < 3:
        return np.nan
    X = np.column_stack([np.log10(x[m]), np.ones(np.sum(m))])
    Y = np.log10(y[m])
    coef = np.linalg.lstsq(X, Y, rcond=None)[0]
    return float(coef[0])

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = np.isfinite(r) & np.isfinite(vo) & np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) & (r > 0) & (vo > 0)
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    vbar = np.sqrt(vbar2)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    jpk = int(np.argmax(vbar))
    Rpk = max(r[jpk], EPS)
    span_ratio = Rmax / Rpk

    i1 = max(3, len(r)//3)
    inner_slope_vbar = fit_slope_loglog(r[:i1], vbar[:i1])

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Rpk": Rpk,
        "Cgeom": Cgeom,
        "span_ratio": span_ratio,
        "inner_slope_vbar": inner_slope_vbar,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0/(P-1.0))
    return np.maximum(r * Mp, EPS)**QEXP

galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# Two candidate high-mass boundary laws from your diagnostics:
# 1) geometry law:
#    logF0 = 2.428536 logCgeom + 1.621166 logRmax + 5.874813
# 2) shape law:
#    logF0 = 2.660297 logRmax - 1.856998 inner_slope_vbar + 6.527452

models = {
    "geom_boundary": {"kfac_vals": np.logspace(-1, 1, 25)},
    "shape_boundary": {"kfac_vals": np.logspace(-1, 1, 25)},
}

results = []
best = None
best_score = np.inf

for model_name, meta in models.items():
    for kfac in meta["kfac_vals"]:
        raw = []
        num = 0.0
        den = 0.0

        for g in galaxies:
            r = g["r"]
            vo = g["vo"]
            gb = g["gb"]
            rho3d = g["rho3d"]

            g0 = 0.68 * g["gb_ref"]
            activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
            source = rho3d * activation

            if model_name == "geom_boundary":
                logF0 = (
                    np.log10(kfac)
                    + 2.428536 * np.log10(g["Cgeom"])
                    + 1.621166 * np.log10(g["Rmax"])
                    + 5.874813
                )
            else:
                logF0 = (
                    np.log10(kfac)
                    + 2.660297 * np.log10(g["Rmax"])
                    - 1.856998 * g["inner_slope_vbar"]
                    + 6.527452
                )

            F0 = 10.0**logF0

            qshape = solve_qshape(r, source, F0)
            if not np.all(np.isfinite(qshape)):
                continue

            y = vo**2
            num += np.sum(y * qshape)
            den += np.sum(qshape**2)
            raw.append((g, qshape, F0))

        if den <= 0 or len(raw) == 0:
            continue

        C = num / den

        rows = []
        for g, qshape, F0 in raw:
            vo = g["vo"]
            vp = np.sqrt(np.maximum(C * qshape, 0.0))
            rmse = np.sqrt(np.mean((vo - vp)**2))
            mae = np.mean(np.abs(vo - vp))

            n = len(vo)
            i1 = max(1, n // 3)
            i2 = max(i1 + 1, 2 * n // 3)
            Vscale = max(np.max(vo), EPS)

            s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
            s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
            s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

            rows.append({
                "name": g["name"],
                "logM": g["logM"],
                "F0": F0,
                "rmse": rmse,
                "mae": mae,
                "s_in": s_in,
                "s_mid": s_mid,
                "s_out": s_out,
            })

        df = pd.DataFrame(rows)
        hi = df["logM"] >= 10.8
        med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
        mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

        out = {
            "model": model_name,
            "kfac": kfac,
            "global_C": C,
            "median_rmse": df["rmse"].median(),
            "mean_rmse": df["rmse"].mean(),
            "median_rmse_hiM": med_hi,
            "mean_rmse_hiM": mean_hi,
            "inner": df["s_in"].median(),
            "mid": df["s_mid"].median(),
            "outer": df["s_out"].median(),
            "corr_logM": corr(df["logM"], df["rmse"]),
        }
        results.append(out)

        score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
        if score < best_score:
            best_score = score
            best = out.copy()
            best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 HIGH-MASS BOUNDARY-LAW TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST HIGH-MASS BOUNDARY LAW")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 HIGH-MASS BOUNDARY-LAW TESTS
         model     kfac  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM
shape_boundary 0.100000  0.006892    25.495078  38.467495        51.009286      63.006523 -0.063853 -0.203505 -0.252479   0.548433
shape_boundary 0.121153  0.006154    26.713425  40.536360        56.253241      66.212533 -0.069014 -0.232588 -0.283468   0.556157
shape_boundary 0.146780  0.005459    28.223602  42.681974        57.862717      69.646373 -0.075731 -0.256985 -0.310784   0.565427
shape_boundary 0.177828  0.004816    30.550997  44.842771        58.986671      73.173636 -0.088887 -0.276344 -0.337632   0.575076
shape_boundary 0.215443  0.004230    31.784598  46.966450        59.809593      76.681375 -0.099806 -0.306683 -0.368050   0.584238
shape_boundary 0.261016  0.003704    33.692801  49.009761        62.184653      80.077909 -0.110584 -0.326715 -0.396775   0.592309
shape_boundary 0.316228  

In [ ]:
import numpy as np
import glob
import os
import pandas as pd
from scipy.optimize import minimize_scalar
from numpy.linalg import lstsq

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 80

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5 * Mtot), len(r) - 1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2 * Rmax), len(r) - 1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": np.maximum(gb, EPS),
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
        "logRmax": np.log10(Rmax),
        "logR50": np.log10(R50),
        "logCgeom": np.log10(Cgeom),
        "logf20": np.log10(np.maximum(f20, EPS)),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0 / (P - 1.0))
    return np.maximum(r * Mp, EPS)**QEXP

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# STEP 1: Per-galaxy best F0 fit
# ------------------------------------------------------------
LOGF0_MIN = -2.0
LOGF0_MAX = 12.0

fit_rows = []

for g in galaxies:
    r = g["r"]
    vo = g["vo"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    def objective(logF0):
        F0 = 10.0**logF0
        qshape = solve_qshape(r, source, F0)
        C = np.sum((vo**2) * qshape) / np.maximum(np.sum(qshape**2), EPS)
        vp = np.sqrt(np.maximum(C * qshape, 0.0))
        return np.sqrt(np.mean((vo - vp)**2))

    res = minimize_scalar(
        objective,
        bounds=(LOGF0_MIN, LOGF0_MAX),
        method="bounded",
        options={"xatol": 1e-3}
    )

    fit_rows.append({
        "name": g["name"],
        "logM": g["logM"],
        "logRmax": g["logRmax"],
        "logR50": g["logR50"],
        "logCgeom": g["logCgeom"],
        "logf20": g["logf20"],
        "logF0": float(res.x),
        "rmse_bestF0": float(res.fun),
    })

fit_df = pd.DataFrame(fit_rows)

# ------------------------------------------------------------
# STEP 2: Fit exact global boundary law:
# log F0 = a log Rmax + b log Cgeom + c log f20 + d
# ------------------------------------------------------------
X = np.column_stack([
    fit_df["logRmax"].values,
    fit_df["logCgeom"].values,
    fit_df["logf20"].values,
    np.ones(len(fit_df))
])
y = fit_df["logF0"].values

coef, _, _, _ = lstsq(X, y, rcond=None)
a_fit, b_fit, c_fit, d_fit = coef

print("\nGLOBAL F0 LAW FIT")
print("log F0 = a log Rmax + b log Cgeom + c log f20 + d")
print(f"a (Rmax exponent)   = {a_fit:.6f}")
print(f"b (Cgeom exponent)  = {b_fit:.6f}")
print(f"c (f20 exponent)    = {c_fit:.6f}")
print(f"d (constant)        = {d_fit:.6f}")

# ------------------------------------------------------------
# STEP 3: Global forward test using the fitted law
# Also allow a small multiplicative scan around the fitted intercept
# ------------------------------------------------------------
kfac_vals = np.logspace(-1, 1, 41)

results = []
best = None
best_score = np.inf
best_df = None

for kfac in kfac_vals:
    raw = []
    num = 0.0
    den = 0.0

    for g in galaxies:
        r = g["r"]
        vo = g["vo"]
        gb = g["gb"]
        rho3d = g["rho3d"]

        g0 = 0.68 * g["gb_ref"]
        activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
        source = rho3d * activation

        logF0_pred = (
            np.log10(kfac)
            + a_fit * g["logRmax"]
            + b_fit * g["logCgeom"]
            + c_fit * g["logf20"]
            + d_fit
        )
        F0 = 10.0**logF0_pred

        qshape = solve_qshape(r, source, F0)
        if not np.all(np.isfinite(qshape)):
            continue

        yv = vo**2
        num += np.sum(yv * qshape)
        den += np.sum(qshape**2)

        raw.append((g, qshape, F0))

    if den <= 0 or len(raw) == 0:
        continue

    C_global = num / den

    rows = []
    for g, qshape, F0 in raw:
        vo = g["vo"]
        vp = np.sqrt(np.maximum(C_global * qshape, 0.0))

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "logRmax": g["logRmax"],
            "logCgeom": g["logCgeom"],
            "logf20": g["logf20"],
            "F0": F0,
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "kfac": kfac,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 GLOBAL TESTS FOR F0(Rmax,Cgeom,f20)")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST F0(Rmax,Cgeom,f20) LAW")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

GLOBAL F0 LAW FIT
log F0 = a log Rmax + b log Cgeom + c log f20 + d
a (Rmax exponent)   = 2.809608
b (Cgeom exponent)  = 1.761375
c (f20 exponent)    = 0.917497
d (constant)        = 4.013955

TOP 20 GLOBAL TESTS FOR F0(Rmax,Cgeom,f20)
    kfac  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM
0.100000  0.007516    28.893109  40.739031        59.904031      69.726923 -0.190000 -0.253627 -0.277399   0.638730
0.112202  0.007039    30.739935  42.467855        61.322660      72.454572 -0.199089 -0.275695 -0.299072   0.649848
0.125893  0.006573    32.306944  44.292217        64.523128      75.369194 -0.208884 -0.295351 -0.322165   0.662105
0.141254  0.006121    33.991460  46.179493        67.882282      78.415110 -0.216020 -0.311917 -0.343814   0.674688
0.158489  0.005685    35.858807  48.104469        71.159749      81.542293 -0.223292 -0.330252 -0.360154   0.687047
0.177828  0.005268    38.067436  50.046831    

In [ ]:
import numpy as np
import glob
import os
import pandas as pd
from scipy.optimize import minimize_scalar
from numpy.linalg import lstsq
from sklearn.decomposition import PCA

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = np.isfinite(r) & np.isfinite(vo) & np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) & (r > 0) & (vo > 0)
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    vbar = np.sqrt(vbar2)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar": vbar,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0/(P-1.0))
    return np.maximum(r * Mp, EPS)**QEXP

# -------- load --------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# -------- best per-galaxy F0 --------
LOGF0_MIN = -2.0
LOGF0_MAX = 12.0
fit_rows = []

for g in galaxies:
    r, vo, gb, rho3d = g["r"], g["vo"], g["gb"], g["rho3d"]
    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    def objective(logF0):
        F0 = 10.0**logF0
        qshape = solve_qshape(r, source, F0)
        C = np.sum((vo**2)*qshape) / np.maximum(np.sum(qshape**2), EPS)
        vp = np.sqrt(np.maximum(C*qshape, 0.0))
        return np.sqrt(np.mean((vo - vp)**2))

    res = minimize_scalar(objective, bounds=(LOGF0_MIN, LOGF0_MAX), method="bounded", options={"xatol":1e-3})

    fit_rows.append({
        "name": g["name"],
        "logF0": float(res.x),
        "rmse_bestF0": float(res.fun),
        "logM": g["logM"],
    })

fit_df = pd.DataFrame(fit_rows)

# -------- build baryonic profile features --------
# sample normalized baryonic curve at fixed x = r/Rmax
xgrid = np.linspace(0.08, 1.0, 12)
prof_rows = []

for g in galaxies:
    x = g["r"] / np.maximum(g["Rmax"], EPS)
    y = g["vbar"] / np.maximum(np.max(g["vbar"]), EPS)
    yg = np.interp(xgrid, x, y)
    row = {"name": g["name"]}
    for i, val in enumerate(yg):
        row[f"p{i}"] = val
    prof_rows.append(row)

prof_df = pd.DataFrame(prof_rows)
full = fit_df.merge(prof_df, on="name", how="inner")

# PCA on baryonic profile shape
Pcols = [f"p{i}" for i in range(len(xgrid))]
Xprof = full[Pcols].values
pca = PCA(n_components=3)
Z = pca.fit_transform(Xprof)

full["PC1"] = Z[:,0]
full["PC2"] = Z[:,1]
full["PC3"] = Z[:,2]

print("\nPCA EXPLAINED VARIANCE")
for i, frac in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"PC{i}: {frac:.4f}")

# fit logF0 = a*PC1 + b*PC2 + c*PC3 + d
X = np.column_stack([full["PC1"], full["PC2"], full["PC3"], np.ones(len(full))])
y = full["logF0"].values
coef, _, _, _ = lstsq(X, y, rcond=None)
a, b, c, d = coef

yhat = X @ coef
fit_rmse = np.sqrt(np.mean((y - yhat)**2))

print("\nPROFILE-BASED F0 LAW")
print("logF0 = a*PC1 + b*PC2 + c*PC3 + d")
print(f"a = {a:.6f}")
print(f"b = {b:.6f}")
print(f"c = {c:.6f}")
print(f"d = {d:.6f}")
print(f"fit_rmse_logF0 = {fit_rmse:.6f}")

# -------- forward global test --------
coef_map = {row["name"]: (row["PC1"], row["PC2"], row["PC3"]) for _, row in full.iterrows()}

raw = []
num = 0.0
den = 0.0

for g in galaxies:
    r, vo, gb, rho3d = g["r"], g["vo"], g["gb"], g["rho3d"]
    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    source = rho3d * activation

    pc1, pc2, pc3 = coef_map[g["name"]]
    logF0_pred = a*pc1 + b*pc2 + c*pc3 + d
    F0 = 10.0**logF0_pred

    qshape = solve_qshape(r, source, F0)
    yv = vo**2
    num += np.sum(yv * qshape)
    den += np.sum(qshape**2)
    raw.append((g, qshape, F0))

C_global = num / np.maximum(den, EPS)

rows = []
for g, qshape, F0 in raw:
    vo = g["vo"]
    vp = np.sqrt(np.maximum(C_global*qshape, 0.0))
    rmse = np.sqrt(np.mean((vo - vp)**2))

    n = len(vo)
    i1 = max(1, n//3)
    i2 = max(i1+1, 2*n//3)
    Vscale = max(np.max(vo), EPS)

    s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
    s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
    s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

    rows.append({
        "name": g["name"],
        "logM": g["logM"],
        "F0": F0,
        "rmse": rmse,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
    })

out_df = pd.DataFrame(rows)
hi = out_df["logM"] >= 10.8

print("\nPROFILE-BASED GLOBAL TEST SUMMARY")
print({
    "global_C": C_global,
    "median_rmse": out_df["rmse"].median(),
    "mean_rmse": out_df["rmse"].mean(),
    "median_rmse_hiM": out_df.loc[hi, "rmse"].median() if np.any(hi) else np.nan,
    "mean_rmse_hiM": out_df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan,
    "inner": out_df["s_in"].median(),
    "mid": out_df["s_mid"].median(),
    "outer": out_df["s_out"].median(),
    "corr_logM": corr(out_df["logM"], out_df["rmse"]),
})

print("\nWORST HIGH-MASS GALAXIES UNDER PROFILE-BASED F0 LAW")
print(
    out_df.loc[hi, ["name","logM","F0","rmse","s_in","s_mid","s_out"]]
          .sort_values("rmse", ascending=False)
          .head(20)
          .to_string(index=False)
)

Usable galaxies: 143

PCA EXPLAINED VARIANCE
PC1: 0.6865
PC2: 0.2330
PC3: 0.0527

PROFILE-BASED F0 LAW
logF0 = a*PC1 + b*PC2 + c*PC3 + d
a = 3.060499
b = 0.810772
c = -1.232846
d = 7.542680
fit_rmse_logF0 = 1.696152

PROFILE-BASED GLOBAL TEST SUMMARY
{'global_C': np.float64(0.007410867767284468), 'median_rmse': 30.866323920398344, 'mean_rmse': np.float64(41.66839907753546), 'median_rmse_hiM': 47.55299254515644, 'mean_rmse_hiM': np.float64(61.91702107769697), 'inner': -0.09494302391411051, 'mid': -0.18162587339182198, 'outer': -0.24905524002719814, 'corr_logM': np.float64(0.5045464295871818)}

WORST HIGH-MASS GALAXIES UNDER PROFILE-BASED F0 LAW
                  name      logM           F0       rmse      s_in     s_mid     s_out
   UGC11914_rotmod.dat 11.059532 4.151890e+06 154.380498 -0.607142 -0.489685 -0.401460
   UGC02487_rotmod.dat 11.647878 3.062240e+09 152.516774 -0.501943 -0.384560 -0.294266
    NGC5985_rotmod.dat 11.448616 1.276490e+07 149.727743 -0.574086 -0.464109 -0.418895


In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape(r, source, F0):
    Fsrc = cumulative_trapz((r**DELTA) * source, r)
    F = Fsrc + F0
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0 / (P - 1.0))
    return np.maximum(r * Mp, EPS)**QEXP

def kernel_integral(r, gb, g0, a, b):
    x = np.clip(r / np.maximum(r[-1], EPS), EPS, 1.0 - 1e-9)
    act = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
    w = (x**a) * ((1.0 - x)**b)
    integrand = w * act
    return np.trapz(integrand, r)

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan kernel parameters
# ------------------------------------------------------------
a_vals = np.linspace(-1.0, 4.0, 11)
b_vals = np.linspace(-0.5, 4.0, 10)
k_vals = np.logspace(-2, 4, 19)

results = []
best = None
best_score = np.inf
best_df = None

for a in a_vals:
    for b in b_vals:
        for K in k_vals:
            raw = []
            num = 0.0
            den = 0.0

            for g in galaxies:
                r = g["r"]
                vo = g["vo"]
                gb = g["gb"]
                rho3d = g["rho3d"]

                g0 = 0.68 * g["gb_ref"]
                activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
                source = rho3d * activation

                Iker = kernel_integral(r, gb, g0, a, b)
                F0 = K * np.maximum(Iker, EPS)

                qshape = solve_qshape(r, source, F0)
                if not np.all(np.isfinite(qshape)):
                    continue

                y = vo**2
                num += np.sum(y * qshape)
                den += np.sum(qshape**2)
                raw.append((g, qshape, F0))

            if den <= 0 or len(raw) == 0:
                continue

            C_global = num / den

            rows = []
            for g, qshape, F0 in raw:
                vo = g["vo"]
                vp = np.sqrt(np.maximum(C_global * qshape, 0.0))

                rmse = np.sqrt(np.mean((vo - vp)**2))
                mae = np.mean(np.abs(vo - vp))

                n = len(vo)
                i1 = max(1, n // 3)
                i2 = max(i1 + 1, 2 * n // 3)
                Vscale = max(np.max(vo), EPS)

                s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
                s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
                s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

                rows.append({
                    "name": g["name"],
                    "logM": g["logM"],
                    "F0": F0,
                    "rmse": rmse,
                    "mae": mae,
                    "s_in": s_in,
                    "s_mid": s_mid,
                    "s_out": s_out,
                })

            df = pd.DataFrame(rows)

            hi = df["logM"] >= 10.8
            med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
            mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

            out = {
                "a": a,
                "b": b,
                "K": K,
                "global_C": C_global,
                "median_rmse": df["rmse"].median(),
                "mean_rmse": df["rmse"].mean(),
                "median_rmse_hiM": med_hi,
                "mean_rmse_hiM": mean_hi,
                "inner": df["s_in"].median(),
                "mid": df["s_mid"].median(),
                "outer": df["s_out"].median(),
                "corr_logM": corr(df["logM"], df["rmse"]),
            }
            results.append(out)

            score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
            if score < best_score:
                best_score = score
                best = out.copy()
                best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 KERNEL-FUNCTIONAL F0 MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST KERNEL-FUNCTIONAL F0 LAW")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143


/tmp/ipykernel_4927/3132137649.py:99: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(integrand, r)



TOP 20 KERNEL-FUNCTIONAL F0 MODELS
   a    b            K  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM    inner      mid    outer  corr_logM
-1.0 -0.5 10000.000000  0.014813    25.753578  34.548774        49.098211      58.356690 0.114400 0.093466 0.085991   0.564641
-0.5 -0.5 10000.000000  0.014815    25.804818  34.554361        49.187719      58.408366 0.113709 0.093044 0.085303   0.564888
 0.0 -0.5 10000.000000  0.014815    25.820614  34.554391        49.212298      58.422261 0.113419 0.092869 0.085096   0.564995
 0.5 -0.5 10000.000000  0.014816    25.827271  34.553888        49.221307      58.427558 0.113279 0.092785 0.085016   0.565050
 1.0 -0.5 10000.000000  0.014816    25.830788  34.553425        49.225305      58.430104 0.113201 0.092740 0.084978   0.565083
 1.5 -0.5 10000.000000  0.014816    25.832936  34.553054        49.227326      58.431528 0.113153 0.092711 0.084956   0.565104
 2.0 -0.5 10000.000000  0.014816    25.834376  34.552758        49.228453  

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_from_eps(r, source, eps_outer):
    """
    Impose M'(Rmax)=eps_outer.
    Then:
      F(Rmax) = Rmax^DELTA * |eps|^(P-2) * eps
      F(r) = F(Rmax) - integral_r^Rmax s^DELTA S(s) ds
           = F_end - [I_end - I(r)]
    """
    I = cumulative_trapz((r**DELTA) * source, r)
    I_end = I[-1]

    F_end = (r[-1]**DELTA) * (np.abs(eps_outer)**(P - 2.0)) * eps_outer
    F = F_end - (I_end - I)

    # require nonnegative base for real Mp if using positive branch
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base**(1.0 / (P - 1.0))

    qshape = np.maximum(r * Mp, EPS)**QEXP
    F0_eff = F_end - I_end
    return qshape, F0_eff

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan outer boundary derivative eps and a global prefactor on eps
# eps is in field-gradient units of the solver, so we use log scan.
# Positive eps only for now because the successful branch has been positive.
# ------------------------------------------------------------
eps_vals = np.logspace(-8, 2, 31)

results = []
best = None
best_score = np.inf
best_df = None

for eps_outer in eps_vals:
    raw = []
    num = 0.0
    den = 0.0

    for g in galaxies:
        r = g["r"]
        vo = g["vo"]
        gb = g["gb"]
        rho3d = g["rho3d"]

        g0 = 0.68 * g["gb_ref"]
        activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
        source = rho3d * activation

        qshape, F0_eff = solve_qshape_from_eps(r, source, eps_outer)

        if not np.all(np.isfinite(qshape)):
            continue

        y = vo**2
        num += np.sum(y * qshape)
        den += np.sum(qshape**2)

        raw.append((g, qshape, F0_eff))

    if den <= 0 or len(raw) == 0:
        continue

    C_global = num / den

    rows = []
    for g, qshape, F0_eff in raw:
        vo = g["vo"]
        vp = np.sqrt(np.maximum(C_global * qshape, 0.0))

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "F0_eff": F0_eff,
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "eps_outer": eps_outer,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
        "median_log10_F0eff": np.median(np.log10(np.maximum(np.abs(df["F0_eff"]), EPS))),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 OUTER-BOUNDARY-CONDITION MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST OUTER-BOUNDARY MODEL")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0_eff","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 OUTER-BOUNDARY-CONDITION MODELS
   eps_outer     global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM  median_log10_F0eff
1.000000e-08 2.624654e+10    89.028221 114.570451       201.268886     201.886036 -0.586354 -0.878691 -0.695962   0.894441            8.442175
2.154435e-08 1.315392e+10    89.201873 114.738869       201.443271     202.058482 -0.588386 -0.879662 -0.699180   0.894436            8.442175
4.641589e-08 6.592448e+09    89.324847 114.858152       201.566696     202.180558 -0.589825 -0.880742 -0.700328   0.894432            8.442175
1.000000e-07 3.304019e+09    89.411925 114.942625       201.654061     202.266980 -0.590843 -0.882701 -0.700968   0.894429            8.442175
2.154435e-07 1.655924e+09    89.473582 115.002441       201.715904     202.328162 -0.591564 -0.884087 -0.701420   0.894427            8.442175
4.641589e-07 8.299258e+08    89.517237 115.044795       201.759682     202.371474

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_robin(r, source, lam):
    """
    Solve the integrated nonlinear transport law with Robin boundary:

        F(r) = r^DELTA |M'|^(P-2) M'
        F'(r) = r^DELTA S(r)
        F(Rmax) = lam * M(Rmax)

    where M(Rmax) = integral_0^Rmax M'(s) ds
    and M' = (F/r^DELTA)^(1/(P-1)) on the positive branch.

    We solve for F0 by scalar fixed-point / bisection on:
        G(F0) = F(Rmax;F0) - lam * M(Rmax;F0) = 0
    """
    I = cumulative_trapz((r**DELTA) * source, r)

    def build_from_F0(F0):
        F = F0 + I
        if np.any(F <= 0):
            return None, None, None
        base = F / np.maximum(r**DELTA, EPS)
        Mp = np.maximum(base, EPS)**(1.0 / (P - 1.0))
        M = cumulative_trapz(Mp, r)
        return F, Mp, M

    def G(F0):
        F, Mp, M = build_from_F0(F0)
        if F is None:
            return np.nan
        return F[-1] - lam * M[-1]

    # Find admissible lower bound: need F0 + I > 0 everywhere
    F0_min = -np.min(I) + 1e-12
    lo = max(F0_min, 1e-12)
    glo = G(lo)
    if not np.isfinite(glo):
        return None

    # Expand upper bracket until sign change or give up
    hi = max(1.0, lo * 10.0)
    ghi = G(hi)
    n_expand = 0
    while np.isfinite(ghi) and np.isfinite(glo) and np.sign(glo) == np.sign(ghi) and n_expand < 80:
        hi *= 10.0
        ghi = G(hi)
        n_expand += 1

    if not np.isfinite(glo) or not np.isfinite(ghi):
        return None

    # If no sign change, use whichever endpoint gives smaller |G|
    if np.sign(glo) == np.sign(ghi):
        F0_star = lo if abs(glo) < abs(ghi) else hi
    else:
        # Bisection
        a, b = lo, hi
        ga, gb_ = glo, ghi
        for _ in range(100):
            c = np.sqrt(a * b)  # log-space midpoint
            gc = G(c)
            if not np.isfinite(gc):
                a = c
                ga = gc
                continue
            if abs(gc) < 1e-8:
                a = b = c
                break
            if np.sign(ga) == np.sign(gc):
                a, ga = c, gc
            else:
                b, gb_ = c, gc
        F0_star = np.sqrt(a * b)

    F, Mp, M = build_from_F0(F0_star)
    if F is None:
        return None

    qshape = np.maximum(r * Mp, EPS)**QEXP
    return {
        "F0_eff": F0_star,
        "F_end": F[-1],
        "M_end": M[-1],
        "Mp_end": Mp[-1],
        "qshape": qshape,
    }

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan global Robin leakage parameter lambda
# ------------------------------------------------------------
lam_vals = np.logspace(-8, 8, 49)

results = []
best = None
best_score = np.inf
best_df = None

for lam in lam_vals:
    raw = []
    num = 0.0
    den = 0.0
    n_fail = 0

    for g in galaxies:
        r = g["r"]
        vo = g["vo"]
        gb = g["gb"]
        rho3d = g["rho3d"]

        g0 = 0.68 * g["gb_ref"]
        activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
        source = rho3d * activation

        sol = solve_qshape_robin(r, source, lam)
        if sol is None:
            n_fail += 1
            continue

        qshape = sol["qshape"]
        if not np.all(np.isfinite(qshape)):
            n_fail += 1
            continue

        y = vo**2
        num += np.sum(y * qshape)
        den += np.sum(qshape**2)

        raw.append((g, sol))

    if den <= 0 or len(raw) == 0:
        continue

    C_global = num / den

    rows = []
    for g, sol in raw:
        vo = g["vo"]
        vp = np.sqrt(np.maximum(C_global * sol["qshape"], 0.0))

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "F0_eff": sol["F0_eff"],
            "F_end": sol["F_end"],
            "M_end": sol["M_end"],
            "Mp_end": sol["Mp_end"],
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "lambda": lam,
        "n_fail": n_fail,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
        "median_log10_F0eff": np.median(np.log10(np.maximum(np.abs(df["F0_eff"]), EPS))),
        "median_log10_Mend": np.median(np.log10(np.maximum(np.abs(df["M_end"]), EPS))),
        "median_log10_Fend": np.median(np.log10(np.maximum(np.abs(df["F_end"]), EPS))),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"] + 5.0 * n_fail
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse", "n_fail"])

print("\nTOP 20 ROBIN-BOUNDARY MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST ROBIN-BOUNDARY MODEL")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0_eff","F_end","M_end","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 ROBIN-BOUNDARY MODELS
      lambda  n_fail  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM  median_log10_F0eff  median_log10_Mend  median_log10_Fend
1.000000e-08       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
2.154435e-08       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
4.641589e-08       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
1.000000e-07       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
2.154435e-07       

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked local closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_robin(r, source, lam):
    """
    Solve the integrated nonlinear transport law with Robin boundary:

        F(r) = r^DELTA |M'|^(P-2) M'
        F'(r) = r^DELTA S(r)
        F(Rmax) = lam * M(Rmax)

    where M(Rmax) = integral_0^Rmax M'(s) ds
    and M' = (F/r^DELTA)^(1/(P-1)) on the positive branch.

    We solve for F0 by scalar fixed-point / bisection on:
        G(F0) = F(Rmax;F0) - lam * M(Rmax;F0) = 0
    """
    I = cumulative_trapz((r**DELTA) * source, r)

    def build_from_F0(F0):
        F = F0 + I
        if np.any(F <= 0):
            return None, None, None
        base = F / np.maximum(r**DELTA, EPS)
        Mp = np.maximum(base, EPS)**(1.0 / (P - 1.0))
        M = cumulative_trapz(Mp, r)
        return F, Mp, M

    def G(F0):
        F, Mp, M = build_from_F0(F0)
        if F is None:
            return np.nan
        return F[-1] - lam * M[-1]

    # Find admissible lower bound: need F0 + I > 0 everywhere
    F0_min = -np.min(I) + 1e-12
    lo = max(F0_min, 1e-12)
    glo = G(lo)
    if not np.isfinite(glo):
        return None

    # Expand upper bracket until sign change or give up
    hi = max(1.0, lo * 10.0)
    ghi = G(hi)
    n_expand = 0
    while np.isfinite(ghi) and np.isfinite(glo) and np.sign(glo) == np.sign(ghi) and n_expand < 80:
        hi *= 10.0
        ghi = G(hi)
        n_expand += 1

    if not np.isfinite(glo) or not np.isfinite(ghi):
        return None

    # If no sign change, use whichever endpoint gives smaller |G|
    if np.sign(glo) == np.sign(ghi):
        F0_star = lo if abs(glo) < abs(ghi) else hi
    else:
        # Bisection
        a, b = lo, hi
        ga, gb_ = glo, ghi
        for _ in range(100):
            c = np.sqrt(a * b)  # log-space midpoint
            gc = G(c)
            if not np.isfinite(gc):
                a = c
                ga = gc
                continue
            if abs(gc) < 1e-8:
                a = b = c
                break
            if np.sign(ga) == np.sign(gc):
                a, ga = c, gc
            else:
                b, gb_ = c, gc
        F0_star = np.sqrt(a * b)

    F, Mp, M = build_from_F0(F0_star)
    if F is None:
        return None

    qshape = np.maximum(r * Mp, EPS)**QEXP
    return {
        "F0_eff": F0_star,
        "F_end": F[-1],
        "M_end": M[-1],
        "Mp_end": Mp[-1],
        "qshape": qshape,
    }

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan global Robin leakage parameter lambda
# ------------------------------------------------------------
lam_vals = np.logspace(-8, 8, 49)

results = []
best = None
best_score = np.inf
best_df = None

for lam in lam_vals:
    raw = []
    num = 0.0
    den = 0.0
    n_fail = 0

    for g in galaxies:
        r = g["r"]
        vo = g["vo"]
        gb = g["gb"]
        rho3d = g["rho3d"]

        g0 = 0.68 * g["gb_ref"]
        activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))
        source = rho3d * activation

        sol = solve_qshape_robin(r, source, lam)
        if sol is None:
            n_fail += 1
            continue

        qshape = sol["qshape"]
        if not np.all(np.isfinite(qshape)):
            n_fail += 1
            continue

        y = vo**2
        num += np.sum(y * qshape)
        den += np.sum(qshape**2)

        raw.append((g, sol))

    if den <= 0 or len(raw) == 0:
        continue

    C_global = num / den

    rows = []
    for g, sol in raw:
        vo = g["vo"]
        vp = np.sqrt(np.maximum(C_global * sol["qshape"], 0.0))

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "F0_eff": sol["F0_eff"],
            "F_end": sol["F_end"],
            "M_end": sol["M_end"],
            "Mp_end": sol["Mp_end"],
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "lambda": lam,
        "n_fail": n_fail,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
        "median_log10_F0eff": np.median(np.log10(np.maximum(np.abs(df["F0_eff"]), EPS))),
        "median_log10_Mend": np.median(np.log10(np.maximum(np.abs(df["M_end"]), EPS))),
        "median_log10_Fend": np.median(np.log10(np.maximum(np.abs(df["F_end"]), EPS))),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"] + 5.0 * n_fail
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse", "n_fail"])

print("\nTOP 20 ROBIN-BOUNDARY MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST ROBIN-BOUNDARY MODEL")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","F0_eff","F_end","M_end","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 ROBIN-BOUNDARY MODELS
      lambda  n_fail  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM  median_log10_F0eff  median_log10_Mend  median_log10_Fend
1.000000e-08       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
2.154435e-08       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
4.641589e-08       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
1.000000e-07       0   0.01498    17.205944  33.364277        60.435135      66.234624 -0.103728 -0.035079 -0.017703   0.719625               -12.0           6.721537           8.442175
2.154435e-07       

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked core closure
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vbar2 = np.maximum(vg**2 + vd**2 + vb**2, EPS)
    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def build_nonlocal_memory(r, S, alpha):
    """
    U(r_i) = integral_0^{r_i} (s/r_i)^alpha S(s) ds
    """
    U = np.zeros_like(r, dtype=float)
    for i in range(len(r)):
        ri = max(r[i], EPS)
        s = r[:i+1]
        Ss = S[:i+1]
        w = (np.maximum(s, EPS) / ri) ** alpha
        U[i] = np.trapz(w * Ss, s)
    return U

def solve_qshape_nonlocal(r, S_eff):
    """
    Solve integrated equation on positive branch with zero integration constant:
      F(r) = integral_0^r s^DELTA S_eff(s) ds
      M'   = (F / r^DELTA)^(1/(P-1))
      qshape = [r M']^QEXP
    """
    F = cumulative_trapz((r**DELTA) * S_eff, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan nonlocal-memory parameters
# ------------------------------------------------------------
alpha_vals = np.linspace(-1.0, 4.0, 11)   # kernel weighting exponent
mu_vals    = np.logspace(-4, 3, 22)       # memory strength

results = []
best = None
best_score = np.inf
best_df = None

for alpha in alpha_vals:
    for mu in mu_vals:
        raw = []
        num = 0.0
        den = 0.0

        for g in galaxies:
            r = g["r"]
            vo = g["vo"]
            gb = g["gb"]
            rho3d = g["rho3d"]

            g0 = 0.68 * g["gb_ref"]
            act = 1.0 - np.exp(-((gb / np.maximum(g0, EPS))**MEXP))

            S_local = rho3d * act
            U = build_nonlocal_memory(r, S_local, alpha)
            S_eff = S_local + mu * U

            qshape = solve_qshape_nonlocal(r, S_eff)
            if not np.all(np.isfinite(qshape)):
                continue

            y = vo**2
            num += np.sum(y * qshape)
            den += np.sum(qshape**2)
            raw.append((g, qshape))

        if den <= 0 or len(raw) == 0:
            continue

        C_global = num / den

        rows = []
        for g, qshape in raw:
            vo = g["vo"]
            vp = np.sqrt(np.maximum(C_global * qshape, 0.0))

            rmse = np.sqrt(np.mean((vo - vp)**2))
            mae = np.mean(np.abs(vo - vp))

            n = len(vo)
            i1 = max(1, n // 3)
            i2 = max(i1 + 1, 2 * n // 3)
            Vscale = max(np.max(vo), EPS)

            s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
            s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
            s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

            rows.append({
                "name": g["name"],
                "logM": g["logM"],
                "rmse": rmse,
                "mae": mae,
                "s_in": s_in,
                "s_mid": s_mid,
                "s_out": s_out,
            })

        df = pd.DataFrame(rows)

        hi = df["logM"] >= 10.8
        med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
        mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

        out = {
            "alpha": alpha,
            "mu": mu,
            "global_C": C_global,
            "median_rmse": df["rmse"].median(),
            "mean_rmse": df["rmse"].mean(),
            "median_rmse_hiM": med_hi,
            "mean_rmse_hiM": mean_hi,
            "inner": df["s_in"].median(),
            "mid": df["s_mid"].median(),
            "outer": df["s_out"].median(),
            "corr_logM": corr(df["logM"], df["rmse"]),
        }
        results.append(out)

        score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
        if score < best_score:
            best_score = score
            best = out.copy()
            best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 NONLOCAL-MEMORY MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST NONLOCAL-MEMORY MODEL")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143


/tmp/ipykernel_4927/872504798.py:95: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  U[i] = np.trapz(w * Ss, s)



TOP 20 NONLOCAL-MEMORY MODELS
 alpha       mu  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM
   4.0 0.002154  0.014925    17.277997  33.371961        60.401828      66.250926 -0.104430 -0.036108 -0.018691   0.719562
   3.5 0.002154  0.014912    17.294656  33.375920        60.404091      66.259105 -0.104602 -0.036401 -0.018981   0.719551
   4.0 0.001000  0.014955    17.239411  33.367795        60.419449      66.242120 -0.104054 -0.035557 -0.018162   0.719596
   3.5 0.001000  0.014949    17.247154  33.369607        60.420390      66.245881 -0.104134 -0.035694 -0.018297   0.719591
   3.0 0.001000  0.014939    17.259059  33.372843        60.422836      66.252680 -0.104260 -0.035912 -0.018519   0.719581
   4.0 0.000464  0.014969    17.221483  33.365900        60.427806      66.238089 -0.103879 -0.035301 -0.017916   0.719611
   3.5 0.000464  0.014966    17.225079  33.366735        60.428219      66.239827 -0.103917 -0.035364 -0.017

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked core field law
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    """
    Surviving branch:
      F(r) = integral_0^r s^DELTA S(s) ds
      M'   = (F / r^DELTA)^(1/(P-1))
      qshape = [r M']^QEXP
    """
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan direct baryonic readout weight
#   V_pred^2 = C * qshape + lambda_b * Vbar^2
# For each lambda_b, solve the best global C analytically
# ------------------------------------------------------------
lambda_b_vals = np.linspace(0.0, 2.0, 81)

results = []
best = None
best_score = np.inf
best_df = None

for lambda_b in lambda_b_vals:
    raw = []
    num = 0.0
    den = 0.0

    for g in galaxies:
        r = g["r"]
        vo = g["vo"]
        gb = g["gb"]
        rho3d = g["rho3d"]
        vbar2 = g["vbar2"]

        g0 = 0.68 * g["gb_ref"]
        activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
        source = rho3d * activation

        qshape = solve_qshape_zeroF0(r, source)
        if not np.all(np.isfinite(qshape)):
            continue

        # y = vo^2 ; fit y ≈ C*qshape + lambda_b*vbar2
        y_eff = vo**2 - lambda_b * vbar2
        num += np.sum(y_eff * qshape)
        den += np.sum(qshape**2)

        raw.append((g, qshape))

    if den <= 0 or len(raw) == 0:
        continue

    C_global = max(num / den, 0.0)

    rows = []
    for g, qshape in raw:
        vo = g["vo"]
        vbar2 = g["vbar2"]

        vpred2 = np.maximum(C_global * qshape + lambda_b * vbar2, 0.0)
        vp = np.sqrt(vpred2)

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "lambda_b": lambda_b,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 TWO-CHANNEL READOUT MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST TWO-CHANNEL READOUT")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 TWO-CHANNEL READOUT MODELS
 lambda_b  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM
    0.450  0.010076    17.706174  28.295409        46.250876      52.473898 -0.046176 -0.048319 -0.091727   0.638734
    0.475  0.009803    17.882079  28.304672        46.442174      52.234607 -0.047207 -0.054419 -0.096267   0.636537
    0.425  0.010348    17.580019  28.305758        46.649439      52.747621 -0.045170 -0.052934 -0.086099   0.641215
    0.400  0.010621    17.418563  28.336398        46.835291      53.056229 -0.049189 -0.049020 -0.079968   0.643987
    0.575  0.008714    18.243909  28.522540        46.869573      51.604994 -0.047702 -0.057854 -0.113706   0.630306
    0.500  0.009531    18.062752  28.332780        47.004664      52.029001 -0.047967 -0.060564 -0.099802   0.634613
    0.600  0.008441    18.465478  28.618743        47.017397      51.525082 -0.042109 -0.055065 -0.115016   0.629309
    0.37

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked core field law
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Menc = np.maximum(r * vbar2 / G, EPS)
    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    """
    Surviving branch:
      F(r) = integral_0^r s^DELTA S(s) ds
      M'   = (F / r^DELTA)^(1/(P-1))
      qshape = [r M']^QEXP
    """
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Scan direct baryonic readout weight
#   V_pred^2 = C * qshape + lambda_b * Vbar^2
# For each lambda_b, solve the best global C analytically
# ------------------------------------------------------------
lambda_b_vals = np.linspace(0.0, 2.0, 81)

results = []
best = None
best_score = np.inf
best_df = None

for lambda_b in lambda_b_vals:
    raw = []
    num = 0.0
    den = 0.0

    for g in galaxies:
        r = g["r"]
        vo = g["vo"]
        gb = g["gb"]
        rho3d = g["rho3d"]
        vbar2 = g["vbar2"]

        g0 = 0.68 * g["gb_ref"]
        activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
        source = rho3d * activation

        qshape = solve_qshape_zeroF0(r, source)
        if not np.all(np.isfinite(qshape)):
            continue

        # y = vo^2 ; fit y ≈ C*qshape + lambda_b*vbar2
        y_eff = vo**2 - lambda_b * vbar2
        num += np.sum(y_eff * qshape)
        den += np.sum(qshape**2)

        raw.append((g, qshape))

    if den <= 0 or len(raw) == 0:
        continue

    C_global = max(num / den, 0.0)

    rows = []
    for g, qshape in raw:
        vo = g["vo"]
        vbar2 = g["vbar2"]

        vpred2 = np.maximum(C_global * qshape + lambda_b * vbar2, 0.0)
        vp = np.sqrt(vpred2)

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "lambda_b": lambda_b,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 TWO-CHANNEL READOUT MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST TWO-CHANNEL READOUT")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143

TOP 20 TWO-CHANNEL READOUT MODELS
 lambda_b  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM
    0.450  0.010076    17.706174  28.295409        46.250876      52.473898 -0.046176 -0.048319 -0.091727   0.638734
    0.475  0.009803    17.882079  28.304672        46.442174      52.234607 -0.047207 -0.054419 -0.096267   0.636537
    0.425  0.010348    17.580019  28.305758        46.649439      52.747621 -0.045170 -0.052934 -0.086099   0.641215
    0.400  0.010621    17.418563  28.336398        46.835291      53.056229 -0.049189 -0.049020 -0.079968   0.643987
    0.575  0.008714    18.243909  28.522540        46.869573      51.604994 -0.047702 -0.057854 -0.113706   0.630306
    0.500  0.009531    18.062752  28.332780        47.004664      52.029001 -0.047967 -0.060564 -0.099802   0.634613
    0.600  0.008441    18.465478  28.618743        47.017397      51.525082 -0.042109 -0.055065 -0.115016   0.629309
    0.37

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# locked core field law
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0, 1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0 * np.pi * r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5 * Mtot), len(r) - 1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2 * Rmax), len(r) - 1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Sigma = Mtot / np.maximum(Rmax**2, EPS)
    bulge_frac = Mbul[-1] / np.maximum(Mtot, EPS)

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "Sigma": Sigma,
        "bulge_frac": bulge_frac,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
        "logCgeom": np.log10(np.maximum(Cgeom, EPS)),
        "logf20": np.log10(np.maximum(f20, EPS)),
        "logSigma": np.log10(np.maximum(Sigma, EPS)),
        "logBulgeFrac": np.log10(np.maximum(bulge_frac, EPS)),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# Load galaxies
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

# ------------------------------------------------------------
# Precompute field-channel qshape
# ------------------------------------------------------------
raw_gals = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation

    qshape = solve_qshape_zeroF0(r, source)
    if np.all(np.isfinite(qshape)):
        raw_gals.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw_gals))

# ------------------------------------------------------------
# Scan lambda_b(structure)
# log lambda_b = a logCgeom + b logf20 + c logSigma + d
# and fit best global C for each combination
# ------------------------------------------------------------
a_vals = np.linspace(-2.0, 2.0, 9)
b_vals = np.linspace(-2.0, 2.0, 9)
c_vals = np.linspace(-2.0, 2.0, 9)
d_vals = np.linspace(-2.0, 1.0, 13)   # lambda scale from 0.01 to 10

results = []
best = None
best_score = np.inf
best_df = None

for a in a_vals:
    for b in b_vals:
        for c in c_vals:
            for d in d_vals:
                num = 0.0
                den = 0.0
                rows_pre = []

                for g, qshape in raw_gals:
                    lam_log = (
                        a * g["logCgeom"] +
                        b * g["logf20"] +
                        c * g["logSigma"] +
                        d
                    )
                    lambda_b = 10.0 ** lam_log

                    y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
                    num += np.sum(y_eff * qshape)
                    den += np.sum(qshape**2)

                    rows_pre.append((g, qshape, lambda_b))

                if den <= 0:
                    continue

                C_global = max(num / den, 0.0)

                rows = []
                for g, qshape, lambda_b in rows_pre:
                    vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
                    vp = np.sqrt(vpred2)
                    vo = g["vo"]

                    rmse = np.sqrt(np.mean((vo - vp)**2))
                    mae = np.mean(np.abs(vo - vp))

                    n = len(vo)
                    i1 = max(1, n // 3)
                    i2 = max(i1 + 1, 2 * n // 3)
                    Vscale = max(np.max(vo), EPS)

                    s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
                    s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
                    s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

                    rows.append({
                        "name": g["name"],
                        "logM": g["logM"],
                        "lambda_b": lambda_b,
                        "rmse": rmse,
                        "mae": mae,
                        "s_in": s_in,
                        "s_mid": s_mid,
                        "s_out": s_out,
                    })

                df = pd.DataFrame(rows)
                hi = df["logM"] >= 10.8
                med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
                mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

                out = {
                    "a": a,
                    "b": b,
                    "c": c,
                    "d": d,
                    "global_C": C_global,
                    "median_rmse": df["rmse"].median(),
                    "mean_rmse": df["rmse"].mean(),
                    "median_rmse_hiM": med_hi,
                    "mean_rmse_hiM": mean_hi,
                    "inner": df["s_in"].median(),
                    "mid": df["s_mid"].median(),
                    "outer": df["s_out"].median(),
                    "corr_logM": corr(df["logM"], df["rmse"]),
                    "median_lambda_b": np.median(df["lambda_b"]),
                }
                results.append(out)

                score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2 * out["median_rmse"]
                if score < best_score:
                    best_score = score
                    best = out.copy()
                    best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 STRUCTURE-DEPENDENT TWO-CHANNEL MODELS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST STRUCTURE-DEPENDENT TWO-CHANNEL MODEL")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143

TOP 20 STRUCTURE-DEPENDENT TWO-CHANNEL MODELS
   a    b   c     d  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM  median_lambda_b
-1.0  0.5 0.0  0.50  0.010047    16.530286  28.143642        46.578229      53.432886 -0.020681 -0.016063 -0.063272   0.646087         0.562288
-0.5  0.0 0.0  0.25  0.006317    17.584426  29.153772        46.874336      53.733943  0.006267 -0.030726 -0.094999   0.642411         1.123841
-0.5  0.0 0.0  0.00  0.010109    15.842586  27.670101        46.884892      52.803544 -0.017965 -0.012937 -0.048833   0.646263         0.631982
 0.0  0.0 0.0 -0.25  0.008851    18.132376  28.479927        47.049821      51.656932 -0.050356 -0.059269 -0.111479   0.630890         0.562341
-1.5  1.0 0.0  1.00  0.009874    17.624548  29.226600        47.197456      55.325256 -0.012601 -0.026492 -0.064819   0.653575         0.552662
-1.0 -1.0 0.0  0.00  

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed field model
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# load + precompute field channel
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

raw = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    if np.all(np.isfinite(qshape)):
        raw.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw))

# ------------------------------------------------------------
# fixed structural mixing law:
# lambda_b(gal) = lambda0 * sqrt(f20) / Cgeom
# scan lambda0, solve best global C analytically
# ------------------------------------------------------------
lambda0_vals = np.logspace(-2, 1.5, 81)

results = []
best = None
best_score = np.inf
best_df = None

for lambda0 in lambda0_vals:
    num = 0.0
    den = 0.0
    rows_pre = []

    for g, qshape in raw:
        lambda_b = lambda0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS)
        y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
        num += np.sum(y_eff * qshape)
        den += np.sum(qshape**2)
        rows_pre.append((g, qshape, lambda_b))

    if den <= 0:
        continue

    C_global = max(num / den, 0.0)

    rows = []
    for g, qshape, lambda_b in rows_pre:
        vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
        vp = np.sqrt(vpred2)
        vo = g["vo"]

        rmse = np.sqrt(np.mean((vo - vp)**2))
        mae = np.mean(np.abs(vo - vp))

        n = len(vo)
        i1 = max(1, n // 3)
        i2 = max(i1 + 1, 2 * n // 3)
        Vscale = max(np.max(vo), EPS)

        s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
        s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
        s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

        rows.append({
            "name": g["name"],
            "logM": g["logM"],
            "Mtot": g["Mtot"],
            "Rmax": g["Rmax"],
            "lambda_b": lambda_b,
            "rmse": rmse,
            "mae": mae,
            "s_in": s_in,
            "s_mid": s_mid,
            "s_out": s_out,
            "vobs_max": np.max(vo),
            "vpred_max": np.max(vp),
        })

    df = pd.DataFrame(rows)

    hi = df["logM"] >= 10.8
    med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
    mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

    out = {
        "lambda0": lambda0,
        "global_C": C_global,
        "median_rmse": df["rmse"].median(),
        "mean_rmse": df["rmse"].mean(),
        "median_rmse_hiM": med_hi,
        "mean_rmse_hiM": mean_hi,
        "inner": df["s_in"].median(),
        "mid": df["s_mid"].median(),
        "outer": df["s_out"].median(),
        "corr_logM": corr(df["logM"], df["rmse"]),
        "median_lambda_b": df["lambda_b"].median(),
    }
    results.append(out)

    score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
    if score < best_score:
        best_score = score
        best = out.copy()
        best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 FIXED-THEORY TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

# ------------------------------------------------------------
# scaling checks on best model
# ------------------------------------------------------------
best_lambda0 = best["lambda0"]
best_C = best["global_C"]

rows_scaling = []
for g, qshape in raw:
    lambda_b = best_lambda0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS)
    vpred2 = np.maximum(best_C * qshape + lambda_b * g["vbar2"], 0.0)
    vp = np.sqrt(vpred2)

    vmax_pred = float(np.max(vp))
    rows_scaling.append({
        "name": g["name"],
        "Mtot": g["Mtot"],
        "Rmax": g["Rmax"],
        "logM": np.log10(g["Mtot"]),
        "logVpred": np.log10(np.maximum(vmax_pred, EPS)),
        "logRmax": np.log10(np.maximum(g["Rmax"], EPS)),
        "lambda_b": lambda_b,
        "rmse": float(np.sqrt(np.mean((g["vo"] - vp)**2))),
    })

sc_df = pd.DataFrame(rows_scaling)

# linear fits: logVpred ~ a logM + b ; logRmax ~ c logM + d
Xv = np.column_stack([sc_df["logM"].values, np.ones(len(sc_df))])
yv = sc_df["logVpred"].values
coef_v, _, _, _ = np.linalg.lstsq(Xv, yv, rcond=None)

Xr = np.column_stack([sc_df["logM"].values, np.ones(len(sc_df))])
yr = sc_df["logRmax"].values
coef_r, _, _, _ = np.linalg.lstsq(Xr, yr, rcond=None)

v_slope, v_int = coef_v
r_slope, r_int = coef_r

print("\nSCALING CHECKS")
print(f"log Vmax_pred = {v_slope:.6f} logM + {v_int:.6f}")
print(f"log Rmax      = {r_slope:.6f} logM + {r_int:.6f}")
print(f"corr(logM, logVmax_pred) = {corr(sc_df['logM'], sc_df['logVpred']):.6f}")
print(f"corr(logM, logRmax)      = {corr(sc_df['logM'], sc_df['logRmax']):.6f}")

print("\nWORST HIGH-MASS GALAXIES UNDER FIXED THEORY")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

print("\nBEST 20 GALAXIES UNDER FIXED THEORY")
print(
    best_df[["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=True)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143

TOP 20 FIXED-THEORY TESTS
 lambda0  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM  median_lambda_b
3.117095  0.010118    16.487392  28.131699        46.693019      53.449403 -0.020968 -0.016438 -0.062266   0.646295         0.554254
3.447466  0.009602    16.531754  28.249763        46.848629      53.394147 -0.018256 -0.017284 -0.062718   0.645188         0.612998
2.818383  0.010584    16.323699  28.086855        47.251463      53.630647 -0.024547 -0.017333 -0.055646   0.648079         0.501140
2.548297  0.011005    16.502887  28.099408        47.500265      53.905172 -0.028710 -0.017635 -0.051482   0.650271         0.453116
2.304093  0.011386    16.901188  28.158091        47.730893      54.249054 -0.030938 -0.018662 -0.050145   0.652783         0.409693
2.083291  0.011730    16.802593  28.254578        47.944089      54.643645 -0.032197 -0.020722 -0.045780   

In [ ]:
import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# PATHS — same as your notebooks
# ============================================================
WORKDIR    = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR     = os.path.join(WORKDIR, "mts_highmass_diagnostic")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ============================================================
# FIXED MODEL — exactly your frozen branch
# ============================================================
R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed    = 1.5
m_inf       = 0.02
A_src       = 0.10
UPS_DISK    = 1.0
UPS_BUL     = 1.0
R_CORE      = 0.5
SMOOTH_WIN  = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0   # km/s — same as your audit

# ============================================================
# HELPERS
# ============================================================
def display_name(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$",        "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1: return y.copy()
    if win % 2 == 0: win += 1
    pad = win // 2
    yp  = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win)/win, mode="valid")

def safe_rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m]-b[m])**2))) if m.sum() else np.nan

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files: return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            print(f"Extracting: {zp}")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(ROTMOD_DIR)
            break
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No rotmod .dat files found.")
    return files

# ============================================================
# FIELD SOLVER — same screened equation as your notebooks
# ============================================================
def solve_field(r_grid, source_grid):
    n   = len(r_grid)
    A   = lil_matrix((n, n))
    rhs = np.zeros(n)
    for i in range(1, n-1):
        dr_p = r_grid[i+1] - r_grid[i]
        dr_m = r_grid[i]   - r_grid[i-1]
        dr_c = 0.5*(r_grid[i+1] - r_grid[i-1])
        fp   = r_grid[i+1] / dr_p
        fm   = r_grid[i-1] / dr_m
        fc   = r_grid[i]   / (Rs_fixed**2 * dr_c)
        A[i, i+1] =  fp / (r_grid[i] * dr_c)
        A[i, i-1] =  fm / (r_grid[i] * dr_c)
        A[i, i  ] = -(fp + fm)/(r_grid[i]*dr_c) - fc
        rhs[i]    = -A_src * source_grid[i]
    # BC: u'(0)=0 → u[0]=u[1]
    A[0, 0] = 1.0; A[0, 1] = -1.0; rhs[0] = 0.0
    # BC: u → m_inf at outer boundary
    A[-1, -1] = 1.0; rhs[-1] = m_inf
    A   = A.tocsr()
    u   = spsolve(A, rhs)
    return u

# ============================================================
# GALAXY PROCESSOR
# ============================================================
def process_galaxy(path):
    raw = np.genfromtxt(path)
    if raw.ndim != 2 or raw.shape[1] < 5:
        return None

    r_d, vo, vg, vd, vb = raw[:,0], raw[:,1], raw[:,2], raw[:,3], raw[:,4]
    mask = (np.isfinite(r_d) & np.isfinite(vo) & np.isfinite(vg) &
            np.isfinite(vd) & np.isfinite(vb) & (r_d > 0) & (vo > 0))
    if mask.sum() < 6:
        return None

    r_d, vo, vg, vd, vb = [x[mask] for x in (r_d, vo, vg, vd, vb)]
    idx = np.argsort(r_d)
    r_d, vo, vg, vd, vb = [x[idx] for x in (r_d, vo, vg, vd, vb)]

    vgas2  = np.maximum(vg**2, 0)
    vdisk2 = np.maximum(vd**2, 0)
    vbul2  = np.maximum(vb**2, 0)
    vbar2  = vgas2 + vdisk2 + vbul2

    G = 4.30091e-6
    Mgas  = r_d * vgas2  / G
    Mdisk = r_d * vdisk2 / G
    Mbul  = r_d * vbul2  / G
    Menc  = np.maximum(Mgas + Mdisk + Mbul, 1e-12)

    dMdr  = np.gradient(Menc, r_d)
    rho3d = np.maximum(dMdr / np.maximum(4*np.pi*r_d**2, 1e-12), SOURCE_FLOOR)
    rho3d = moving_average(rho3d, SMOOTH_WIN)

    # interpolate onto fine grid
    r_grid = np.linspace(max(R_MIN, r_d[0]*0.5), min(R_MAX, r_d[-1]*1.2), N_R)

    ip_rho  = PchipInterpolator(r_d, rho3d,  extrapolate=True)
    ip_vbar = PchipInterpolator(r_d, vbar2,  extrapolate=True)
    ip_vbul = PchipInterpolator(r_d, vbul2,  extrapolate=True)
    ip_vobs = PchipInterpolator(r_d, vo,     extrapolate=False)

    src_grid  = np.maximum(ip_rho(r_grid),  SOURCE_FLOOR)
    vbar_grid = np.maximum(ip_vbar(r_grid), 1e-12)
    vbul_grid = np.maximum(ip_vbul(r_grid), 0)

    u = solve_field(r_grid, src_grid)
    u = np.maximum(u, 0)
    u_inf = float(u[-1]) if u[-1] > 0 else m_inf

    # find transition radius rt where U(rt) = F_FRAC * U_inf
    target = F_FRAC * u_inf
    cross  = np.where(u >= target)[0]
    if len(cross) == 0:
        return None
    it = cross[0]
    rt = float(r_grid[it])
    ut = float(u[it])

    # amplitude
    vbar2_t  = float(vbar_grid[it])
    vbul2_t  = float(vbul_grid[it])
    vflat2   = C_AMP * (rt * ut)**ALPHA * max(vbar2_t, 1e-6)**BETA

    # shape: V/Vflat = u/u_inf
    vpred_grid = np.sqrt(np.maximum(vflat2 * (u / u_inf)**2, 0))

    # evaluate at data points
    valid_r = (r_d >= r_grid[0]) & (r_d <= r_grid[-1])
    if valid_r.sum() < 4:
        return None

    ip_vpred = interp1d(r_grid, vpred_grid, kind='linear', bounds_error=False,
                        fill_value=np.nan)
    vp = ip_vpred(r_d[valid_r])
    vo_v = vo[valid_r]
    r_v  = r_d[valid_r]

    rmse = safe_rmse(vo_v, vp)

    n    = len(vo_v)
    i1   = max(1, n//3)
    i2   = max(i1+1, 2*n//3)
    Vsc  = max(np.nanmax(vo_v), 1e-6)
    s_in  = float(np.nanmean(vp[:i1]   - vo_v[:i1]))   / Vsc
    s_mid = float(np.nanmean(vp[i1:i2] - vo_v[i1:i2])) / Vsc
    s_out = float(np.nanmean(vp[i2:]   - vo_v[i2:]))   / Vsc

    Mtot   = float(Menc[-1])
    vmax_o = float(np.max(vo))

    # ---- DIAGNOSTIC QUANTITIES ----
    # bulge fraction at transition
    f_bul_t = vbul2_t / max(vbar2_t, 1e-12)

    # compactness at transition: Vbar^2 / rt  (proxy for central mass density)
    compactness_t = vbar2_t / max(rt, 1e-6)

    # how far out is rt relative to total extent?
    rt_frac = rt / max(r_d[-1], 1e-6)

    # slope of observed rotation curve in outer third
    r_out   = r_v[i2:]
    vo_out  = vo_v[i2:]
    if len(r_out) >= 3:
        p       = np.polyfit(r_out, vo_out, 1)
        dv_dr_out = float(p[0])   # positive = rising, negative = falling
    else:
        dv_dr_out = np.nan

    # field saturation: how close is u at outer edge to u_inf?
    u_outer_frac = float(u[-1]) / max(u_inf, 1e-12)

    # residual shape: is the outer prediction systematically high or low?
    outer_bias = float(np.nanmean(vp[i2:] - vo_v[i2:]))

    return {
        "name":          display_name(path),
        "logM":          np.log10(max(Mtot, 1)),
        "Mtot":          Mtot,
        "vmax_obs":      vmax_o,
        "rt":            rt,
        "rt_frac":       rt_frac,
        "f_bul_t":       f_bul_t,
        "compactness_t": compactness_t,
        "vbar2_t":       vbar2_t,
        "vflat2_pred":   vflat2,
        "vflat_pred":    np.sqrt(max(vflat2, 0)),
        "u_inf":         u_inf,
        "u_outer_frac":  u_outer_frac,
        "rmse":          rmse,
        "s_in":          s_in,
        "s_mid":         s_mid,
        "s_out":         s_out,
        "dv_dr_out":     dv_dr_out,
        "outer_bias":    outer_bias,
        "n_points":      n,
    }

# ============================================================
# RUN
# ============================================================
files = bootstrap_rotmods()
print(f"Found {len(files)} rotmod files")

rows = []
for f in files:
    try:
        r = process_galaxy(f)
        if r is not None:
            rows.append(r)
    except Exception as e:
        print(f"  FAILED: {os.path.basename(f)} — {e}")

df = pd.DataFrame(rows)
print(f"Processed: {len(df)} galaxies\n")

# ============================================================
# SPLIT: high-vmax vs low-vmax
# ============================================================
hi  = df["vmax_obs"] >= HIGH_VMAX_CUT
lo  = ~hi

print(f"High-vmax (>={HIGH_VMAX_CUT} km/s): {hi.sum()}  |  Low-vmax: {lo.sum()}")
print()

# ============================================================
# REPORT 1: median structural properties — high vs low
# ============================================================
cols_compare = [
    "rmse", "s_in", "s_mid", "s_out",
    "rt", "rt_frac", "f_bul_t", "compactness_t",
    "u_outer_frac", "dv_dr_out", "outer_bias"
]

print("=" * 65)
print("STRUCTURAL COMPARISON: HIGH-VMAX vs LOW-VMAX (medians)")
print("=" * 65)
for c in cols_compare:
    h = df.loc[hi, c].median()
    l = df.loc[lo, c].median()
    print(f"  {c:<20s}  hi={h:+10.4f}   lo={l:+10.4f}   diff={h-l:+.4f}")

# ============================================================
# REPORT 2: within high-vmax, worst vs best
# ============================================================
hi_df = df[hi].copy()
hi_df_sorted = hi_df.sort_values("rmse", ascending=False)

top_bad = hi_df_sorted.head(10)
top_ok  = hi_df_sorted.tail(10)

print()
print("=" * 65)
print("HIGH-VMAX WORST 10 (by RMSE)")
print("=" * 65)
print(top_bad[["name","logM","vmax_obs","rmse","s_in","s_mid","s_out",
               "rt_frac","f_bul_t","outer_bias","dv_dr_out"]].to_string(index=False))

print()
print("=" * 65)
print("HIGH-VMAX BEST 10 (by RMSE)")
print("=" * 65)
print(top_ok[["name","logM","vmax_obs","rmse","s_in","s_mid","s_out",
              "rt_frac","f_bul_t","outer_bias","dv_dr_out"]].to_string(index=False))

# ============================================================
# REPORT 3: correlations within high-vmax population
# ============================================================
print()
print("=" * 65)
print("CORRELATIONS WITH RMSE (high-vmax only)")
print("=" * 65)
diag_cols = ["logM","vmax_obs","rt","rt_frac","f_bul_t",
             "compactness_t","u_outer_frac","dv_dr_out","outer_bias"]
for c in diag_cols:
    sub = hi_df[[c, "rmse"]].dropna()
    if len(sub) >= 5:
        r = np.corrcoef(sub[c], sub["rmse"])[0,1]
        print(f"  corr(rmse, {c:<20s}) = {r:+.4f}")

# ============================================================
# REPORT 4: outer bias sign breakdown
# ============================================================
print()
print("=" * 65)
print("OUTER BIAS DIRECTION (high-vmax)")
print("  positive = model OVER-predicts outer velocity")
print("  negative = model UNDER-predicts outer velocity")
print("=" * 65)
over  = (hi_df["outer_bias"] >  2.0).sum()
under = (hi_df["outer_bias"] < -2.0).sum()
near  = ((hi_df["outer_bias"] >= -2.0) & (hi_df["outer_bias"] <= 2.0)).sum()
print(f"  Over-predicting  (bias > +2):  {over}")
print(f"  Under-predicting (bias < -2):  {under}")
print(f"  Near-zero        (|bias|<=2):  {near}")

# ============================================================
# REPORT 5: does the outer rotation curve slope correlate with error?
# ============================================================
print()
print("=" * 65)
print("OUTER SLOPE vs RMSE (high-vmax) — rising/flat/falling breakdown")
print("=" * 65)
hi_nona = hi_df.dropna(subset=["dv_dr_out"])
rising  = hi_nona[hi_nona["dv_dr_out"] >  0.5]
falling = hi_nona[hi_nona["dv_dr_out"] < -0.5]
flat    = hi_nona[(hi_nona["dv_dr_out"] >= -0.5) & (hi_nona["dv_dr_out"] <= 0.5)]
print(f"  Rising  curves: n={len(rising):3d}  median RMSE={rising['rmse'].median():.2f}")
print(f"  Flat    curves: n={len(flat):3d}  median RMSE={flat['rmse'].median():.2f}")
print(f"  Falling curves: n={len(falling):3d}  median RMSE={falling['rmse'].median():.2f}")

# ============================================================
# SAVE full table
# ============================================================
out_csv = os.path.join(OUTDIR, "highmass_diagnostic_full.csv")
df.sort_values("rmse", ascending=False).to_csv(out_csv, index=False)
print(f"\nFull table saved: {out_csv}")
print("\nDONE.")

Found 175 rotmod files
Processed: 165 galaxies

High-vmax (>=150.0 km/s): 56  |  Low-vmax: 109

STRUCTURAL COMPARISON: HIGH-VMAX vs LOW-VMAX (medians)
  rmse                  hi=+7098432944.4017   lo=+277284582.2792   diff=+6821148362.1225
  s_in                  hi=+39795725.9480   lo=+5518045.5511   diff=+34277680.3969
  s_mid                 hi=+1591636.6430   lo=+288594.5094   diff=+1303042.1336
  s_out                 hi=+20626.8024   lo=+24748.7773   diff=-4121.9749
  rt                    hi=   +0.4350   lo=   +0.2600   diff=+0.1750
  rt_frac               hi=   +0.0153   lo=   +0.0335   diff=-0.0182
  f_bul_t               hi=   +0.8989   lo=   +0.7707   diff=+0.1282
  compactness_t         hi=+12686.6532   lo= +358.5659   diff=+12328.0873
  u_outer_frac          hi=   +1.0000   lo=   +1.0000   diff=+0.0000
  dv_dr_out             hi=   -0.3975   lo=   +1.0370   diff=-1.4345
  outer_bias            hi=+4658930.9659   lo=+1998279.8654   diff=+2660651.1005

HIGH-VMAX WORST 10 (by

In [ ]:
import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# COMPACTNESS DAMPING CORRECTION SCAN
# ------------------------------------------------------------
# Base amplitude law (frozen):
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Damping correction:
#   c_t = Vbar^2(rt) / rt          (compactness at transition)
#   V_flat,corr^2 = V_flat,base^2 / (1 + eta * (c_t / c0)^delta)
#
# Hypothesis from diagnostic:
#   corr(rmse, compactness_t) = +0.71
#   55/56 high-vmax galaxies OVER-predicted in outer region
#   => amplitude is too high for compact systems
#   => we need a DAMPING term, not a boost
#
# Everything else (source, field, shape) stays frozen.
# ============================================================

WORKDIR    = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR     = os.path.join(WORKDIR, "mts_compactness_damping")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ============================================================
# FROZEN MODEL CONSTANTS
# ============================================================
R_MIN        = 1.0e-3
R_MAX        = 55.0
N_R          = 700
Rs_fixed     = 1.5
m_inf        = 0.02
A_src        = 0.10
UPS_DISK     = 1.0
UPS_BUL      = 1.0
R_CORE       = 0.5
SMOOTH_WIN   = 9
SOURCE_FLOOR = 1.0e-6
F_FRAC       = 0.60
C_AMP        = 164.0
ALPHA        = 0.175
BETA         = 0.55
G            = 4.30091e-6

# ============================================================
# SCAN GRID
# ------------------------------------------------------------
# eta   — damping strength  (0 = no correction)
# delta — how sharply compactness kicks in
# c0    — pivot compactness scale (km^2/s^2 / kpc)
#
# c0 range anchored to the diagnostic data:
#   median compactness_t hi-vmax = ~12687
#   median compactness_t lo-vmax = ~359
#   so c0 should live somewhere in that range
# ============================================================
ETA_LIST   = [0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.00]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00, 1.50]
C0_LIST    = [500.0, 1000.0, 2000.0, 5000.0, 10000.0, 20000.0, 50000.0]

HIGH_VMAX_CUT = 150.0

# ============================================================
# HELPERS
# ============================================================
def display_name(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$",        "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1: return y.copy()
    if win % 2 == 0: win += 1
    pad = win // 2
    yp  = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win)/win, mode="valid")

def safe_rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m]-b[m])**2))) if m.sum() else np.nan

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files: return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            print(f"Extracting: {zp}")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(ROTMOD_DIR)
            break
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No rotmod .dat files found.")
    return files

def solve_field(r_grid, source_grid):
    n   = len(r_grid)
    A   = lil_matrix((n, n))
    rhs = np.zeros(n)
    for i in range(1, n-1):
        dr_p = r_grid[i+1] - r_grid[i]
        dr_m = r_grid[i]   - r_grid[i-1]
        dr_c = 0.5*(r_grid[i+1] - r_grid[i-1])
        fp   = r_grid[i+1] / dr_p
        fm   = r_grid[i-1] / dr_m
        fc   = r_grid[i]   / (Rs_fixed**2 * dr_c)
        A[i, i+1] =  fp / (r_grid[i] * dr_c)
        A[i, i-1] =  fm / (r_grid[i] * dr_c)
        A[i, i  ] = -(fp + fm)/(r_grid[i]*dr_c) - fc
        rhs[i]    = -A_src * source_grid[i]
    A[0, 0] =  1.0; A[0, 1] = -1.0; rhs[0]  = 0.0
    A[-1,-1] = 1.0;                  rhs[-1] = m_inf
    u = spsolve(A.tocsr(), rhs)
    return u

# ============================================================
# PRE-COMPUTE: field solution + transition quantities per galaxy
# (independent of eta/delta/c0 — only amplitude changes)
# ============================================================
def precompute_galaxy(path):
    raw = np.genfromtxt(path)
    if raw.ndim != 2 or raw.shape[1] < 5:
        return None

    r_d, vo, vg, vd, vb = raw[:,0], raw[:,1], raw[:,2], raw[:,3], raw[:,4]
    mask = (np.isfinite(r_d) & np.isfinite(vo) & np.isfinite(vg) &
            np.isfinite(vd) & np.isfinite(vb) & (r_d > 0) & (vo > 0))
    if mask.sum() < 6:
        return None

    r_d, vo, vg, vd, vb = [x[mask] for x in (r_d, vo, vg, vd, vb)]
    idx = np.argsort(r_d)
    r_d, vo, vg, vd, vb = [x[idx] for x in (r_d, vo, vg, vd, vb)]

    vgas2  = np.maximum(vg**2, 0)
    vdisk2 = np.maximum(vd**2, 0)
    vbul2  = np.maximum(vb**2, 0)
    vbar2  = vgas2 + vdisk2 + vbul2

    Menc  = np.maximum(r_d*vgas2/G + r_d*vdisk2/G + r_d*vbul2/G, 1e-12)
    dMdr  = np.gradient(Menc, r_d)
    rho3d = np.maximum(dMdr / np.maximum(4*np.pi*r_d**2, 1e-12), SOURCE_FLOOR)
    rho3d = moving_average(rho3d, SMOOTH_WIN)

    r_grid = np.linspace(max(R_MIN, r_d[0]*0.5), min(R_MAX, r_d[-1]*1.2), N_R)

    ip_rho  = PchipInterpolator(r_d, rho3d, extrapolate=True)
    ip_vbar = PchipInterpolator(r_d, vbar2, extrapolate=True)

    src_grid  = np.maximum(ip_rho(r_grid),  SOURCE_FLOOR)
    vbar_grid = np.maximum(ip_vbar(r_grid), 1e-12)

    u     = np.maximum(solve_field(r_grid, src_grid), 0)
    u_inf = float(u[-1]) if u[-1] > 0 else m_inf

    target = F_FRAC * u_inf
    cross  = np.where(u >= target)[0]
    if len(cross) == 0:
        return None
    it = cross[0]

    rt      = float(r_grid[it])
    ut      = float(u[it])
    vbar2_t = float(vbar_grid[it])
    c_t     = vbar2_t / max(rt, 1e-6)   # compactness at transition

    # base amplitude (no correction)
    vflat2_base = C_AMP * (rt * ut)**ALPHA * max(vbar2_t, 1e-6)**BETA

    # shape profile (frozen)
    shape = (u / u_inf)**2   # V²/Vflat² = (u/u_inf)²

    # observed data on grid
    valid_r  = (r_d >= r_grid[0]) & (r_d <= r_grid[-1])
    if valid_r.sum() < 4:
        return None

    return {
        "name":         display_name(path),
        "logM":         np.log10(max(Menc[-1], 1)),
        "vmax_obs":     float(np.max(vo)),
        "r_d":          r_d[valid_r],
        "vo":           vo[valid_r],
        "r_grid":       r_grid,
        "shape":        shape,
        "vflat2_base":  vflat2_base,
        "c_t":          c_t,
        "rt":           rt,
    }

def eval_correction(g, eta, delta, c0):
    c_t        = g["c_t"]
    damp       = 1.0 + eta * (c_t / max(c0, 1e-6))**delta
    vflat2_cor = g["vflat2_base"] / damp
    vpred_grid = np.sqrt(np.maximum(vflat2_cor * g["shape"], 0))

    ip = interp1d(g["r_grid"], vpred_grid, kind='linear',
                  bounds_error=False, fill_value=np.nan)
    vp = ip(g["r_d"])
    vo = g["vo"]

    rmse = safe_rmse(vo, vp)

    n  = len(vo)
    i1 = max(1, n//3)
    i2 = max(i1+1, 2*n//3)
    Vs = max(np.nanmax(vo), 1e-6)

    s_in  = float(np.nanmean(vp[:i1]   - vo[:i1]))   / Vs
    s_mid = float(np.nanmean(vp[i1:i2] - vo[i1:i2])) / Vs
    s_out = float(np.nanmean(vp[i2:]   - vo[i2:]))   / Vs

    return rmse, s_in, s_mid, s_out

# ============================================================
# LOAD & PRE-COMPUTE
# ============================================================
files = bootstrap_rotmods()
print(f"Found {len(files)} rotmod files")

precomp = []
for f in files:
    try:
        g = precompute_galaxy(f)
        if g is not None:
            precomp.append(g)
    except Exception as e:
        print(f"  SKIP {os.path.basename(f)}: {e}")

print(f"Pre-computed: {len(precomp)} galaxies\n")

hi_idx = [i for i,g in enumerate(precomp) if g["vmax_obs"] >= HIGH_VMAX_CUT]
lo_idx = [i for i,g in enumerate(precomp) if g["vmax_obs"] <  HIGH_VMAX_CUT]
print(f"High-vmax: {len(hi_idx)}  |  Low-vmax: {len(lo_idx)}\n")

# ============================================================
# SCAN
# ============================================================
scan_results = []

total = len(ETA_LIST) * len(DELTA_LIST) * len(C0_LIST)
done  = 0

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for c0 in C0_LIST:
            rmse_all = []
            rmse_hi  = []
            rmse_lo  = []
            s_out_hi = []

            for g in precomp:
                rmse, s_in, s_mid, s_out = eval_correction(g, eta, delta, c0)
                if np.isfinite(rmse):
                    rmse_all.append(rmse)
                    if g["vmax_obs"] >= HIGH_VMAX_CUT:
                        rmse_hi.append(rmse)
                        s_out_hi.append(s_out)
                    else:
                        rmse_lo.append(rmse)

            scan_results.append({
                "eta":            eta,
                "delta":          delta,
                "c0":             c0,
                "median_rmse_all": np.median(rmse_all) if rmse_all else np.nan,
                "median_rmse_hi":  np.median(rmse_hi)  if rmse_hi  else np.nan,
                "median_rmse_lo":  np.median(rmse_lo)  if rmse_lo  else np.nan,
                "mean_rmse_hi":    np.mean(rmse_hi)    if rmse_hi  else np.nan,
                "median_sout_hi":  np.median(s_out_hi) if s_out_hi else np.nan,
            })

            done += 1
            if done % 20 == 0:
                print(f"  {done}/{total} done...")

scan_df = pd.DataFrame(scan_results)

# ============================================================
# SCORE: minimise high-mass RMSE without wrecking low-mass
# ------------------------------------------------------------
# penalty if low-mass median RMSE increases by more than 10%
# relative to eta=0 baseline
# ============================================================
baseline_lo = scan_df[scan_df["eta"] == 0.0]["median_rmse_lo"].median()
baseline_hi = scan_df[scan_df["eta"] == 0.0]["median_rmse_hi"].median()
print(f"\nBaseline (eta=0) median RMSE — hi: {baseline_hi:.4e}  lo: {baseline_lo:.4e}")

scan_df["lo_penalty"] = np.maximum(
    0, (scan_df["median_rmse_lo"] - baseline_lo) / baseline_lo
)
# combined score: hi RMSE + penalty for hurting low-mass
scan_df["score"] = scan_df["median_rmse_hi"] * (1.0 + 2.0 * scan_df["lo_penalty"])

scan_df_sorted = scan_df.sort_values("score")

print("\n" + "=" * 75)
print("TOP 25 CONFIGURATIONS (scored: minimise hi RMSE, protect lo RMSE)")
print("=" * 75)
print(scan_df_sorted.head(25).to_string(index=False))

print("\n" + "=" * 75)
print("TOP 10 BY RAW HIGH-VMAX RMSE (ignoring low-mass protection)")
print("=" * 75)
print(scan_df_sorted.sort_values("median_rmse_hi").head(10).to_string(index=False))

# ============================================================
# DEEP DIVE on best configuration
# ============================================================
best = scan_df_sorted.iloc[0]
best_eta   = float(best["eta"])
best_delta = float(best["delta"])
best_c0    = float(best["c0"])

print(f"\n{'='*75}")
print(f"BEST CONFIG: eta={best_eta}  delta={best_delta}  c0={best_c0}")
print(f"{'='*75}")

rows_best = []
for g in precomp:
    rmse, s_in, s_mid, s_out = eval_correction(g, best_eta, best_delta, best_c0)
    damp = 1.0 + best_eta * (g["c_t"] / max(best_c0, 1e-6))**best_delta
    rows_best.append({
        "name":     g["name"],
        "logM":     g["logM"],
        "vmax_obs": g["vmax_obs"],
        "c_t":      g["c_t"],
        "damp":     damp,
        "rmse":     rmse,
        "s_in":     s_in,
        "s_mid":    s_mid,
        "s_out":    s_out,
    })

best_df = pd.DataFrame(rows_best)

hi_best = best_df[best_df["vmax_obs"] >= HIGH_VMAX_CUT]
lo_best = best_df[best_df["vmax_obs"] <  HIGH_VMAX_CUT]

print(f"\nHigh-vmax  median RMSE: {hi_best['rmse'].median():.4e}  "
      f"(was {baseline_hi:.4e}, change {100*(hi_best['rmse'].median()-baseline_hi)/baseline_hi:+.1f}%)")
print(f"Low-vmax   median RMSE: {lo_best['rmse'].median():.4e}  "
      f"(was {baseline_lo:.4e}, change {100*(lo_best['rmse'].median()-baseline_lo)/baseline_lo:+.1f}%)")

print("\nHIGH-VMAX WORST 10 UNDER BEST CONFIG")
print(hi_best.sort_values("rmse", ascending=False)
      [["name","logM","vmax_obs","c_t","damp","rmse","s_in","s_mid","s_out"]]
      .head(10).to_string(index=False))

print("\nHIGH-VMAX BEST 10 UNDER BEST CONFIG")
print(hi_best.sort_values("rmse")
      [["name","logM","vmax_obs","c_t","damp","rmse","s_in","s_mid","s_out"]]
      .head(10).to_string(index=False))

# ============================================================
# CHECK: did the s_out bias improve?
# ============================================================
print("\nOUTER BIAS DIRECTION UNDER BEST CONFIG (high-vmax)")
print("  positive = model still over-predicts outer velocity")
print("  negative = model now under-predicts outer velocity")
over  = (hi_best["s_out"] >  0.01).sum()
under = (hi_best["s_out"] < -0.01).sum()
near  = ((hi_best["s_out"] >= -0.01) & (hi_best["s_out"] <= 0.01)).sum()
print(f"  Over-predicting  : {over}")
print(f"  Under-predicting : {under}")
print(f"  Near-zero        : {near}")

# ============================================================
# SAVE
# ============================================================
scan_df_sorted.to_csv(os.path.join(OUTDIR, "scan_results.csv"), index=False)
best_df.to_csv(os.path.join(OUTDIR, "best_config_galaxy_detail.csv"), index=False)
print(f"\nResults saved to {OUTDIR}")
print("\nDONE.")

Found 175 rotmod files
Pre-computed: 165 galaxies

High-vmax: 56  |  Low-vmax: 109

  20/280 done...
  40/280 done...
  60/280 done...
  80/280 done...
  100/280 done...
  120/280 done...
  140/280 done...
  160/280 done...
  180/280 done...
  200/280 done...
  220/280 done...
  240/280 done...
  260/280 done...
  280/280 done...

Baseline (eta=0) median RMSE — hi: 7.0984e+09  lo: 2.7728e+08

TOP 25 CONFIGURATIONS (scored: minimise hi RMSE, protect lo RMSE)
 eta  delta     c0  median_rmse_all  median_rmse_hi  median_rmse_lo  mean_rmse_hi  median_sout_hi  lo_penalty        score
1.00   1.50  500.0     2.520310e+08    8.240146e+08    1.449187e+08  2.583135e+09     1939.257839         0.0 8.240146e+08
0.75   1.50  500.0     2.803394e+08    9.049374e+08    1.489953e+08  2.936357e+09     2193.913348         0.0 9.049374e+08
0.50   1.50  500.0     3.065931e+08    1.101392e+09    1.568254e+08  3.503182e+09     2683.149483         0.0 1.101392e+09
1.00   1.50 1000.0     3.376186e+08    1.30645

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed theory from the last successful run
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5
LAMBDA0 = 3.1170947235801254
CGLOB   = 0.010117642510818817

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def fit_slope_loglog(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if np.sum(m) < 3:
        return np.nan
    X = np.column_stack([np.log10(x[m]), np.ones(np.sum(m))])
    Y = np.log10(y[m])
    coef, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    return float(coef[0])

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)
    vbar   = np.sqrt(vbar2)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Sigma = Mtot / np.maximum(Rmax**2, EPS)
    bulge_frac = Mbul[-1] / np.maximum(Mtot, EPS)
    disk_frac  = Mdisk[-1] / np.maximum(Mtot, EPS)
    gas_frac   = Mgas[-1] / np.maximum(Mtot, EPS)

    jpk = int(np.argmax(vbar))
    Rpk = max(r[jpk], EPS)
    Vpk = max(vbar[jpk], EPS)
    Gpk = max(gb[jpk], EPS)

    i1 = max(3, len(r)//3)
    i2 = max(i1+2, 2*len(r)//3)

    inner_slope_vbar = fit_slope_loglog(r[:i1], vbar[:i1])
    outer_slope_vbar = fit_slope_loglog(r[i2:], vbar[i2:])
    inner_slope_gb   = fit_slope_loglog(r[:i1], gb[:i1])
    outer_slope_gb   = fit_slope_loglog(r[i2:], gb[i2:])

    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Rpk": Rpk,
        "Cgeom": Cgeom,
        "f20": f20,
        "Sigma": Sigma,
        "bulge_frac": bulge_frac,
        "disk_frac": disk_frac,
        "gas_frac": gas_frac,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
        "logRmax": np.log10(np.maximum(Rmax, EPS)),
        "logR50": np.log10(np.maximum(R50, EPS)),
        "logRpk": np.log10(np.maximum(Rpk, EPS)),
        "logCgeom": np.log10(np.maximum(Cgeom, EPS)),
        "logf20": np.log10(np.maximum(f20, EPS)),
        "logSigma": np.log10(np.maximum(Sigma, EPS)),
        "logBulgeFrac": np.log10(np.maximum(bulge_frac, EPS)),
        "logDiskFrac": np.log10(np.maximum(disk_frac, EPS)),
        "logGasFrac": np.log10(np.maximum(gas_frac, EPS)),
        "logGpk": np.log10(np.maximum(Gpk, EPS)),
        "inner_slope_vbar": inner_slope_vbar,
        "outer_slope_vbar": outer_slope_vbar,
        "inner_slope_gb": inner_slope_gb,
        "outer_slope_gb": outer_slope_gb,
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

rows = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]
    vo = g["vo"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    lambda_b = LAMBDA0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS)
    vpred2 = np.maximum(CGLOB * qshape + lambda_b * g["vbar2"], 0.0)
    vp = np.sqrt(vpred2)

    rmse = np.sqrt(np.mean((vo - vp)**2))
    mae = np.mean(np.abs(vo - vp))

    n = len(vo)
    k1 = max(1, n // 3)
    k2 = max(k1 + 1, 2 * n // 3)
    Vscale = max(np.max(vo), EPS)

    s_in  = np.mean(vp[:k1]   - vo[:k1])   / Vscale
    s_mid = np.mean(vp[k1:k2] - vo[k1:k2]) / Vscale
    s_out = np.mean(vp[k2:]   - vo[k2:])   / Vscale

    if g["logM"] >= 10.8:
        neg_count = int(s_in < -0.05) + int(s_mid < -0.05) + int(s_out < -0.05)
        pos_count = int(s_in >  0.05) + int(s_mid >  0.05) + int(s_out >  0.05)
        if neg_count >= 2:
            sign_class = "neg"
        elif pos_count >= 2:
            sign_class = "pos"
        else:
            sign_class = "mixed"
    else:
        sign_class = "lowmass"

    rows.append({
        **{k:v for k,v in g.items() if not isinstance(v, np.ndarray)},
        "lambda_b": lambda_b,
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "sign_class": sign_class,
    })

df = pd.DataFrame(rows)
dfh = df[df["logM"] >= 10.8].copy()

print("\nHIGH-MASS CLASS COUNTS")
print(dfh["sign_class"].value_counts().to_string())

for cls in ["neg", "pos", "mixed"]:
    sub = dfh[dfh["sign_class"] == cls].copy()
    if len(sub) == 0:
        continue
    print(f"\nCLASS = {cls.upper()} | N = {len(sub)}")
    print("\nMembers:")
    print(sub[["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]].sort_values("rmse", ascending=False).to_string(index=False))
    print("\nStructure summary:")
    print(sub[[
        "logM","logCgeom","logf20","logSigma","logBulgeFrac","logGasFrac",
        "logRmax","logR50","logRpk","logGpk",
        "inner_slope_vbar","outer_slope_vbar","inner_slope_gb","outer_slope_gb",
        "lambda_b","rmse"
    ]].describe().to_string())

print("\nCLASS MEANS")
print(dfh.groupby("sign_class")[[
    "logM","logCgeom","logf20","logSigma","logBulgeFrac","logGasFrac",
    "logRmax","logR50","logRpk","logGpk",
    "inner_slope_vbar","outer_slope_vbar","inner_slope_gb","outer_slope_gb",
    "lambda_b","rmse"
]].mean().to_string())

# variable ranking by separation power across neg vs pos/mixed
cand_cols = [
    "logCgeom","logf20","logSigma","logBulgeFrac","logGasFrac",
    "logRmax","logR50","logRpk","logGpk",
    "inner_slope_vbar","outer_slope_vbar","inner_slope_gb","outer_slope_gb",
    "lambda_b"
]

rank_rows = []
neg = dfh[dfh["sign_class"] == "neg"]
pos = dfh[dfh["sign_class"] == "pos"]
mix = dfh[dfh["sign_class"] == "mixed"]

for col in cand_cols:
    vals = dfh[col].values
    labels = (dfh["sign_class"] == "neg").astype(float).values
    sep_neg = corr(vals, labels)

    if len(neg) > 0 and len(pos) > 0:
        d_neg_pos = abs(neg[col].mean() - pos[col].mean())
    else:
        d_neg_pos = np.nan

    if len(neg) > 0 and len(mix) > 0:
        d_neg_mix = abs(neg[col].mean() - mix[col].mean())
    else:
        d_neg_mix = np.nan

    rank_rows.append({
        "variable": col,
        "corr_with_neg_class": sep_neg,
        "abs_mean_diff_neg_pos": d_neg_pos,
        "abs_mean_diff_neg_mixed": d_neg_mix,
    })

rank_df = pd.DataFrame(rank_rows).sort_values(
    ["abs_mean_diff_neg_mixed","abs_mean_diff_neg_pos"],
    ascending=False
)

print("\nTOP STRUCTURAL SEPARATORS")
print(rank_df.to_string(index=False))

Usable galaxies: 143

HIGH-MASS CLASS COUNTS
sign_class
pos      22
neg      14
mixed    11

CLASS = NEG | N = 14

Members:
               name      logM  lambda_b       rmse      s_in     s_mid     s_out
UGC02487_rotmod.dat 11.647878  0.512270 117.264155 -0.398323 -0.261415 -0.235604
UGC06787_rotmod.dat 10.811363  0.434935 116.366596 -0.580569 -0.288585 -0.294459
UGC05253_rotmod.dat 11.157370  0.468758 111.496875 -0.658816 -0.355970 -0.179690
UGC02916_rotmod.dat 11.037242  0.561440  98.258869 -0.621209 -0.405429 -0.224466
 NGC2841_rotmod.dat 11.332356  0.363181  97.496710 -0.399485 -0.247358 -0.220339
UGC11914_rotmod.dat 11.059532  0.502284  84.931807 -0.432536 -0.188693 -0.085447
UGC09133_rotmod.dat 11.435481  0.251936  82.089662 -0.434033 -0.136697 -0.010738
 NGC5985_rotmod.dat 11.448616  0.483148  80.015573 -0.364985 -0.195179 -0.176089
UGC06786_rotmod.dat 10.811819  0.410700  71.135699 -0.424197 -0.192577 -0.230712
UGC02953_rotmod.dat 11.393352  0.313031  69.097256 -0.327324 -0.09

In [ ]:
import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.interpolate import PchipInterpolator, interp1d
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# INNER COVERAGE AUDIT + SAFE BOUNDARY RE-RUN
# ------------------------------------------------------------
# Problem identified:
#   Many SPARC galaxies don't start at r=0.
#   PchipInterpolator extrapolating inward gives pathological
#   source densities => spurious small rt => c_t explodes.
#
# This script:
#   1. Audits inner coverage for all galaxies
#   2. Classifies: GOOD (r_min small) vs SPARSE (r_min large)
#   3. For ALL galaxies: applies safe inner BC —
#      instead of extrapolating inward, hold source constant
#      at first data point value (flat inner floor)
#   4. Flags whether rt fell in the extrapolated zone
#   5. Shows whether pathological c_t values are fixed
# ============================================================

WORKDIR    = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR     = os.path.join(WORKDIR, "mts_inner_coverage_fix")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ============================================================
# FROZEN MODEL
# ============================================================
R_MIN        = 1.0e-3
R_MAX        = 55.0
N_R          = 700
Rs_fixed     = 1.5
m_inf        = 0.02
A_src        = 0.10
SMOOTH_WIN   = 9
SOURCE_FLOOR = 1.0e-6
F_FRAC       = 0.60
C_AMP        = 164.0
ALPHA        = 0.175
BETA         = 0.55
G            = 4.30091e-6

# Best config from previous scan
ETA   = 1.0
DELTA = 1.5
C0    = 500.0

HIGH_VMAX_CUT = 150.0

# Coverage thresholds
INNER_FRAC_THRESH = 0.05   # r_min / r_max > this => SPARSE
INNER_ABS_THRESH  = 1.0    # kpc — r_min > this => SPARSE

# ============================================================
# HELPERS
# ============================================================
def display_name(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$",        "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1: return y.copy()
    if win % 2 == 0: win += 1
    pad = win // 2
    yp  = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win)/win, mode="valid")

def safe_rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m]-b[m])**2))) if m.sum() else np.nan

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files: return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            print(f"Extracting: {zp}")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(ROTMOD_DIR)
            break
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No rotmod .dat files found.")
    return files

def solve_field(r_grid, source_grid):
    n   = len(r_grid)
    A   = lil_matrix((n, n))
    rhs = np.zeros(n)
    for i in range(1, n-1):
        dr_p = r_grid[i+1] - r_grid[i]
        dr_m = r_grid[i]   - r_grid[i-1]
        dr_c = 0.5*(r_grid[i+1] - r_grid[i-1])
        fp   = r_grid[i+1] / dr_p
        fm   = r_grid[i-1] / dr_m
        fc   = r_grid[i]   / (Rs_fixed**2 * dr_c)
        A[i, i+1] =  fp / (r_grid[i] * dr_c)
        A[i, i-1] =  fm / (r_grid[i] * dr_c)
        A[i, i  ] = -(fp + fm)/(r_grid[i]*dr_c) - fc
        rhs[i]    = -A_src * source_grid[i]
    A[0, 0] =  1.0; A[0, 1] = -1.0; rhs[0]  = 0.0
    A[-1,-1] = 1.0;                  rhs[-1] = m_inf
    return spsolve(A.tocsr(), rhs)

def load_raw(path):
    raw = np.genfromtxt(path)
    if raw.ndim != 2 or raw.shape[1] < 5:
        return None
    r_d, vo, vg, vd, vb = raw[:,0], raw[:,1], raw[:,2], raw[:,3], raw[:,4]
    mask = (np.isfinite(r_d) & np.isfinite(vo) & np.isfinite(vg) &
            np.isfinite(vd) & np.isfinite(vb) & (r_d > 0) & (vo > 0))
    if mask.sum() < 6:
        return None
    r_d, vo, vg, vd, vb = [x[mask] for x in (r_d, vo, vg, vd, vb)]
    idx = np.argsort(r_d)
    return [x[idx] for x in (r_d, vo, vg, vd, vb)]

# ============================================================
# PART 1 — AUDIT
# ============================================================
def audit_coverage(path):
    data = load_raw(path)
    if data is None:
        return None
    r_d, vo, vg, vd, vb = data
    r_min      = float(r_d[0])
    r_max      = float(r_d[-1])
    inner_frac = r_min / max(r_max, 1e-6)
    sparse     = (inner_frac > INNER_FRAC_THRESH) or (r_min > INNER_ABS_THRESH)
    dr_first   = float(r_d[1] - r_d[0]) if len(r_d) > 1 else 0.0
    dr_median  = float(np.median(np.diff(r_d)))
    return {
        "name":       display_name(path),
        "r_min":      r_min,
        "r_max":      r_max,
        "inner_frac": inner_frac,
        "dr_first":   dr_first,
        "dr_median":  dr_median,
        "n_points":   len(r_d),
        "vmax_obs":   float(np.max(vo)),
        "sparse":     sparse,
    }

# ============================================================
# PART 2 — PROCESS WITH SAFE INNER BC
# ------------------------------------------------------------
# Safe BC: instead of letting PchipInterpolator extrapolate
# inward (which can go wildly wrong), hold the source and
# vbar2 constant at the value of the first data point for
# any r < r_d[0].  This is physically conservative — it
# assumes the inner region looks like the innermost ring
# we actually measured rather than inventing something.
# ============================================================
def process_galaxy_safe(path):
    data = load_raw(path)
    if data is None:
        return None
    r_d, vo, vg, vd, vb = data

    vgas2  = np.maximum(vg**2, 0)
    vdisk2 = np.maximum(vd**2, 0)
    vbul2  = np.maximum(vb**2, 0)
    vbar2  = vgas2 + vdisk2 + vbul2

    Menc  = np.maximum(r_d*vgas2/G + r_d*vdisk2/G + r_d*vbul2/G, 1e-12)
    dMdr  = np.gradient(Menc, r_d)
    rho3d = np.maximum(dMdr / np.maximum(4*np.pi*r_d**2, 1e-12), SOURCE_FLOOR)
    rho3d = moving_average(rho3d, SMOOTH_WIN)

    r_grid = np.linspace(R_MIN, min(R_MAX, r_d[-1]*1.2), N_R)

    # safe interpolation — no extrapolation beyond data
    ip_rho  = PchipInterpolator(r_d, rho3d, extrapolate=False)
    ip_vbar = PchipInterpolator(r_d, vbar2, extrapolate=False)

    src_grid  = np.zeros(N_R)
    vbar_grid = np.zeros(N_R)

    in_range  = (r_grid >= r_d[0]) & (r_grid <= r_d[-1])
    inner     = r_grid < r_d[0]
    outer     = r_grid > r_d[-1]

    # interpolated region
    src_grid[in_range]  = np.maximum(ip_rho(r_grid[in_range]),  SOURCE_FLOOR)
    vbar_grid[in_range] = np.maximum(ip_vbar(r_grid[in_range]), 1e-12)

    # inner zone: hold constant at first data point — no invented density spike
    src_grid[inner]  = max(float(rho3d[0]),  SOURCE_FLOOR)
    vbar_grid[inner] = max(float(vbar2[0]),  1e-12)

    # outer zone: hold constant at last data point
    src_grid[outer]  = max(float(rho3d[-1]), SOURCE_FLOOR)
    vbar_grid[outer] = max(float(vbar2[-1]), 1e-12)

    u     = np.maximum(solve_field(r_grid, src_grid), 0)
    u_inf = float(u[-1]) if u[-1] > 0 else m_inf

    target = F_FRAC * u_inf
    cross  = np.where(u >= target)[0]
    if len(cross) == 0:
        return None
    it = cross[0]

    rt      = float(r_grid[it])
    ut      = float(u[it])
    vbar2_t = float(vbar_grid[it])
    c_t     = vbar2_t / max(rt, 1e-6)

    rt_in_extrap = rt < r_d[0]

    vflat2_base = C_AMP * (rt * ut)**ALPHA * max(vbar2_t, 1e-6)**BETA
    damp        = 1.0 + ETA * (c_t / max(C0, 1e-6))**DELTA
    vflat2_cor  = vflat2_base / damp
    shape       = (u / u_inf)**2
    vpred_grid  = np.sqrt(np.maximum(vflat2_cor * shape, 0))

    valid_r = (r_d >= r_grid[0]) & (r_d <= r_grid[-1])
    if valid_r.sum() < 4:
        return None

    ip_pred = interp1d(r_grid, vpred_grid, kind='linear',
                       bounds_error=False, fill_value=np.nan)
    vp   = ip_pred(r_d[valid_r])
    vo_v = vo[valid_r]

    rmse = safe_rmse(vo_v, vp)
    n    = len(vo_v)
    i1   = max(1, n//3)
    i2   = max(i1+1, 2*n//3)
    Vs   = max(float(np.nanmax(vo_v)), 1e-6)
    s_in  = float(np.nanmean(vp[:i1]   - vo_v[:i1]))   / Vs
    s_mid = float(np.nanmean(vp[i1:i2] - vo_v[i1:i2])) / Vs
    s_out = float(np.nanmean(vp[i2:]   - vo_v[i2:]))   / Vs

    r_min      = float(r_d[0])
    r_max      = float(r_d[-1])
    inner_frac = r_min / max(r_max, 1e-6)
    sparse     = (inner_frac > INNER_FRAC_THRESH) or (r_min > INNER_ABS_THRESH)

    return {
        "name":         display_name(path),
        "logM":         np.log10(max(float(Menc[-1]), 1)),
        "vmax_obs":     float(np.max(vo)),
        "r_min":        r_min,
        "r_max":        r_max,
        "inner_frac":   inner_frac,
        "sparse":       sparse,
        "rt":           rt,
        "rt_in_extrap": rt_in_extrap,
        "c_t":          c_t,
        "damp":         damp,
        "rmse":         rmse,
        "s_in":         s_in,
        "s_mid":        s_mid,
        "s_out":        s_out,
    }

# ============================================================
# RUN
# ============================================================
files = bootstrap_rotmods()
print(f"Found {len(files)} rotmod files\n")

# --- Audit ---
audit_rows = []
for f in files:
    try:
        r = audit_coverage(f)
        if r: audit_rows.append(r)
    except: pass

audit_df = pd.DataFrame(audit_rows)
n_sparse = int(audit_df["sparse"].sum())
n_good   = int((~audit_df["sparse"]).sum())

print("=" * 65)
print("PART 1 — INNER COVERAGE AUDIT")
print("=" * 65)
print(f"  Total galaxies             : {len(audit_df)}")
print(f"  GOOD  (r_min < threshold)  : {n_good}")
print(f"  SPARSE (r_min >= threshold): {n_sparse}")
print(f"\n  Thresholds:")
print(f"    inner_frac > {INNER_FRAC_THRESH}  (r_min / r_max)")
print(f"    OR r_min   > {INNER_ABS_THRESH} kpc")
print(f"\n  r_min percentiles (all galaxies):")
for pct in [0, 10, 25, 50, 75, 90, 100]:
    val = np.percentile(audit_df["r_min"], pct)
    print(f"    p{pct:3d} = {val:.4f} kpc")

print(f"\n  SPARSE galaxies (worst inner coverage, top 20):")
sparse_sorted = audit_df[audit_df["sparse"]].sort_values("inner_frac", ascending=False)
print(sparse_sorted[["name","r_min","r_max","inner_frac","vmax_obs","n_points"]]
      .head(20).to_string(index=False))

# --- Process ---
print("\n\nPART 2 — SAFE INNER BC PROCESSING")
print("Running field solver on all galaxies...")
rows = []
for f in files:
    try:
        r = process_galaxy_safe(f)
        if r: rows.append(r)
    except Exception as e:
        print(f"  SKIP {os.path.basename(f)}: {e}")

df = pd.DataFrame(rows)
print(f"Processed: {len(df)} galaxies")

hi     = df[df["vmax_obs"] >= HIGH_VMAX_CUT]
lo     = df[df["vmax_obs"] <  HIGH_VMAX_CUT]
hi_g   = hi[~hi["sparse"]]
hi_s   = hi[hi["sparse"]]

print("\n" + "=" * 65)
print("RESULTS SUMMARY — SAFE BC  (eta=1.0, delta=1.5, c0=500)")
print("=" * 65)
print(f"  All            median RMSE : {df['rmse'].median():.4e}  (n={len(df)})")
print(f"  High-vmax      median RMSE : {hi['rmse'].median():.4e}  (n={len(hi)})")
print(f"  Low-vmax       median RMSE : {lo['rmse'].median():.4e}  (n={len(lo)})")
print(f"  Hi-vmax GOOD   median RMSE : {hi_g['rmse'].median():.4e}  (n={len(hi_g)})")
print(f"  Hi-vmax SPARSE median RMSE : {hi_s['rmse'].median():.4e}  (n={len(hi_s)})")

n_extrap    = int(df["rt_in_extrap"].sum())
n_extrap_hi = int(hi["rt_in_extrap"].sum())
print(f"\n  rt fell in extrapolated inner zone:")
print(f"    All       : {n_extrap} / {len(df)}")
print(f"    High-vmax : {n_extrap_hi} / {len(hi)}")

print(f"\nOUTER BIAS — ALL HIGH-VMAX (safe BC)")
over  = int((hi["s_out"] >  0.01).sum())
under = int((hi["s_out"] < -0.01).sum())
near  = int(((hi["s_out"] >= -0.01) & (hi["s_out"] <= 0.01)).sum())
print(f"  Over-predicting  : {over}")
print(f"  Under-predicting : {under}")
print(f"  Near-zero        : {near}")

print(f"\nOUTER BIAS — GOOD HIGH-VMAX ONLY (safe BC)")
over_g  = int((hi_g["s_out"] >  0.01).sum())
under_g = int((hi_g["s_out"] < -0.01).sum())
near_g  = int(((hi_g["s_out"] >= -0.01) & (hi_g["s_out"] <= 0.01)).sum())
print(f"  Over-predicting  : {over_g}")
print(f"  Under-predicting : {under_g}")
print(f"  Near-zero        : {near_g}")

print(f"\nCOMPACTNESS c_t DISTRIBUTION after safe BC (high-vmax):")
ct_clean = hi["c_t"].replace([np.inf, -np.inf], np.nan).dropna()
for pct in [0, 25, 50, 75, 90, 95, 100]:
    val = np.percentile(ct_clean, pct)
    print(f"  p{pct:3d} = {val:.2f}")

print(f"\nPREVIOUSLY PATHOLOGICAL GALAXIES — status after safe BC:")
bad_names = ["NGC2903","NGC3521","NGC5055","NGC0801","NGC2998",
             "UGC02953","UGC03546","NGC5005","ESO563-G021","NGC3198"]
for name in bad_names:
    row = df[df["name"] == name]
    if len(row):
        r = row.iloc[0]
        print(f"  {name:15s}  r_min={r['r_min']:.3f}  c_t={r['c_t']:12.1f}  "
              f"damp={r['damp']:8.2f}  rmse={r['rmse']:.3e}  "
              f"rt_extrap={r['rt_in_extrap']}  sparse={r['sparse']}")

print(f"\nHIGH-VMAX WORST 10 (safe BC)")
print(hi.sort_values("rmse", ascending=False)
      [["name","logM","vmax_obs","r_min","inner_frac","c_t","damp",
        "rmse","s_out","rt_in_extrap","sparse"]]
      .head(10).to_string(index=False))

print(f"\nHIGH-VMAX BEST 10 (safe BC)")
print(hi.sort_values("rmse")
      [["name","logM","vmax_obs","r_min","inner_frac","c_t","damp",
        "rmse","s_out","rt_in_extrap","sparse"]]
      .head(10).to_string(index=False))

# ============================================================
# SAVE
# ============================================================
audit_df.to_csv(os.path.join(OUTDIR, "inner_coverage_audit.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "safe_bc_results.csv"), index=False)
print(f"\nSaved to {OUTDIR}")
print("\nDONE.")

Found 175 rotmod files

PART 1 — INNER COVERAGE AUDIT
  Total galaxies             : 165
  GOOD  (r_min < threshold)  : 67
  SPARSE (r_min >= threshold): 98

  Thresholds:
    inner_frac > 0.05  (r_min / r_max)
    OR r_min   > 1.0 kpc

  r_min percentiles (all galaxies):
    p  0 = 0.0800 kpc
    p 10 = 0.1600 kpc
    p 25 = 0.3200 kpc
    p 50 = 0.6200 kpc
    p 75 = 1.0800 kpc
    p 90 = 1.8180 kpc
    p100 = 9.9300 kpc

  SPARSE galaxies (worst inner coverage, top 20):
    name  r_min  r_max  inner_frac  vmax_obs  n_points
 NGC3949   1.74   7.07    0.246110     169.0         7
 NGC3953   3.50  15.68    0.223214     224.0         8
UGC06973   1.74   7.85    0.221656     180.0         9
 NGC3992   9.19  46.02    0.199696     272.0         9
UGC06923   0.93   5.16    0.180233      81.1         6
 NGC4013   5.49  31.01    0.177040     198.0        36
  F561-1   1.61   9.66    0.166667      50.4         6
 F563-V1   1.31   7.87    0.166455      29.5         6
UGC06917   1.74  10.47    0

In [ ]:
import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.interpolate import PchipInterpolator, interp1d
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# F_FRAC × Rs JOINT SCAN
# ------------------------------------------------------------
# Core problem identified:
#   rt (transition radius) always falls BELOW r_min of data
#   for every galaxy — 165/165.
#   This means the amplitude law is always anchored at a point
#   we have no measurements for.
#
# Two parameters control where rt lands:
#
#   F_FRAC — fraction of u_inf that defines transition
#             higher F_FRAC => rt moves outward (later crossing)
#
#   Rs     — field screening length
#             larger Rs => field rises more slowly => rt moves out
#             smaller Rs => field rises steeply => rt stays small
#
# This scan tests whether pushing rt into observed territory
# (rt >= r_min of each galaxy) improves predictions, and finds
# the best (F_FRAC, Rs) combination.
#
# Everything else stays frozen.
# ============================================================

WORKDIR    = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR     = os.path.join(WORKDIR, "mts_ffrac_rs_scan")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ============================================================
# FROZEN CONSTANTS
# ============================================================
R_MIN        = 1.0e-3
R_MAX        = 55.0
N_R          = 700
m_inf        = 0.02
A_src        = 0.10
SMOOTH_WIN   = 9
SOURCE_FLOOR = 1.0e-6
C_AMP        = 164.0
ALPHA        = 0.175
BETA         = 0.55
G            = 4.30091e-6

# Compactness damping from best scan
ETA   = 1.0
DELTA = 1.5
C0    = 500.0

HIGH_VMAX_CUT = 150.0

# ============================================================
# SCAN GRID
# ------------------------------------------------------------
# F_FRAC: push from 0.60 toward 0.99
#   at 0.99 rt is near the outer edge — probably too far
#   sweet spot likely 0.75 - 0.95
#
# Rs: current frozen value is 1.5
#   scan 0.5 to 8.0 — wider range to see the landscape
# ============================================================
FFRAC_LIST = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.98]
RS_LIST    = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]

# ============================================================
# HELPERS
# ============================================================
def display_name(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$",        "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1: return y.copy()
    if win % 2 == 0: win += 1
    pad = win // 2
    yp  = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win)/win, mode="valid")

def safe_rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m]-b[m])**2))) if m.sum() else np.nan

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files: return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            print(f"Extracting: {zp}")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(ROTMOD_DIR)
            break
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No rotmod .dat files found.")
    return files

def solve_field(r_grid, source_grid, Rs):
    n   = len(r_grid)
    A   = lil_matrix((n, n))
    rhs = np.zeros(n)
    for i in range(1, n-1):
        dr_p = r_grid[i+1] - r_grid[i]
        dr_m = r_grid[i]   - r_grid[i-1]
        dr_c = 0.5*(r_grid[i+1] - r_grid[i-1])
        fp   = r_grid[i+1] / dr_p
        fm   = r_grid[i-1] / dr_m
        fc   = r_grid[i]   / (Rs**2 * dr_c)
        A[i, i+1] =  fp / (r_grid[i] * dr_c)
        A[i, i-1] =  fm / (r_grid[i] * dr_c)
        A[i, i  ] = -(fp + fm)/(r_grid[i]*dr_c) - fc
        rhs[i]    = -A_src * source_grid[i]
    A[0, 0] =  1.0; A[0, 1] = -1.0; rhs[0]  = 0.0
    A[-1,-1] = 1.0;                  rhs[-1] = m_inf
    return spsolve(A.tocsr(), rhs)

def load_raw(path):
    raw = np.genfromtxt(path)
    if raw.ndim != 2 or raw.shape[1] < 5:
        return None
    r_d, vo, vg, vd, vb = raw[:,0], raw[:,1], raw[:,2], raw[:,3], raw[:,4]
    mask = (np.isfinite(r_d) & np.isfinite(vo) & np.isfinite(vg) &
            np.isfinite(vd) & np.isfinite(vb) & (r_d > 0) & (vo > 0))
    if mask.sum() < 6:
        return None
    r_d, vo, vg, vd, vb = [x[mask] for x in (r_d, vo, vg, vd, vb)]
    idx = np.argsort(r_d)
    return [x[idx] for x in (r_d, vo, vg, vd, vb)]

# ============================================================
# PRE-COMPUTE: source grid per galaxy (independent of F_FRAC/Rs)
# Then for each (F_FRAC, Rs) combo: solve field, find rt, score
# ============================================================
def precompute(path):
    data = load_raw(path)
    if data is None:
        return None
    r_d, vo, vg, vd, vb = data

    vgas2  = np.maximum(vg**2, 0)
    vdisk2 = np.maximum(vd**2, 0)
    vbul2  = np.maximum(vb**2, 0)
    vbar2  = vgas2 + vdisk2 + vbul2

    Menc  = np.maximum(r_d*vgas2/G + r_d*vdisk2/G + r_d*vbul2/G, 1e-12)
    dMdr  = np.gradient(Menc, r_d)
    rho3d = np.maximum(dMdr / np.maximum(4*np.pi*r_d**2, 1e-12), SOURCE_FLOOR)
    rho3d = moving_average(rho3d, SMOOTH_WIN)

    r_grid = np.linspace(R_MIN, min(R_MAX, r_d[-1]*1.2), N_R)

    # interpolate source onto grid using data range only
    # hold constant outside data range (safe BC)
    ip_rho  = PchipInterpolator(r_d, rho3d, extrapolate=False)
    ip_vbar = PchipInterpolator(r_d, vbar2, extrapolate=False)

    src_grid  = np.zeros(N_R)
    vbar_grid = np.zeros(N_R)

    in_range = (r_grid >= r_d[0]) & (r_grid <= r_d[-1])
    inner    = r_grid < r_d[0]
    outer    = r_grid > r_d[-1]

    src_grid[in_range]  = np.maximum(ip_rho(r_grid[in_range]),  SOURCE_FLOOR)
    vbar_grid[in_range] = np.maximum(ip_vbar(r_grid[in_range]), 1e-12)
    src_grid[inner]     = max(float(rho3d[0]),  SOURCE_FLOOR)
    vbar_grid[inner]    = max(float(vbar2[0]),  1e-12)
    src_grid[outer]     = max(float(rho3d[-1]), SOURCE_FLOOR)
    vbar_grid[outer]    = max(float(vbar2[-1]), 1e-12)

    valid_r = (r_d >= r_grid[0]) & (r_d <= r_grid[-1])
    if valid_r.sum() < 4:
        return None

    return {
        "name":      display_name(path),
        "logM":      np.log10(max(float(Menc[-1]), 1)),
        "vmax_obs":  float(np.max(vo)),
        "r_min":     float(r_d[0]),
        "r_max":     float(r_d[-1]),
        "r_grid":    r_grid,
        "src_grid":  src_grid,
        "vbar_grid": vbar_grid,
        "r_d":       r_d[valid_r],
        "vo":        vo[valid_r],
    }

def evaluate(g, Rs, F_FRAC):
    u     = np.maximum(solve_field(g["r_grid"], g["src_grid"], Rs), 0)
    u_inf = float(u[-1]) if u[-1] > 0 else m_inf

    target = F_FRAC * u_inf
    cross  = np.where(u >= target)[0]
    if len(cross) == 0:
        return None
    it = cross[0]

    rt      = float(g["r_grid"][it])
    ut      = float(u[it])
    vbar2_t = float(g["vbar_grid"][it])
    c_t     = vbar2_t / max(rt, 1e-6)

    rt_in_data = rt >= g["r_min"]

    vflat2_base = C_AMP * (rt * ut)**ALPHA * max(vbar2_t, 1e-6)**BETA
    damp        = 1.0 + ETA * (c_t / max(C0, 1e-6))**DELTA
    vflat2_cor  = vflat2_base / damp
    shape       = (u / u_inf)**2
    vpred_grid  = np.sqrt(np.maximum(vflat2_cor * shape, 0))

    ip_pred = interp1d(g["r_grid"], vpred_grid, kind='linear',
                       bounds_error=False, fill_value=np.nan)
    vp   = ip_pred(g["r_d"])
    vo_v = g["vo"]

    rmse = safe_rmse(vo_v, vp)
    n    = len(vo_v)
    i1   = max(1, n//3)
    i2   = max(i1+1, 2*n//3)
    Vs   = max(float(np.nanmax(vo_v)), 1e-6)
    s_out = float(np.nanmean(vp[i2:] - vo_v[i2:])) / Vs

    return {
        "rt":          rt,
        "rt_in_data":  rt_in_data,
        "c_t":         c_t,
        "damp":        damp,
        "rmse":        rmse,
        "s_out":       s_out,
    }

# ============================================================
# LOAD
# ============================================================
files = bootstrap_rotmods()
print(f"Found {len(files)} rotmod files")

precomp = []
for f in files:
    try:
        g = precompute(f)
        if g: precomp.append(g)
    except Exception as e:
        print(f"  SKIP {os.path.basename(f)}: {e}")

print(f"Pre-computed: {len(precomp)} galaxies\n")

hi_idx = [i for i,g in enumerate(precomp) if g["vmax_obs"] >= HIGH_VMAX_CUT]
lo_idx = [i for i,g in enumerate(precomp) if g["vmax_obs"] <  HIGH_VMAX_CUT]
print(f"High-vmax: {len(hi_idx)}  |  Low-vmax: {len(lo_idx)}\n")

# ============================================================
# SCAN
# ============================================================
total = len(FFRAC_LIST) * len(RS_LIST)
done  = 0
scan_rows = []

for F_FRAC in FFRAC_LIST:
    for Rs in RS_LIST:

        rmse_hi, rmse_lo = [], []
        sout_hi          = []
        n_rt_in_data_hi  = 0
        n_rt_in_data_all = 0
        ct_hi            = []

        for g in precomp:
            res = evaluate(g, Rs, F_FRAC)
            if res is None:
                continue
            if res["rt_in_data"]:
                n_rt_in_data_all += 1

            if g["vmax_obs"] >= HIGH_VMAX_CUT:
                if np.isfinite(res["rmse"]):
                    rmse_hi.append(res["rmse"])
                    sout_hi.append(res["s_out"])
                    ct_hi.append(res["c_t"])
                if res["rt_in_data"]:
                    n_rt_in_data_hi += 1
            else:
                if np.isfinite(res["rmse"]):
                    rmse_lo.append(res["rmse"])

        scan_rows.append({
            "F_FRAC":           F_FRAC,
            "Rs":               Rs,
            "median_rmse_hi":   np.median(rmse_hi)  if rmse_hi else np.nan,
            "median_rmse_lo":   np.median(rmse_lo)  if rmse_lo else np.nan,
            "median_rmse_all":  np.median(rmse_hi + rmse_lo),
            "median_sout_hi":   np.median(sout_hi)  if sout_hi else np.nan,
            "median_ct_hi":     np.median(ct_hi)    if ct_hi   else np.nan,
            "n_rt_in_data_hi":  n_rt_in_data_hi,
            "n_rt_in_data_all": n_rt_in_data_all,
            "pct_rt_in_data_hi": 100.0*n_rt_in_data_hi/max(len(hi_idx),1),
        })

        done += 1
        if done % 8 == 0:
            print(f"  {done}/{total} done...")

scan_df = pd.DataFrame(scan_rows)

# ============================================================
# SCORING
# ------------------------------------------------------------
# Primary: minimise high-vmax RMSE
# Secondary: maximise % of high-vmax galaxies where rt is
#            in the observed data range (the whole point)
# Penalty: if low-vmax RMSE degrades more than 20%
# ============================================================
baseline_lo = scan_df[scan_df["F_FRAC"]==0.60][scan_df["Rs"]==1.5]["median_rmse_lo"].values
baseline_lo = float(baseline_lo[0]) if len(baseline_lo) else scan_df["median_rmse_lo"].median()

scan_df["lo_penalty"] = np.maximum(
    0, (scan_df["median_rmse_lo"] - baseline_lo) / max(baseline_lo, 1e-9)
)
scan_df["score"] = (scan_df["median_rmse_hi"]
                    * (1.0 + 2.0 * scan_df["lo_penalty"])
                    / np.maximum(scan_df["pct_rt_in_data_hi"] / 100.0, 0.01))

scan_df_sorted = scan_df.sort_values("score")

print("\n" + "=" * 80)
print("FULL SCAN RESULTS — sorted by score")
print("(score rewards: low hi-RMSE, high % rt-in-data, no lo-RMSE degradation)")
print("=" * 80)
print(scan_df_sorted.to_string(index=False))

print("\n" + "=" * 80)
print("TOP 10 BY RAW HIGH-VMAX RMSE")
print("=" * 80)
print(scan_df.sort_values("median_rmse_hi").head(10).to_string(index=False))

print("\n" + "=" * 80)
print("% OF HIGH-VMAX GALAXIES WITH rt IN OBSERVED DATA RANGE")
print("=" * 80)
pivot = scan_df.pivot(index="F_FRAC", columns="Rs", values="pct_rt_in_data_hi")
print(pivot.to_string())

print("\n" + "=" * 80)
print("MEDIAN HIGH-VMAX RMSE GRID")
print("=" * 80)
pivot_rmse = scan_df.pivot(index="F_FRAC", columns="Rs", values="median_rmse_hi")
print(pivot_rmse.applymap(lambda x: f"{x:.3e}").to_string())

print("\n" + "=" * 80)
print("MEDIAN c_t GRID (high-vmax) — want this to come down")
print("=" * 80)
pivot_ct = scan_df.pivot(index="F_FRAC", columns="Rs", values="median_ct_hi")
print(pivot_ct.applymap(lambda x: f"{x:.2e}").to_string())

# ============================================================
# DEEP DIVE on best config
# ============================================================
best      = scan_df_sorted.iloc[0]
best_ff   = float(best["F_FRAC"])
best_Rs   = float(best["Rs"])

print(f"\n{'='*80}")
print(f"BEST CONFIG: F_FRAC={best_ff}  Rs={best_Rs}")
print(f"{'='*80}")

rows_best = []
for g in precomp:
    res = evaluate(g, best_Rs, best_ff)
    if res is None:
        continue
    rows_best.append({
        "name":        g["name"],
        "logM":        g["logM"],
        "vmax_obs":    g["vmax_obs"],
        "r_min":       g["r_min"],
        "rt":          res["rt"],
        "rt_in_data":  res["rt_in_data"],
        "c_t":         res["c_t"],
        "damp":        res["damp"],
        "rmse":        res["rmse"],
        "s_out":       res["s_out"],
    })

best_df = pd.DataFrame(rows_best)
hi_best = best_df[best_df["vmax_obs"] >= HIGH_VMAX_CUT]
lo_best = best_df[best_df["vmax_obs"] <  HIGH_VMAX_CUT]

print(f"\nHigh-vmax  median RMSE : {hi_best['rmse'].median():.4e}  (n={len(hi_best)})")
print(f"Low-vmax   median RMSE : {lo_best['rmse'].median():.4e}  (n={len(lo_best)})")
print(f"rt in observed range   : {hi_best['rt_in_data'].sum()} / {len(hi_best)} high-vmax")

print(f"\nc_t distribution (high-vmax, best config):")
ct_vals = hi_best["c_t"].replace([np.inf,-np.inf], np.nan).dropna()
for pct in [0, 25, 50, 75, 90, 100]:
    print(f"  p{pct:3d} = {np.percentile(ct_vals, pct):.4e}")

print(f"\nOUTER BIAS DIRECTION (high-vmax, best config)")
over  = int((hi_best["s_out"] >  0.01).sum())
under = int((hi_best["s_out"] < -0.01).sum())
near  = int(((hi_best["s_out"] >= -0.01) & (hi_best["s_out"] <= 0.01)).sum())
print(f"  Over-predicting  : {over}")
print(f"  Under-predicting : {under}")
print(f"  Near-zero        : {near}")

print(f"\nWORST 10 HIGH-VMAX (best config)")
print(hi_best.sort_values("rmse", ascending=False)
      [["name","logM","vmax_obs","r_min","rt","rt_in_data","c_t","damp","rmse","s_out"]]
      .head(10).to_string(index=False))

print(f"\nBEST 10 HIGH-VMAX (best config)")
print(hi_best.sort_values("rmse")
      [["name","logM","vmax_obs","r_min","rt","rt_in_data","c_t","damp","rmse","s_out"]]
      .head(10).to_string(index=False))

# ============================================================
# ALSO: show the original config (F_FRAC=0.60, Rs=1.5) for
# direct comparison — same code path, apples to apples
# ============================================================
orig_row = scan_df[(scan_df["F_FRAC"]==0.60) & (scan_df["Rs"]==1.5)]
print(f"\n{'='*80}")
print(f"ORIGINAL CONFIG (F_FRAC=0.60, Rs=1.5) for comparison:")
print(orig_row[["F_FRAC","Rs","median_rmse_hi","median_rmse_lo",
                "pct_rt_in_data_hi","median_ct_hi"]].to_string(index=False))

# ============================================================
# SAVE
# ============================================================
scan_df_sorted.to_csv(os.path.join(OUTDIR, "ffrac_rs_scan.csv"), index=False)
best_df.to_csv(os.path.join(OUTDIR, "best_config_detail.csv"), index=False)
print(f"\nSaved to {OUTDIR}")
print("\nDONE.")

Found 175 rotmod files
Pre-computed: 165 galaxies

High-vmax: 56  |  Low-vmax: 109

  8/64 done...
  16/64 done...
  24/64 done...
  32/64 done...
  40/64 done...
  48/64 done...
  56/64 done...
  64/64 done...

FULL SCAN RESULTS — sorted by score
(score rewards: low hi-RMSE, high % rt-in-data, no lo-RMSE degradation)
 F_FRAC  Rs  median_rmse_hi  median_rmse_lo  median_rmse_all  median_sout_hi  median_ct_hi  n_rt_in_data_hi  n_rt_in_data_all  pct_rt_in_data_hi  lo_penalty        score
   0.60 0.5    1.080493e+06    3.138093e+05     3.735211e+05        0.152515   11691046.15                0                 0                0.0    0.000000 1.080493e+08
   0.70 0.5    1.080493e+06    3.138093e+05     3.735211e+05        0.152515   11691046.15                0                 0                0.0    0.000000 1.080493e+08
   0.80 0.5    1.080493e+06    3.138093e+05     3.735211e+05        0.152515   11691046.15                0                 0                0.0    0.000000 1.080493e+08


/tmp/ipykernel_4927/3150841727.py:368: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(pivot_rmse.applymap(lambda x: f"{x:.3e}").to_string())
/tmp/ipykernel_4927/3150841727.py:374: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(pivot_ct.applymap(lambda x: f"{x:.2e}").to_string())



High-vmax  median RMSE : 1.0805e+06  (n=56)
Low-vmax   median RMSE : 3.1381e+05  (n=109)
rt in observed range   : 0 / 56 high-vmax

c_t distribution (high-vmax, best config):
  p  0 = 4.6077e+05
  p 25 = 2.3515e+06
  p 50 = 1.1691e+07
  p 75 = 3.3096e+07
  p 90 = 4.1732e+07
  p100 = 1.0956e+08

OUTER BIAS DIRECTION (high-vmax, best config)
  Over-predicting  : 31
  Under-predicting : 25
  Near-zero        : 0

WORST 10 HIGH-VMAX (best config)
       name      logM  vmax_obs  r_min    rt  rt_in_data        c_t         damp         rmse     s_out
    NGC2998 11.316494     214.0   0.33 0.001       False  2385593.6 3.295649e+05 2.867863e+07 24.731800
    NGC6946 10.876567     181.0   0.19 0.001       False   660281.6 4.798968e+04 2.528362e+07  0.185947
    NGC2903 10.948754     216.0   0.32 0.001       False 39196752.5 2.194927e+07 1.462417e+07 -0.815168
    NGC1090 10.991659     176.0   0.35 0.001       False  2474446.1 3.481475e+05 1.421004e+07 -0.632018
   UGC02953 11.393352     319.0 

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed field model
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Gpk = float(np.max(gb))
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "Gpk": Gpk,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# load + precompute
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

raw = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    if np.all(np.isfinite(qshape)):
        raw.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw))

GSTAR = np.median([g["Gpk"] for g, _ in raw])
print("GSTAR (median Gpk):", GSTAR)

# ------------------------------------------------------------
# scan eta and lambda0
# lambda_b = lambda0 * sqrt(f20)/Cgeom * (Gpk/GSTAR)^eta
# ------------------------------------------------------------
eta_vals = np.linspace(-2.0, 2.0, 41)
lambda0_vals = np.logspace(-2, 1.5, 81)

results = []
best = None
best_score = np.inf
best_df = None

for eta in eta_vals:
    for lambda0 in lambda0_vals:
        num = 0.0
        den = 0.0
        rows_pre = []

        for g, qshape in raw:
            shape_fac = (np.maximum(g["Gpk"], EPS) / np.maximum(GSTAR, EPS)) ** eta
            lambda_b = lambda0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS) * shape_fac

            y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
            num += np.sum(y_eff * qshape)
            den += np.sum(qshape**2)
            rows_pre.append((g, qshape, lambda_b))

        if den <= 0:
            continue

        C_global = max(num / den, 0.0)

        rows = []
        for g, qshape, lambda_b in rows_pre:
            vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
            vp = np.sqrt(vpred2)
            vo = g["vo"]

            rmse = np.sqrt(np.mean((vo - vp)**2))
            mae = np.mean(np.abs(vo - vp))

            n = len(vo)
            i1 = max(1, n // 3)
            i2 = max(i1 + 1, 2 * n // 3)
            Vscale = max(np.max(vo), EPS)

            s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
            s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
            s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

            rows.append({
                "name": g["name"],
                "logM": g["logM"],
                "lambda_b": lambda_b,
                "rmse": rmse,
                "mae": mae,
                "s_in": s_in,
                "s_mid": s_mid,
                "s_out": s_out,
            })

        df = pd.DataFrame(rows)
        hi = df["logM"] >= 10.8
        med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
        mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

        out = {
            "eta": eta,
            "lambda0": lambda0,
            "global_C": C_global,
            "median_rmse": df["rmse"].median(),
            "mean_rmse": df["rmse"].mean(),
            "median_rmse_hiM": med_hi,
            "mean_rmse_hiM": mean_hi,
            "inner": df["s_in"].median(),
            "mid": df["s_mid"].median(),
            "outer": df["s_out"].median(),
            "corr_logM": corr(df["logM"], df["rmse"]),
            "median_lambda_b": np.median(df["lambda_b"]),
        }
        results.append(out)

        score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
        if score < best_score:
            best_score = score
            best = out.copy()
            best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 GPK-MODULATED FIXED-THEORY TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST GPK-MODULATED THEORY")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143
GSTAR (median Gpk): 2342.3742424242423

TOP 20 GPK-MODULATED FIXED-THEORY TESTS
 eta   lambda0  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM    inner      mid    outer  corr_logM  median_lambda_b
-1.5 15.622481  0.011447    76.653484  85.319941        42.518296      55.475195 0.422707 0.537702 0.452174  -0.375904         3.199354
-1.6 17.278260  0.011268    83.902053  96.913502        42.850333      56.151184 0.470850 0.593978 0.497365  -0.419336         3.488926
-1.2 21.134890  0.008979    79.492510  80.203760        42.948198      56.294680 0.482877 0.639314 0.477696  -0.323122         4.414259
-1.4 14.125375  0.011573    69.892063  75.230864        43.100708      54.842462 0.370253 0.484848 0.378422  -0.315361         2.933816
-1.7 17.278260  0.011404    84.560190 104.662112        43.217374      56.659975 0.473949 0.602758 0.496997  -0.438220         3.473805
-1.6 15.622481  0.011624    81.14597

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed field model
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Sigma = Mtot / np.maximum(Rmax**2, EPS)
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "Sigma": Sigma,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# load + precompute
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

raw = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    if np.all(np.isfinite(qshape)):
        raw.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw))

SIGMASTAR = np.median([g["Sigma"] for g, _ in raw])
print("SIGMASTAR (median Sigma):", SIGMASTAR)

# ------------------------------------------------------------
# scan eta and lambda0
# lambda_b = lambda0 * sqrt(f20)/Cgeom * (Sigma/SIGMASTAR)^eta
# ------------------------------------------------------------
eta_vals = np.linspace(-2.0, 2.0, 41)
lambda0_vals = np.logspace(-2, 1.5, 81)

results = []
best = None
best_score = np.inf
best_df = None

for eta in eta_vals:
    for lambda0 in lambda0_vals:
        num = 0.0
        den = 0.0
        rows_pre = []

        for g, qshape in raw:
            shape_fac = (np.maximum(g["Sigma"], EPS) / np.maximum(SIGMASTAR, EPS)) ** eta
            lambda_b = lambda0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS) * shape_fac

            y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
            num += np.sum(y_eff * qshape)
            den += np.sum(qshape**2)
            rows_pre.append((g, qshape, lambda_b))

        if den <= 0:
            continue

        C_global = max(num / den, 0.0)

        rows = []
        for g, qshape, lambda_b in rows_pre:
            vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
            vp = np.sqrt(vpred2)
            vo = g["vo"]

            rmse = np.sqrt(np.mean((vo - vp)**2))
            mae = np.mean(np.abs(vo - vp))

            n = len(vo)
            i1 = max(1, n // 3)
            i2 = max(i1 + 1, 2 * n // 3)
            Vscale = max(np.max(vo), EPS)

            s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
            s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
            s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

            rows.append({
                "name": g["name"],
                "logM": g["logM"],
                "lambda_b": lambda_b,
                "rmse": rmse,
                "mae": mae,
                "s_in": s_in,
                "s_mid": s_mid,
                "s_out": s_out,
            })

        df = pd.DataFrame(rows)
        hi = df["logM"] >= 10.8
        med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
        mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

        out = {
            "eta": eta,
            "lambda0": lambda0,
            "global_C": C_global,
            "median_rmse": df["rmse"].median(),
            "mean_rmse": df["rmse"].mean(),
            "median_rmse_hiM": med_hi,
            "mean_rmse_hiM": mean_hi,
            "inner": df["s_in"].median(),
            "mid": df["s_mid"].median(),
            "outer": df["s_out"].median(),
            "corr_logM": corr(df["logM"], df["rmse"]),
            "median_lambda_b": np.median(df["lambda_b"]),
        }
        results.append(out)

        score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
        if score < best_score:
            best_score = score
            best = out.copy()
            best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 SIGMA-MODULATED FIXED-THEORY TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST SIGMA-MODULATED THEORY")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143
SIGMASTAR (median Sigma): 58084698.97658812

TOP 20 SIGMA-MODULATED FIXED-THEORY TESTS
 eta  lambda0  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM    inner      mid     outer  corr_logM  median_lambda_b
-1.1 6.309573  0.009730    21.903664  30.249117        41.799463      50.330404 0.051794 0.047889  0.018804   0.606197         1.115787
-1.2 5.704926  0.010310    23.087190  30.701929        42.157252      51.128757 0.044951 0.050127  0.024294   0.608979         1.027180
-1.1 6.978306  0.009173    22.668336  30.787782        42.528199      50.151154 0.068821 0.057098  0.020748   0.591055         1.234045
-1.2 6.309573  0.009815    23.184750  31.213666        42.902728      50.861247 0.061893 0.055282  0.023282   0.592980         1.136048
-1.3 5.704926  0.010360    23.888940  31.641851        43.007482      51.625653 0.044471 0.050220  0.027408   0.594210         1.032980
-0.7 8.535913  0.006910    19

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed field model
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Sigma = Mtot / np.maximum(Rmax**2, EPS)
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "Sigma": Sigma,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# load + precompute
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

raw = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    if np.all(np.isfinite(qshape)):
        raw.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw))

SIGMASTAR = np.median([g["Sigma"] for g, _ in raw])
print("SIGMASTAR (median Sigma):", SIGMASTAR)

# ------------------------------------------------------------
# scan eta and lambda0
# lambda_b = lambda0 * sqrt(f20)/Cgeom * (Sigma/SIGMASTAR)^eta
# ------------------------------------------------------------
eta_vals = np.linspace(-2.0, 2.0, 41)
lambda0_vals = np.logspace(-2, 1.5, 81)

results = []
best = None
best_score = np.inf
best_df = None

for eta in eta_vals:
    for lambda0 in lambda0_vals:
        num = 0.0
        den = 0.0
        rows_pre = []

        for g, qshape in raw:
            shape_fac = (np.maximum(g["Sigma"], EPS) / np.maximum(SIGMASTAR, EPS)) ** eta
            lambda_b = lambda0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS) * shape_fac

            y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
            num += np.sum(y_eff * qshape)
            den += np.sum(qshape**2)
            rows_pre.append((g, qshape, lambda_b))

        if den <= 0:
            continue

        C_global = max(num / den, 0.0)

        rows = []
        for g, qshape, lambda_b in rows_pre:
            vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
            vp = np.sqrt(vpred2)
            vo = g["vo"]

            rmse = np.sqrt(np.mean((vo - vp)**2))
            mae = np.mean(np.abs(vo - vp))

            n = len(vo)
            i1 = max(1, n // 3)
            i2 = max(i1 + 1, 2 * n // 3)
            Vscale = max(np.max(vo), EPS)

            s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
            s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
            s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

            rows.append({
                "name": g["name"],
                "logM": g["logM"],
                "lambda_b": lambda_b,
                "rmse": rmse,
                "mae": mae,
                "s_in": s_in,
                "s_mid": s_mid,
                "s_out": s_out,
            })

        df = pd.DataFrame(rows)
        hi = df["logM"] >= 10.8
        med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
        mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

        out = {
            "eta": eta,
            "lambda0": lambda0,
            "global_C": C_global,
            "median_rmse": df["rmse"].median(),
            "mean_rmse": df["rmse"].mean(),
            "median_rmse_hiM": med_hi,
            "mean_rmse_hiM": mean_hi,
            "inner": df["s_in"].median(),
            "mid": df["s_mid"].median(),
            "outer": df["s_out"].median(),
            "corr_logM": corr(df["logM"], df["rmse"]),
            "median_lambda_b": np.median(df["lambda_b"]),
        }
        results.append(out)

        score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
        if score < best_score:
            best_score = score
            best = out.copy()
            best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 SIGMA-MODULATED FIXED-THEORY TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST SIGMA-MODULATED THEORY")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143
SIGMASTAR (median Sigma): 58084698.97658812

TOP 20 SIGMA-MODULATED FIXED-THEORY TESTS
 eta  lambda0  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM    inner      mid     outer  corr_logM  median_lambda_b
-1.1 6.309573  0.009730    21.903664  30.249117        41.799463      50.330404 0.051794 0.047889  0.018804   0.606197         1.115787
-1.2 5.704926  0.010310    23.087190  30.701929        42.157252      51.128757 0.044951 0.050127  0.024294   0.608979         1.027180
-1.1 6.978306  0.009173    22.668336  30.787782        42.528199      50.151154 0.068821 0.057098  0.020748   0.591055         1.234045
-1.2 6.309573  0.009815    23.184750  31.213666        42.902728      50.861247 0.061893 0.055282  0.023282   0.592980         1.136048
-1.3 5.704926  0.010360    23.888940  31.641851        43.007482      51.625653 0.044471 0.050220  0.027408   0.594210         1.032980
-0.7 8.535913  0.006910    19

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed field model
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Sigma = Mtot / np.maximum(Rmax**2, EPS)
    Gpk = float(np.max(gb))
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "Sigma": Sigma,
        "Gpk": Gpk,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# load + precompute
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

raw = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    if np.all(np.isfinite(qshape)):
        raw.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw))

SIGMASTAR = np.median([g["Sigma"] for g, _ in raw])
GSTAR = np.median([g["Gpk"] for g, _ in raw])

print("SIGMASTAR:", SIGMASTAR)
print("GSTAR:", GSTAR)

# ------------------------------------------------------------
# scan eta1, eta2, lambda0
# lambda_b = lambda0 * sqrt(f20)/Cgeom * (Sigma/SIGMASTAR)^eta1 * (Gpk/GSTAR)^eta2
# ------------------------------------------------------------
eta1_vals = np.linspace(-1.5, 1.5, 13)   # Sigma exponent
eta2_vals = np.linspace(-1.5, 1.5, 13)   # Gpk exponent
lambda0_vals = np.logspace(-2, 1.5, 61)

results = []
best = None
best_score = np.inf
best_df = None

for eta1 in eta1_vals:
    for eta2 in eta2_vals:
        for lambda0 in lambda0_vals:
            num = 0.0
            den = 0.0
            rows_pre = []

            for g, qshape in raw:
                sigma_fac = (np.maximum(g["Sigma"], EPS) / np.maximum(SIGMASTAR, EPS)) ** eta1
                gpk_fac   = (np.maximum(g["Gpk"],   EPS) / np.maximum(GSTAR,   EPS)) ** eta2

                lambda_b = (
                    lambda0
                    * np.sqrt(np.maximum(g["f20"], EPS))
                    / np.maximum(g["Cgeom"], EPS)
                    * sigma_fac
                    * gpk_fac
                )

                y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
                num += np.sum(y_eff * qshape)
                den += np.sum(qshape**2)
                rows_pre.append((g, qshape, lambda_b))

            if den <= 0:
                continue

            C_global = max(num / den, 0.0)

            rows = []
            for g, qshape, lambda_b in rows_pre:
                vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
                vp = np.sqrt(vpred2)
                vo = g["vo"]

                rmse = np.sqrt(np.mean((vo - vp)**2))
                mae = np.mean(np.abs(vo - vp))

                n = len(vo)
                i1 = max(1, n // 3)
                i2 = max(i1 + 1, 2 * n // 3)
                Vscale = max(np.max(vo), EPS)

                s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
                s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
                s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

                rows.append({
                    "name": g["name"],
                    "logM": g["logM"],
                    "lambda_b": lambda_b,
                    "rmse": rmse,
                    "mae": mae,
                    "s_in": s_in,
                    "s_mid": s_mid,
                    "s_out": s_out,
                })

            df = pd.DataFrame(rows)
            hi = df["logM"] >= 10.8
            med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
            mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

            out = {
                "eta1": eta1,
                "eta2": eta2,
                "lambda0": lambda0,
                "global_C": C_global,
                "median_rmse": df["rmse"].median(),
                "mean_rmse": df["rmse"].mean(),
                "median_rmse_hiM": med_hi,
                "mean_rmse_hiM": mean_hi,
                "inner": df["s_in"].median(),
                "mid": df["s_mid"].median(),
                "outer": df["s_out"].median(),
                "corr_logM": corr(df["logM"], df["rmse"]),
                "median_lambda_b": np.median(df["lambda_b"]),
            }
            results.append(out)

            score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
            if score < best_score:
                best_score = score
                best = out.copy()
                best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 SIGMA+GPK MODULATED FIXED-THEORY TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST SIGMA+GPK MODULATED THEORY")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143
SIGMASTAR: 58084698.97658812
GSTAR: 2342.3742424242423

TOP 20 SIGMA+GPK MODULATED FIXED-THEORY TESTS
 eta1  eta2   lambda0  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM     inner       mid     outer  corr_logM  median_lambda_b
-1.50 -1.00 16.155981  0.011041    63.840856  89.143332        40.635864      55.404009  0.327317  0.324778  0.291395  -0.294676         2.795052
-1.50 -0.75 10.797752  0.011786    46.570176  61.706351        41.306755      54.357891  0.252892  0.260529  0.226156  -0.102840         2.175062
-0.25 -1.25 27.648188  0.008544    88.483695  99.813528        41.478491      56.137384  0.570243  0.770394  0.596109  -0.426340         4.969599
-0.25 -1.50 27.648188  0.009455    94.466100 120.307911        41.875453      57.004348  0.641507  0.802343  0.658379  -0.471357         5.150941
-1.25 -1.00 14.125375  0.011713    60.526111  76.486315        41.968510      54.520693  0.292456  0

In [ ]:
import numpy as np
import glob
import os
import pandas as pd

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-12
MAX_POINTS = 120

# fixed field model
DELTA = 1.3
P = 2.3
QEXP = 0.9
MEXP = 1.5

files = sorted(glob.glob(ROT_PATH))

def cumulative_trapz(y, x):
    out = np.zeros_like(y, dtype=float)
    for i in range(1, len(y)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) < 3:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return np.corrcoef(a[m], b[m])[0,1]

def downsample(arrs, max_points=MAX_POINTS):
    n = len(arrs[0])
    if n <= max_points:
        return arrs
    idx = np.unique(np.linspace(0, n-1, max_points).astype(int))
    return [a[idx] for a in arrs]

def build_galaxy(path):
    data = np.genfromtxt(path)
    if data.ndim != 2 or data.shape[1] < 5:
        return None

    r, vo, vg, vd, vb = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4]
    m = (
        np.isfinite(r) & np.isfinite(vo) &
        np.isfinite(vg) & np.isfinite(vd) & np.isfinite(vb) &
        (r > 0) & (vo > 0)
    )
    if np.sum(m) < 8:
        return None

    r, vo, vg, vd, vb = r[m], vo[m], vg[m], vd[m], vb[m]
    o = np.argsort(r)
    r, vo, vg, vd, vb = r[o], vo[o], vg[o], vd[o], vb[o]
    r, vo, vg, vd, vb = downsample([r, vo, vg, vd, vb])

    vgas2  = np.maximum(vg**2, 0.0)
    vdisk2 = np.maximum(vd**2, 0.0)
    vbul2  = np.maximum(vb**2, 0.0)
    vbar2  = np.maximum(vgas2 + vdisk2 + vbul2, EPS)

    gb = vbar2 / np.maximum(r, EPS)

    Mgas  = np.maximum(r * vgas2  / G, 0.0)
    Mdisk = np.maximum(r * vdisk2 / G, 0.0)
    Mbul  = np.maximum(r * vbul2  / G, 0.0)
    Menc  = np.maximum(Mgas + Mdisk + Mbul, EPS)

    dMdr = np.gradient(Menc, r)
    rho3d = np.maximum(dMdr / np.maximum(4.0*np.pi*r**2, EPS), EPS)

    Mtot = Menc[-1]
    Rmax = r[-1]

    idx50 = min(np.searchsorted(Menc, 0.5*Mtot), len(r)-1)
    R50 = max(r[idx50], EPS)
    Cgeom = Rmax / R50

    idx20 = min(np.searchsorted(r, 0.2*Rmax), len(r)-1)
    f20 = float(Menc[idx20] / np.maximum(Mtot, EPS))

    Sigma = Mtot / np.maximum(Rmax**2, EPS)
    gb_ref = np.median(np.maximum(gb, EPS))

    return {
        "name": os.path.basename(path),
        "r": r,
        "vo": vo,
        "vbar2": vbar2,
        "gb": gb,
        "rho3d": rho3d,
        "Mtot": Mtot,
        "Rmax": Rmax,
        "R50": R50,
        "Cgeom": Cgeom,
        "f20": f20,
        "Sigma": Sigma,
        "gb_ref": gb_ref,
        "logM": np.log10(Mtot),
    }

def solve_qshape_zeroF0(r, source):
    F = cumulative_trapz((r**DELTA) * source, r)
    base = np.maximum(F / np.maximum(r**DELTA, EPS), 0.0)
    Mp = base ** (1.0 / (P - 1.0))
    qshape = np.maximum(r * Mp, EPS) ** QEXP
    return qshape

# ------------------------------------------------------------
# load + precompute
# ------------------------------------------------------------
galaxies = []
for f in files:
    try:
        g = build_galaxy(f)
        if g is not None:
            galaxies.append(g)
    except Exception:
        continue

print("Usable galaxies:", len(galaxies))

raw = []
for g in galaxies:
    r = g["r"]
    gb = g["gb"]
    rho3d = g["rho3d"]

    g0 = 0.68 * g["gb_ref"]
    activation = 1.0 - np.exp(-((gb / np.maximum(g0, EPS)) ** MEXP))
    source = rho3d * activation
    qshape = solve_qshape_zeroF0(r, source)

    if np.all(np.isfinite(qshape)):
        raw.append((g, qshape))

print("Precomputed usable field-channel galaxies:", len(raw))

SIGMASTAR = np.median([g["Sigma"] for g, _ in raw])
print("SIGMASTAR (median Sigma):", SIGMASTAR)

# ------------------------------------------------------------
# scan lambda0, eta, delta in damping law
# lambda_b = lambda0 * sqrt(f20)/Cgeom / (1 + eta*(Sigma/SIGMASTAR)^delta)
# eta >= 0, delta >= 0
# ------------------------------------------------------------
lambda0_vals = np.logspace(-1, 1.5, 61)
eta_vals = np.logspace(-2, 2, 41)
delta_vals = np.linspace(0.25, 3.0, 24)

results = []
best = None
best_score = np.inf
best_df = None

for lambda0 in lambda0_vals:
    for eta_damp in eta_vals:
        for delta_damp in delta_vals:
            num = 0.0
            den = 0.0
            rows_pre = []

            for g, qshape in raw:
                comp = (np.maximum(g["Sigma"], EPS) / np.maximum(SIGMASTAR, EPS)) ** delta_damp
                damp = 1.0 / (1.0 + eta_damp * comp)
                lambda_b = lambda0 * np.sqrt(np.maximum(g["f20"], EPS)) / np.maximum(g["Cgeom"], EPS) * damp

                y_eff = g["vo"]**2 - lambda_b * g["vbar2"]
                num += np.sum(y_eff * qshape)
                den += np.sum(qshape**2)
                rows_pre.append((g, qshape, lambda_b))

            if den <= 0:
                continue

            C_global = max(num / den, 0.0)

            rows = []
            for g, qshape, lambda_b in rows_pre:
                vpred2 = np.maximum(C_global * qshape + lambda_b * g["vbar2"], 0.0)
                vp = np.sqrt(vpred2)
                vo = g["vo"]

                rmse = np.sqrt(np.mean((vo - vp)**2))
                mae = np.mean(np.abs(vo - vp))

                n = len(vo)
                i1 = max(1, n // 3)
                i2 = max(i1 + 1, 2 * n // 3)
                Vscale = max(np.max(vo), EPS)

                s_in  = np.mean(vp[:i1]   - vo[:i1])   / Vscale
                s_mid = np.mean(vp[i1:i2] - vo[i1:i2]) / Vscale
                s_out = np.mean(vp[i2:]   - vo[i2:])   / Vscale

                rows.append({
                    "name": g["name"],
                    "logM": g["logM"],
                    "lambda_b": lambda_b,
                    "rmse": rmse,
                    "mae": mae,
                    "s_in": s_in,
                    "s_mid": s_mid,
                    "s_out": s_out,
                })

            df = pd.DataFrame(rows)
            hi = df["logM"] >= 10.8
            med_hi = df.loc[hi, "rmse"].median() if np.any(hi) else np.nan
            mean_hi = df.loc[hi, "rmse"].mean() if np.any(hi) else np.nan

            out = {
                "lambda0": lambda0,
                "eta_damp": eta_damp,
                "delta_damp": delta_damp,
                "global_C": C_global,
                "median_rmse": df["rmse"].median(),
                "mean_rmse": df["rmse"].mean(),
                "median_rmse_hiM": med_hi,
                "mean_rmse_hiM": mean_hi,
                "inner": df["s_in"].median(),
                "mid": df["s_mid"].median(),
                "outer": df["s_out"].median(),
                "corr_logM": corr(df["logM"], df["rmse"]),
                "median_lambda_b": np.median(df["lambda_b"]),
            }
            results.append(out)

            score = (med_hi if np.isfinite(med_hi) else 1e9) + 0.2*out["median_rmse"]
            if score < best_score:
                best_score = score
                best = out.copy()
                best_df = df.copy()

res_df = pd.DataFrame(results).sort_values(["median_rmse_hiM", "median_rmse"])

print("\nTOP 20 COMPACTNESS-DAMPED FIXED-THEORY TESTS")
print(res_df.head(20).to_string(index=False))

print("\nBEST MODEL")
print(best)

print("\nWORST HIGH-MASS GALAXIES UNDER BEST COMPACTNESS-DAMPED THEORY")
hi = best_df["logM"] >= 10.8
print(
    best_df.loc[hi, ["name","logM","lambda_b","rmse","s_in","s_mid","s_out"]]
           .sort_values("rmse", ascending=False)
           .head(20)
           .to_string(index=False)
)

Usable galaxies: 143
Precomputed usable field-channel galaxies: 143
SIGMASTAR (median Sigma): 58084698.97658812

TOP 20 COMPACTNESS-DAMPED FIXED-THEORY TESTS
  lambda0  eta_damp  delta_damp  global_C  median_rmse  mean_rmse  median_rmse_hiM  mean_rmse_hiM    inner      mid    outer  corr_logM  median_lambda_b
10.000000  0.630957    2.521739  0.011093    18.995191  28.482233        41.831552      52.497147 0.038874 0.040252 0.019329   0.673087         0.959493
11.006942  0.794328    2.282609  0.010967    20.072131  28.496509        42.042324      52.224856 0.044736 0.041601 0.018466   0.669195         0.969377
11.006942  0.794328    2.402174  0.011059    20.212367  28.649534        42.133560      52.423164 0.046398 0.044678 0.019815   0.671544         0.976727
26.101572  3.162278    1.445652  0.010210    21.128315  29.109607        42.138433      50.962938 0.057048 0.052211 0.023604   0.644224         1.093405
23.713737  3.162278    1.565217  0.010759    21.701736  29.225662        42.1